## Environment Setup & Dependency Installation
**Run this cell first on every new Colab session.** All library versions are pinned to ensure exact reproducibility of results across different runtime environments and future Colab updates.

| Library | Version | Purpose |
|---|---|---|
| `tensorflow` | ≥2.13 | ST-GNN and LSTM training |
| `scikit-learn` | ≥1.3 | Baselines, clustering, metrics |
| `pandas` | ≥2.0 | Data manipulation |
| `numpy` | ≥1.24 | Numerical operations |
| `scipy` | ≥1.11 | Wilcoxon test, confidence intervals |
| `folium` | ≥0.14 | Interactive map visualization |
| `seaborn` | ≥0.12 | Academic plot formatting |
| `matplotlib` | ≥3.7 | All chart artifacts |
| `kagglehub` | ≥0.2 | Automated dataset ingestion |

In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP — Pinned dependency installation
# Run this cell first. All versions tested on Google Colab (Python 3.10+).
# ==============================================================================
!pip install -q tensorflow>=2.13
!pip install -q scikit-learn>=1.3
!pip install -q pandas>=2.0
!pip install -q numpy>=1.24
!pip install -q scipy>=1.11
!pip install -q folium>=0.14
!pip install -q seaborn>=0.12
!pip install -q matplotlib>=3.7
!pip install -q kagglehub>=0.2
!pip install h3
print('[SUCCESS] All dependencies installed and pinned.')
print('[INFO] Restart runtime if TensorFlow was newly installed, then re-run from Phase 1.')

# RESEARCH THESIS: The "Butterfly Effect" in Traffic
**Predicting Secondary Accident Chains via Spatio-Temporal Graph Neural Networks (ST-GNN)**

**Objective:**
This research empirically tests the spatial propagation of secondary traffic accidents using a localized, dynamic Adjacency Matrix. By restricting the spatial topology, this pipeline tests the boundaries of traffic shockwave physics across diverse urban and rural environments.

---


> **Note:** The city label below reflects `TARGET_CITY` set in Phase 1 Cell 6. To run for a different city, update only `TARGET_CITY` in Phase 1. All downstream phases, filenames, plots, and outputs update automatically.


Experiment_A = fixed urban transfer

Experiment_B = native urban topology discovery

Experiment_C1 = urban-to-rural transfer

Experiment_C1_5 = rural geometry diagnostic

Experiment_C2 = native rural topology discovery


# Los Angeles
(The Sprawling Web): A massive, multi-centric highway sprawl with vast distances between major interchanges.

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Los Angeles"  # Options: "Los Angeles", "Chicago", "Miami", "Indianapolis", "Iowa"

# Hyperparameter Search Space (Sensitivity Analysis)
SEARCH_NODES = [100, 500]       # MAUP scale sensitivity (node count)
SEARCH_RADII = [0.5, 1.0, 2.0]  # Physical interaction horizons (miles)
SEARCH_K = [3, 5, 8]            # Intersection routing topologies
SEQ_LENGTH = 4                  # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena...")

experiment_results = []
RESULTS_PATH = f"{BASE_DIR}/ST_GNN_Results_{TARGET_CITY}.csv"
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")
            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )
                total_pos = np.sum(y_tr); total_neg = (y_tr.size) - total_pos
                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1; continue
                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH, n_nodes=n_nodes, n_features=X_tr.shape[-1],
                    adj_matrix=adj, pos_weight=pos_weight_val
                )
                print("[*] Commencing Training...")
                history = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_val),
                    epochs=15, batch_size=64, callbacks=[early_stop], verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten(); y_pred_flat = y_pred_probs_val.flatten()
                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]; final_recall = recalls[best_idx]
                    final_f1 = 2*(final_precision*final_recall)/(final_precision+final_recall) if (final_precision+final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(auc_pr, 4),
                    'Optimal_Threshold': round(optimal_threshold, 4),
                    'Precision_at_70_Recall': round(final_precision, 4),
                    'Actual_Recall': round(final_recall, 4),
                    'Val_F1_Score': round(final_f1, 4)
                })
                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            tf.keras.backend.clear_session(); gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")


## PHASE 5B: MAUP Sensitivity Analysis
**Methodology:** The Modifiable Areal Unit Problem (MAUP) states that results derived from spatially aggregated data are sensitive to the choice of areal unit — in this case, the number of K-Means macro-nodes. A result that only holds for one node count is not robust.

This section loads the Phase 5 arena results and produces a sensitivity plot comparing AUC-PR across both node configurations (100 and 500) for each radius. A stable model should show consistent relative rankings regardless of spatial granularity.

In [ ]:
# ==============================================================================
# PHASE 5B: MAUP SENSITIVITY ANALYSIS (Scale)
# ==============================================================================
print("[*] Running MAUP Sensitivity Analysis...")

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")
arena_results_df = pd.read_csv(RESULTS_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'MAUP Sensitivity Analysis — Effect of Spatial Granularity on AUC-PR\nCity: {TARGET_CITY.upper()}',
             fontsize=13, fontweight='bold')
node_colors = NODE_COLORS

for ax_idx, radius in enumerate(SEARCH_RADII):
    ax = axes[ax_idx]
    subset = arena_results_df[arena_results_df['Radius_mi'] == radius].copy()
    for n_nodes_val, grp in subset.groupby('Nodes'):
        grp_sorted = grp.sort_values('K_Neighbors')
        ax.plot(
            grp_sorted['K_Neighbors'], grp_sorted['Val_AUC_PR'],
            marker='o', linewidth=2, color=node_colors.get(n_nodes_val, 'grey'),
            label=f'{n_nodes_val} Nodes'
        )
    ax.set_title(f'Radius = {radius}mi', fontweight='bold')
    ax.set_xlabel('K Neighbors', fontweight='bold'); ax.set_ylabel('Validation AUC-PR', fontweight='bold')
    ax.set_xticks(SEARCH_K); ax.legend(title='Node Count'); ax.grid(True, alpha=0.3)

plt.tight_layout()
MAUP_PATH = f"{BASE_DIR}/{TARGET_CITY}_MAUP_Sensitivity.png"
plt.savefig(MAUP_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] MAUP Sensitivity artifact saved to: {MAUP_PATH}")
plt.show()

print("\n[*] AUC-PR by Node Count (Mean across all K and Radius configurations):")
maup_summary = arena_results_df.groupby('Nodes')['Val_AUC_PR'].agg(['mean','std','min','max'])
print(maup_summary.round(4).to_string())


##Phase 5C: The MAUP Zonation Protocol

In [ ]:
# ==============================================================================
# PHASE 5C: MAUP ZONATION SENSITIVITY (N=100 Fixed)
# ==============================================================================
print(f"[*] Initiating Phase 5C: MAUP Zonation Stability Protocol for {TARGET_CITY.upper()}...")

# Anchor (Radius, K) for N=100 from Phase 5 (no hard-coded Phase 6 dependency)
subset_100 = arena_results_df[arena_results_df['Nodes'] == 100]
if subset_100.empty:
    raise ValueError("[ERROR] No N=100 results found to anchor zonation protocol.")
best_row = subset_100.sort_values('Val_AUC_PR', ascending=False).iloc[0]
FIXED_NODES = 100
FIXED_RADIUS = float(best_row['Radius_mi'])
FIXED_K = int(best_row['K_Neighbors'])
print(f"[*] Zonation test anchored at: N={FIXED_NODES}, Radius={FIXED_RADIUS}mi, K={FIXED_K}")

# Chronological sort to prevent time/geometry leakage
df_target['Start_Time'] = pd.to_datetime(df_target['Start_Time'])
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)

# Dynamic UTM projection for the city
transformer, epsg_code = _dynamic_utm_transformer(df_target)
utm_coords = _project_to_utm(df_target, transformer)

# --- Window-aligned train mask (PERFECT ALIGNMENT WITH _windows_and_split 70%) ---
start_min = df_target['Start_Time'].min().floor('h')
end_max   = df_target['Start_Time'].max().floor('h')
full_time = pd.date_range(start=start_min, end=end_max, freq='h')
total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)
cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]
train_mask = df_target['Start_Time'] < cutoff_time
utm_train = utm_coords[train_mask.values]
# -------------------------------------------------------------------------------

ZONATION_SEEDS = 20
zonation_auc_scores = []
test_prevalence = None

print(f"[*] Executing {ZONATION_SEEDS} K-Means Spatial Partitions (Train-Era Fitted, UTM={epsg_code})...")
for s in range(ZONATION_SEEDS):
    kmeans = KMeans(n_clusters=FIXED_NODES, random_state=s, n_init=10)
    kmeans.fit(utm_train)               # fit ONLY on window-aligned train era (no soft leakage)
    df_target['Region_ID'] = kmeans.predict(utm_coords)

    X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
        df_target, n_nodes=FIXED_NODES, radius_mi=FIXED_RADIUS, k_neighbors=FIXED_K, seq_len=SEQ_LENGTH
    )

    node_degrees = adj.sum(axis=1)
    if not (node_degrees > 0).all():
        isolated_count = (node_degrees == 0).sum()
        raise ValueError(f"[CRITICAL FAILURE] Adjacency contains {isolated_count} isolated nodes at Seed {s}. Adjust K or configuration.")

    if test_prevalence is None:
        test_prevalence = float(y_te.mean())

    # Recompute pos_weight for THIS seed's training fold
    pos = y_tr.sum(); neg = (y_tr.size - pos)
    pos_weight_val = float(neg / max(pos, 1))

    model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
    model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64, verbose=0)

    y_pred = model.predict(X_te, verbose=0).flatten()
    auc_pr = average_precision_score(y_te.flatten(), y_pred)
    zonation_auc_scores.append(auc_pr)

    del model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj
    tf.keras.backend.clear_session()

# ==============================================================================
# THE PERMANENT FIX: DYNAMIC H3 METHOD BINDING
# ==============================================================================
# Dynamically binds the correct H3 functions locally, immunizing the script against Colab memory drops.
h3_encode = getattr(h3, "latlng_to_cell", getattr(h3, "geo_to_h3", None))
h3_decode = getattr(h3, "cell_to_latlng", getattr(h3, "h3_to_geo", None))

if h3_encode is None or h3_decode is None:
    raise ImportError("[CRITICAL FAILURE] Unrecognized H3 library version. Encode/decode methods missing.")

print("\n[*] Executing Alternate Zoning Family: H3 Hexagonal Super-Regions (Train-Era Calibrated)...")

def choose_h3_resolution(df_like, target=100, tol=5, rmin=5, rmax=10):
    best = None
    for res in range(rmin, rmax+1):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        if abs(n - target) <= tol: return res, ids
        if best is None or abs(n - target) < abs(best[0] - target):
            best = (n, res, ids)
    return best[1], best[2]

# Use the exact same window-aligned train mask as above
h3_res, _ = choose_h3_resolution(df_target[train_mask], target=FIXED_NODES)
h3_ids_full = df_target.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], h3_res), axis=1)
n_hex = h3_ids_full.nunique()

if n_hex != FIXED_NODES:
    print(f"    -> Aggregating {n_hex} H3 hexes into exactly {FIXED_NODES} contiguous super-regions...")
    uniq_hex = pd.Series(h3_ids_full.astype(str).values).unique()
    hex_latlon = np.array([h3_decode(h) for h in uniq_hex])  # -> (lat, lng)
    hex_x, hex_y = transformer.transform(hex_latlon[:,1], hex_latlon[:,0])
    spatial_connectivity = kneighbors_graph(np.c_[hex_x, hex_y], n_neighbors=6, include_self=False)
    agg_cluster = AgglomerativeClustering(n_clusters=FIXED_NODES, connectivity=spatial_connectivity, linkage='ward')
    h_labels = agg_cluster.fit_predict(np.c_[hex_x, hex_y])
    hex_to_super = {h: int(l) for h, l in zip(uniq_hex, h_labels)}
    df_target['Region_ID'] = pd.Series(h3_ids_full.astype(str).values).map(hex_to_super).values
else:
    df_target['Region_ID'] = h3_ids_full.astype('category').cat.codes

X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
    df_target, n_nodes=FIXED_NODES, radius_mi=FIXED_RADIUS, k_neighbors=FIXED_K, seq_len=SEQ_LENGTH
)
pos = y_tr.sum(); neg = (y_tr.size - pos)
pos_weight_val = float(neg / max(pos, 1))
h3_model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
h3_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64, verbose=0)
h3_auc_pr = average_precision_score(y_te.flatten(), h3_model.predict(X_te, verbose=0).flatten())

from scipy.stats import percentileofscore
mean_auc = np.mean(zonation_auc_scores); sd_auc = np.std(zonation_auc_scores)
cv_percent = (sd_auc / mean_auc) * 100 if mean_auc > 0 else 0.0
q_lo, q_hi = np.quantile(zonation_auc_scores, [0.025, 0.975])
h3_pct = percentileofscore(zonation_auc_scores, h3_auc_pr)

print("\n========================================================")
print("[*] ZONATION MAUP RESULTS")
print("========================================================")
print(f"Test-Set Prevalence Baseline  : {float(test_prevalence):.4f}")
print(f"K-Means PR-AUC (n={ZONATION_SEEDS})      : {mean_auc:.4f} ± {sd_auc:.4f}")
print(f"95% Empirical Quantile CI     : [{q_lo:.4f}, {q_hi:.4f}]")
print(f"Coefficient of Variation (CV) : {cv_percent:.2f}%")
print(f"H3 Super-Region PR-AUC        : {h3_auc_pr:.4f}")
print(f"H3 Percentile within K-Means  : {h3_pct:.1f}th percentile")

del h3_model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj
gc.collect()

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# 1) DYNAMICALLY EXTRACT CHAMPION (Sorted by Val_AUC_PR)
champion_row = arena_results_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]
CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])
print(f"[*] Champion Topology Locked: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# 2) ISOLATE THE FINAL TENSORS (Champion)
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)
total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)
y_val_flat = y_val.flatten(); y_te_flat = y_te.flatten()

# 3) TRADITIONAL ML BASELINES (LR & RF) with SafeNodeClassifier
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1); X_te_ml = X_te.reshape(X_te.shape[0], -1)
lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)
    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        # SafeNodeClassifier always returns 2-col proba; guard is retained
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0
    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5
    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)
    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")
del X_tr_ml, X_te_ml, lr_model, rf_model; gc.collect()

# 4) DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
early_stop_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_final], verbose=0)
    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(auc_lstm); lstm_f1_scores.append(f1_lstm)
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_final], verbose=0)
    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(auc_stgnn); stgnn_f1_scores.append(f1_stgnn)
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

lstm_auc_mean,  lstm_auc_std  = np.mean(lstm_auc_scores),  np.std(lstm_auc_scores)
stgnn_auc_mean, stgnn_auc_std = np.mean(stgnn_auc_scores), np.std(stgnn_auc_scores)
lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")
    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This can occur when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/Thesis_Final_Proof_Matrix.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'LR_AUC': round(lr_auc, 4),
    'RF_AUC': round(rf_auc, 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(lstm_ci[0], 4),
    'LSTM_CI_Upper': round(lstm_ci[1], 4),
    'STGNN_CI_Lower': round(stgnn_ci[0], 4),
    'STGNN_CI_Upper': round(stgnn_ci[1], 4),
    'Wilcoxon_p_value': round(p_value, 4) if p_value is not None else None
}])
if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (AUC-PR)
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')
max_y = max(auc_scores); plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')
plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()


## PHASE 6B: Feature Importance & SHAP-Equivalent Analysis
**Methodology:** To determine which input features actually drive STPC predictions, the Champion Random Forest model's built-in feature importances are extracted. Since the RF operates on flattened `(Time_Steps × Nodes × Features)` vectors, importances are aggregated back across the 4 original feature dimensions to produce a per-feature contribution score.

This analysis answers a critical scientific question: does the model rely on **accident density** (Total_Accidents), **incident severity** (Mean_Severity), **road clearance time** (Mean_Distance), or **weather friction** (Mean_Humidity) to identify STPCs? The result directly informs the theoretical interpretation of the shockwave propagation mechanism.

In [ ]:
# ==============================================================================
# PHASE 6B: FEATURE IMPORTANCE ANALYSIS (Random Forest)
# ==============================================================================
print(f"[*] Extracting Feature Importances from Champion Random Forest...")

_, rf_importance_model = build_traditional_ml_baselines()
X_tr_ml_fi = X_tr.reshape(X_tr.shape[0], -1)
rf_importance_model.fit(X_tr_ml_fi, y_tr)

all_importances = np.array([est.feature_importances_ for est in rf_importance_model.estimators_])
mean_importances = all_importances.mean(axis=0)

features_list = FEATURES_LIST
n_features = len(features_list)
feature_contributions = np.zeros(n_features)
for feat_idx in range(n_features):
    feature_contributions[feat_idx] = mean_importances[feat_idx::n_features].sum()
feature_contributions /= feature_contributions.sum()

fig, ax = plt.subplots(figsize=(8, 5))
colors_fi = ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641']
bars_fi = ax.barh(features_list, feature_contributions, color=colors_fi, edgecolor='black', alpha=0.88)
ax.set_xlabel('Aggregated Feature Importance (Normalised)', fontsize=11, fontweight='bold')
ax.set_title(f'RF Feature Importance — STPC Prediction Drivers\nCity: {TARGET_CITY.upper()} | Champion Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars_fi, feature_contributions):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
FI_PATH = f"{BASE_DIR}/{TARGET_CITY}_Feature_Importance.png"
plt.savefig(FI_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] Feature Importance artifact saved to: {FI_PATH}")
plt.show()

del rf_importance_model, X_tr_ml_fi, all_importances; gc.collect()

## PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
**Methodology:** To visually validate the necessity of the Spatio-Temporal Graph Neural Network (ST-GNN), this phase maps the empirical ground truth of the "Butterfly Effect" within the target urban morphology.
By isolating the highest-density temporal window (the "Worst Day"), we plot the primary physical triggers (Primary Accidents) against the resulting friction cascades (Secondary Accidents). This visual artifact provides spatial evidence of the Champion Topology discovered in Phase 5, displaying the true physical spread of the traffic shockwave across the target urban morphology.

In [ ]:
#==============================================================================
# PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
# ==============================================================================
print(f"[*] Initializing Phase 7: Spatial Morphology Visualization for {TARGET_CITY.upper()}...")

df_map = df_target.copy()
df_map['Start_Time'] = pd.to_datetime(df_map['Start_Time'])
daily_secondary_counts = df_map[df_map[f'Is_STPC_{CHAMP_RADIUS}'] == 1].groupby(df_map['Start_Time'].dt.date).size()
worst_day = daily_secondary_counts.idxmax()
print(f"[*] Isolating Maximum Density Temporal Window: {worst_day}")

day_data = df_map[df_map['Start_Time'].dt.date == worst_day]
primary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 0]
secondary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 1]

num_primary = len(primary_accidents); num_secondary = len(secondary_accidents)
print(f"[*] Plotted Graph Nodes: {num_primary} Primary Triggers | {num_secondary} Secondary Cascades")

center_lat = day_data['Start_Lat'].mean(); center_lng = day_data['Start_Lng'].mean()
road_map = folium.Map(location=[center_lat, center_lng], zoom_start=11, control_scale=True)

for _, row in primary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=5, color='#3498db', fill=True, fill_color='#3498db', fill_opacity=0.6,
                        tooltip="Primary Accident (Isolated Trigger)").add_to(road_map)
for _, row in secondary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=8, color='black', weight=1.5, fill=True, fill_color='#e74c3c', fill_opacity=0.9,
                        tooltip=f"STPC (Proximate Collision Cascade) (Within {CHAMP_RADIUS}mi Shockwave)").add_to(road_map)

legend_html = f'''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 320px; height: auto;
    background-color: white; border: 2px solid grey; z-index: 9999;
    font-size: 14px; font-family: Arial, sans-serif;
    padding: 15px; box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
    ">
    <h4 style="margin-top: 0; margin-bottom: 5px; font-weight: bold; font-size: 15px;">
        Empirical Spatial Morphology
    </h4>
    <hr style="margin: 5px 0;">
    <b>City:</b> {TARGET_CITY.upper()}<br>
    <b>Temporal Window:</b> {worst_day}<br>
    <b>Champion Topology:</b> {CHAMP_RADIUS}mi Radius<br>
    <hr style="margin: 10px 0;">
    <div style="margin-bottom: 8px;">
        <span style="display: inline-block; width: 12px; height: 12px; background-color: #3498db; border-radius: 50%; margin-right: 8px; opacity: 0.6;"></span>
        <b>Primary Trigger</b> (n={num_primary})
    </div>
    <div>
        <span style="display: inline-block; width: 14px; height: 14px; background-color: #e74c3c; border: 2px solid black; border-radius: 50%; margin-right: 8px;"></span>
        <b>Secondary Cascade</b> (n={num_secondary})
    </div>
</div>
'''
road_map.get_root().html.add_child(Element(legend_html))
output_html = f'{BASE_DIR}/{TARGET_CITY}_Shockwave_Morphology_Map.html'
road_map.save(output_html)
print(f"\n==================================================================")
print(f"[SUCCESS] THESIS ARTIFACT GENERATED: {output_html}")
print(f"==================================================================")


from IPython.display import display
display(road_map)

##Phase 8: Temporal Drift & Calibration

In [ ]:
# ==============================================================================
# PHASE 8: TEMPORAL DRIFT & CALIBRATION (2020 Adversarial External Validation)
# ==============================================================================
print(f"[*] Initiating Phase 8: Temporal Drift Protocol for {TARGET_CITY.upper()}...")

import statsmodels.api as sm
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def build_st_graph_tensors_temporal_external(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4,
                                             cutoff=pd.Timestamp('2020-01-01')):
    """
    Fit KMeans on pre-cutoff (train/val) UTM coordinates, assign all data to regions,
    build adjacency on training cluster centers, then construct hourly tensors and
    return (X_tr, X_val) from pre-2020 and (X_te_2020) from 2020, with purge & scaling.
    """
    df = df_raw.copy().sort_values('Start_Time').reset_index(drop=True)
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]

    train_mask = df['Start_Time'] < cutoff
    test_mask  = df['Start_Time'].dt.year == 2020

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    kmeans.fit(df.loc[train_mask, ['_X','_Y']].values)
    df['Region_ID'] = kmeans.predict(df[['_X','_Y']].values)

    centers_utm = kmeans.cluster_centers_
    adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    X, y, t_end = [], [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len]); y.append(city_targets[i+seq_len]); t_end.append(full_time[i+seq_len])
    X, y, t_end = np.array(X), np.array(y), np.array(t_end)

    dev_idx = np.where(t_end < cutoff)[0]
    test_idx = np.where((t_end >= pd.Timestamp('2020-01-01')) & (t_end < pd.Timestamp('2021-01-01')))[0]
    if len(dev_idx) == 0 or len(test_idx) == 0:
        raise ValueError("[ERROR] Not enough samples in dev or 2020 windows.")

    dev_X, dev_y = X[dev_idx], y[dev_idx]
    total_dev = dev_X.shape[0]
    train_bound = int(total_dev * 0.82)  # internal dev split (train/val)
    purge = seq_len

    X_tr_raw, y_tr = dev_X[:train_bound], dev_y[:train_bound]
    X_val_raw, y_val = dev_X[train_bound + purge:], dev_y[train_bound + purge:]
    X_te_raw,  y_te  = X[test_idx], y[test_idx]

    num_features = X_tr_raw.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_tr_raw.reshape(-1, num_features)).reshape(X_tr_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te  = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, adj

X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext = build_st_graph_tensors_temporal_external(
    df_target, CHAMP_NODES, CHAMP_RADIUS, CHAMP_K, seq_len=SEQ_LENGTH, cutoff=pd.Timestamp('2020-01-01')
)
print(f"[*] Validation Era Prevalence (Pre-2020) : {float(y_val_ext.mean()):.4f}")
print(f"[*] Adversarial Era Prevalence (2020)    : {float(y_te_2020.mean()):.4f}")

pos_ext = y_tr_ext.sum(); neg_ext = (y_tr_ext.size - pos_ext)
pos_weight_ext = float(neg_ext / max(pos_ext, 1))

model_drift = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr_ext.shape[-1], adj_ext, pos_weight_ext)
model_drift.fit(X_tr_ext, y_tr_ext, validation_data=(X_val_ext, y_val_ext), epochs=15, batch_size=64, verbose=0)

y_pred_probs_2020 = model_drift.predict(X_te_2020, verbose=0).flatten()
y_true_2020 = y_te_2020.flatten()

def bootstrap_pr_auc(y_true, y_pred, n_bootstraps=500):
    scores = []
    N = len(y_pred)
    for _ in range(n_bootstraps):
        idx = np.random.randint(0, N, N)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(average_precision_score(y_true[idx], y_pred[idx]))
    if len(scores) == 0:
        return np.nan, np.nan
    return np.percentile(scores, [2.5, 97.5])

drift_auc_pr = average_precision_score(y_true_2020, y_pred_probs_2020)
boot_ci_lo, boot_ci_hi = bootstrap_pr_auc(y_true_2020, y_pred_probs_2020)
drift_brier = brier_score_loss(y_true_2020, y_pred_probs_2020)

def calibration_slope_intercept(y_true, p_hat):
    eps = 1e-6
    logit_ph = np.log(np.clip(p_hat, eps, 1-eps) / np.clip(1-p_hat, eps, 1-eps))
    X = sm.add_constant(logit_ph)  # intercept + slope
    model = sm.GLM(y_true, X, family=sm.families.Binomial())
    res = model.fit()
    return float(res.params[0]), float(res.params[1])

cal_intercept_2020, cal_slope_2020 = calibration_slope_intercept(y_true_2020, y_pred_probs_2020)

print("\n========================================================")
print("[*] TEMPORAL DRIFT & CALIBRATION ASSESSMENT (2020)")
print("========================================================")
print(f"Ranking Retention (PR-AUC) : {drift_auc_pr:.4f} (95% CI: [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}])")
print(f"Calibration Brier Score    : {drift_brier:.4f}")
print(f"Calibration Intercept      : {cal_intercept_2020:.3f} (Deviation from 0 = Risk Shift)")
print(f"Calibration Slope          : {cal_slope_2020:.3f} (Deviation from 1 = Over/Underfitting)")

prob_true, prob_pred = calibration_curve(y_true_2020, y_pred_probs_2020, n_bins=10, strategy='quantile')
bins = np.percentile(y_pred_probs_2020, np.linspace(0, 100, 11))
bin_indices = np.digitize(y_pred_probs_2020, bins[1:-1])
bin_counts = np.bincount(bin_indices, minlength=10)

print("\n[*] Reliability Curve Binning Sanity Check:")
for idx, count in enumerate(bin_counts):
    print(f"    -> Bin {idx+1}: n = {int(count)} samples")
if np.min(bin_counts) < 50:
    print("    [WARNING] Some bins have < 50 samples. Consider reducing n_bins for smoother calibration estimates.")

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='ST-GNN (2020 Adversarial Test)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.title(f"Reliability Curve: Structural Integrity vs Baseline Shift\nCity: {TARGET_CITY.upper()} (2020)")
plt.xlabel("Mean Predicted Probability"); plt.ylabel("Fraction of Actual STPCs")
plt.legend(); plt.grid(True); plt.show()

# Chicago
(The Rigid Grid): A hyper-dense, highly structured urban grid.

# Experiment A ( fixed-topology transferability from LA to Chicago)

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Chicago"  # Options: "Los Angeles", "Chicago", "Miami", "Indianapolis", "Iowa"

# Hyperparameter Search Space (Sensitivity Analysis)
SEARCH_NODES = [100]       # MAUP scale sensitivity (node count)
SEARCH_RADII = [2.0]  # Physical interaction horizons (miles)
SEARCH_K = [3]            # Intersection routing topologies
SEQ_LENGTH = 4                  # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena...")

experiment_results = []
RESULTS_PATH = f"{BASE_DIR}/ST_GNN_Results_{TARGET_CITY}.csv"
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")
            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )
                total_pos = np.sum(y_tr); total_neg = (y_tr.size) - total_pos
                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1; continue
                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH, n_nodes=n_nodes, n_features=X_tr.shape[-1],
                    adj_matrix=adj, pos_weight=pos_weight_val
                )
                print("[*] Commencing Training...")
                history = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_val),
                    epochs=15, batch_size=64, callbacks=[early_stop], verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten(); y_pred_flat = y_pred_probs_val.flatten()
                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]; final_recall = recalls[best_idx]
                    final_f1 = 2*(final_precision*final_recall)/(final_precision+final_recall) if (final_precision+final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(auc_pr, 4),
                    'Optimal_Threshold': round(optimal_threshold, 4),
                    'Precision_at_70_Recall': round(final_precision, 4),
                    'Actual_Recall': round(final_recall, 4),
                    'Val_F1_Score': round(final_f1, 4)
                })
                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            tf.keras.backend.clear_session(); gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")


## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# 1) FROZEN LA CHAMPION TOPOLOGY (Experiment A: Transferability Test)
CHAMP_NODES = 100
CHAMP_RADIUS = 2.0
CHAMP_K = 3
print(f"[*] Frozen LA Topology Applied to {TARGET_CITY.upper()}: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
# 2) ISOLATE THE FINAL TENSORS (Champion)
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)
total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)
y_val_flat = y_val.flatten(); y_te_flat = y_te.flatten()

# 3) TRADITIONAL ML BASELINES (LR & RF) with SafeNodeClassifier
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1); X_te_ml = X_te.reshape(X_te.shape[0], -1)
lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)
    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        # SafeNodeClassifier always returns 2-col proba; guard is retained
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0
    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5
    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)
    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")
del X_tr_ml, X_te_ml, lr_model, rf_model; gc.collect()

# 4) DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
early_stop_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_final], verbose=0)
    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(auc_lstm); lstm_f1_scores.append(f1_lstm)
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_final], verbose=0)
    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(auc_stgnn); stgnn_f1_scores.append(f1_stgnn)
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

lstm_auc_mean,  lstm_auc_std  = np.mean(lstm_auc_scores),  np.std(lstm_auc_scores)
stgnn_auc_mean, stgnn_auc_std = np.mean(stgnn_auc_scores), np.std(stgnn_auc_scores)
lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")
    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This can occur when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/Thesis_Final_Proof_Matrix.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'LR_AUC': round(lr_auc, 4),
    'RF_AUC': round(rf_auc, 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(lstm_ci[0], 4),
    'LSTM_CI_Upper': round(lstm_ci[1], 4),
    'STGNN_CI_Lower': round(stgnn_ci[0], 4),
    'STGNN_CI_Upper': round(stgnn_ci[1], 4),
    'Wilcoxon_p_value': round(p_value, 4) if p_value is not None else None
}])
if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (AUC-PR)
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')
max_y = max(auc_scores); plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')
plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()


#Experiment B (The Topological Optimization)

Experiment B: Native Topology Discovery in Chicago

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Chicago"

# Experiment B: Unified Urban Search Space (Autonomous Discovery)
# We expand the radii and K-neighbors to allow the model to discover
# Chicago's dense, grid-like grid morphology versus LA's sprawling web.
SEARCH_NODES = [100, 250, 500]
SEARCH_RADII = [0.5, 1.0, 2.0]
SEARCH_K = [3, 5, 8]
SEQ_LENGTH = 4                      # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: EXPERIMENT B - NATIVE TOPOLOGY DISCOVERY (The Challenger Arena)
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena (Native Search)...")

experiment_results = []

# SCIENTIFIC FIX: Rename output path to explicitly protect Experiment A data
RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Arena_Results.csv"

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")

            try:
                # 1. Generate Environment
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )

                # 2. Imbalance Math
                total_pos = np.sum(y_tr)
                total_neg = (y_tr.size) - total_pos

                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1
                    continue

                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                # 3. Build & Compile Model
                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH,
                    n_nodes=n_nodes,
                    n_features=X_tr.shape[-1],
                    adj_matrix=adj,
                    pos_weight=pos_weight_val
                )

                # 4. Train Model
                print("[*] Commencing Training...")

                # SCIENTIFIC FIX: Fresh initialization per experiment to prevent memory/state leaks
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                history = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=15,
                    batch_size=64,
                    callbacks=[early_stop],
                    verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                # 5. Scientific Evaluation (AUC-PR on Validation Vault)
                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)

                y_val_flat = y_val.flatten()
                y_pred_flat = y_pred_probs_val.flatten()

                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]
                    final_recall = recalls[best_idx]
                    final_f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                # 6. Log Results
                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(float(auc_pr), 4),
                    'Optimal_Threshold': round(float(optimal_threshold), 4),
                    'Precision_at_70_Recall': round(float(final_precision), 4),
                    'Actual_Recall': round(float(final_recall), 4),
                    'Val_F1_Score': round(float(final_f1), 4)
                })

                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            # 7. Aggressive Cleanup to protect Colab RAM

            if 'early_stop' in locals(): del early_stop
            if 'model' in locals(): del model
            if 'history' in locals(): del history
            if 'y_pred_probs_val' in locals(): del y_pred_probs_val
            if 'X_tr' in locals(): del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")

## PHASE 5B: MAUP Sensitivity Analysis
**Methodology:** The Modifiable Areal Unit Problem (MAUP) states that results derived from spatially aggregated data are sensitive to the choice of areal unit — in this case, the number of K-Means macro-nodes. A result that only holds for one node count is not robust.

This section loads the Phase 5 arena results and produces a sensitivity plot comparing AUC-PR across both node configurations (100 and 500) for each radius. A stable model should show consistent relative rankings regardless of spatial granularity.

In [ ]:
# ==============================================================================
# PHASE 5B: MAUP SENSITIVITY ANALYSIS (Scale)
# ==============================================================================
print("[*] Running MAUP Sensitivity Analysis...")

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")
arena_results_df = pd.read_csv(RESULTS_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'MAUP Sensitivity Analysis — Effect of Spatial Granularity on AUC-PR\nCity: {TARGET_CITY.upper()}',
             fontsize=13, fontweight='bold')
node_colors = NODE_COLORS

for ax_idx, radius in enumerate(SEARCH_RADII):
    ax = axes[ax_idx]
    subset = arena_results_df[arena_results_df['Radius_mi'] == radius].copy()
    for n_nodes_val, grp in subset.groupby('Nodes'):
        grp_sorted = grp.sort_values('K_Neighbors')
        ax.plot(
            grp_sorted['K_Neighbors'], grp_sorted['Val_AUC_PR'],
            marker='o', linewidth=2, color=node_colors.get(n_nodes_val, 'grey'),
            label=f'{n_nodes_val} Nodes'
        )
    ax.set_title(f'Radius = {radius}mi', fontweight='bold')
    ax.set_xlabel('K Neighbors', fontweight='bold'); ax.set_ylabel('Validation AUC-PR', fontweight='bold')
    ax.set_xticks(SEARCH_K); ax.legend(title='Node Count'); ax.grid(True, alpha=0.3)

plt.tight_layout()
MAUP_PATH = f"{BASE_DIR}/{TARGET_CITY}_MAUP_Sensitivity.png"
plt.savefig(MAUP_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] MAUP Sensitivity artifact saved to: {MAUP_PATH}")
plt.show()

print("\n[*] AUC-PR by Node Count (Mean across all K and Radius configurations):")
maup_summary = arena_results_df.groupby('Nodes')['Val_AUC_PR'].agg(['mean','std','min','max'])
print(maup_summary.round(4).to_string())


##Phase 5C: The MAUP Zonation Protocol

In [ ]:
# ==============================================================================
# PHASE 5C: MAUP ZONATION SENSITIVITY (Champion-Based)
# ==============================================================================
print(f"[*] Initiating Phase 5C: MAUP Zonation Stability Protocol for {TARGET_CITY.upper()}...")

# ----------------------------------------------------------------------
# 5C.0 Load Phase 5 arena results explicitly (independent of Phase 5B)
# ----------------------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_results_df = pd.read_csv(RESULTS_PATH)
if arena_results_df.empty:
    raise ValueError("[ERROR] Arena results file is empty. No valid configurations available for Phase 5C.")

# ----------------------------------------------------------------------
# 5C.1 Anchor Phase 5C to the ACTUAL champion discovered in Phase 5
# ----------------------------------------------------------------------
champion_row = arena_results_df.sort_values('Val_AUC_PR', ascending=False).iloc[0]

FIXED_NODES = int(champion_row['Nodes'])
FIXED_RADIUS = float(champion_row['Radius_mi'])
FIXED_K = int(champion_row['K_Neighbors'])

print(f"[*] Zonation test anchored at CHAMPION topology: "
      f"N={FIXED_NODES}, Radius={FIXED_RADIUS}mi, K={FIXED_K}")

# ----------------------------------------------------------------------
# 5C.2 Chronological sort to preserve time integrity
# ----------------------------------------------------------------------
df_target['Start_Time'] = pd.to_datetime(df_target['Start_Time'])
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)

# ----------------------------------------------------------------------
# 5C.3 Dynamic UTM projection for metric-consistent zonation
# ----------------------------------------------------------------------
transformer, epsg_code = _dynamic_utm_transformer(df_target)
utm_coords = _project_to_utm(df_target, transformer)

# ----------------------------------------------------------------------
# 5C.4 Window-aligned train mask (perfectly aligned with _windows_and_split)
# ----------------------------------------------------------------------
start_min = df_target['Start_Time'].min().floor('h')
end_max   = df_target['Start_Time'].max().floor('h')
full_time = pd.date_range(start=start_min, end=end_max, freq='h')

total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)

if len(full_time) == 0 or total_windows == 0:
    raise ValueError("[ERROR] No temporal windows available for zonation protocol.")

cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]
train_mask = df_target['Start_Time'] < cutoff_time
utm_train = utm_coords[train_mask.values]

if utm_train.shape[0] < FIXED_NODES:
    raise ValueError(
        f"[ERROR] Insufficient train-era accident points ({utm_train.shape[0]}) "
        f"to fit KMeans with n_clusters={FIXED_NODES}."
    )

# ----------------------------------------------------------------------
# 5C.5 K-Means zonation stability loop
# ----------------------------------------------------------------------
ZONATION_SEEDS = 20
zonation_auc_scores = []
test_prevalence = None

print(f"[*] Executing {ZONATION_SEEDS} K-Means Spatial Partitions "
      f"(Train-Era Fitted, UTM={epsg_code})...")

for s in range(ZONATION_SEEDS):
    print(f"    -> Zonation Seed {s+1}/{ZONATION_SEEDS}")

    try:
        # Best-effort reproducibility
        random.seed(42 + s)
        np.random.seed(42 + s)
        tf.random.set_seed(42 + s)

        kmeans = KMeans(n_clusters=FIXED_NODES, random_state=s, n_init=10)
        kmeans.fit(utm_train)  # fit ONLY on train-era coordinates
        df_target['Region_ID'] = kmeans.predict(utm_coords)

        X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
            df_target,
            n_nodes=FIXED_NODES,
            radius_mi=FIXED_RADIUS,
            k_neighbors=FIXED_K,
            seq_len=SEQ_LENGTH
        )

        node_degrees = adj.sum(axis=1)
        if not (node_degrees > 0).all():
            isolated_count = int((node_degrees == 0).sum())
            raise ValueError(f"[CRITICAL FAILURE] Adjacency contains {isolated_count} isolated nodes at Seed {s}.")

        if test_prevalence is None:
            test_prevalence = float(y_te.mean())

        pos = y_tr.sum()
        neg = (y_tr.size - pos)
        pos_weight_val = float(neg / max(pos, 1))

        # SCIENTIFIC FIX: Callback initialized INSIDE the loop so it resets its memory state per run
        early_stop_zonation = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=64,
            callbacks=[early_stop_zonation],
            verbose=0
        )

        y_pred = model.predict(X_te, verbose=0).flatten()
        auc_pr = average_precision_score(y_te.flatten(), y_pred)
        zonation_auc_scores.append(float(auc_pr))

        # Cleanup
        del model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, y_pred, early_stop_zonation
        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:
        print(f"    [WARNING] Seed {s} failed: {e}")
        tf.keras.backend.clear_session()
        gc.collect()
        continue

if len(zonation_auc_scores) == 0:
    raise RuntimeError("[CRITICAL] All K-Means zonation runs failed. Cannot compute zonation robustness.")

# ----------------------------------------------------------------------
# 5C.6 Dynamic H3 method binding (Colab/runtime-safe)
# ----------------------------------------------------------------------
h3_encode = getattr(h3, "latlng_to_cell", getattr(h3, "geo_to_h3", None))
h3_decode = getattr(h3, "cell_to_latlng", getattr(h3, "h3_to_geo", None))

if h3_encode is None or h3_decode is None:
    raise ImportError("[CRITICAL FAILURE] Unrecognized H3 library version. Encode/decode methods missing.")

print("\n[*] Executing Alternate Zoning Family: H3 Hexagonal Super-Regions (Train-Era Calibrated)...")

def choose_h3_resolution(df_like, target=100, tol=5, rmin=5, rmax=10):
    best = None
    for res in range(rmin, rmax + 1):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        if abs(n - target) <= tol:
            return res, ids
        if best is None or abs(n - target) < abs(best[0] - target):
            best = (n, res, ids)
    return best[1], best[2]

# Use the exact same window-aligned train mask
h3_res, _ = choose_h3_resolution(df_target[train_mask], target=FIXED_NODES)
h3_ids_full = df_target.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], h3_res), axis=1)
n_hex = h3_ids_full.nunique()

if n_hex != FIXED_NODES:
    print(f"    -> H3 produced {n_hex} hexes; target is {FIXED_NODES} super-regions...")

    uniq_hex = pd.Series(h3_ids_full.astype(str).values).unique()
    n_hex_nodes = len(uniq_hex)

    if n_hex_nodes < 2:
        raise ValueError("[CRITICAL FAILURE] H3 produced fewer than 2 unique hexes; spatial connectivity graph cannot be built.")

    # SCIENTIFIC FIX: You can only aggregate DOWN, not UP.
    if n_hex_nodes < FIXED_NODES:
        raise ValueError(
            f"[CRITICAL FAILURE] H3 produced only {n_hex_nodes} unique hexes, "
            f"which is strictly fewer than the target {FIXED_NODES}. "
            f"Cannot aggregate into more super-regions than available hexes. "
            f"Increase H3 resolution search range (rmax) or revise choose_h3_resolution()."
        )

    print(f"    -> Aggregating {n_hex_nodes} H3 hexes into exactly {FIXED_NODES} contiguous super-regions...")

    hex_latlon = np.array([h3_decode(h) for h in uniq_hex])  # (lat, lng)

    # PyProj transformer with always_xy=True expects (lon, lat)
    hex_x, hex_y = transformer.transform(hex_latlon[:, 1], hex_latlon[:, 0])

    safe_k = min(6, n_hex_nodes - 1)

    spatial_connectivity = kneighbors_graph(
        np.c_[hex_x, hex_y],
        n_neighbors=safe_k,
        include_self=False
    )

    agg_cluster = AgglomerativeClustering(
        n_clusters=FIXED_NODES,
        connectivity=spatial_connectivity,
        linkage='ward'
    )
    h_labels = agg_cluster.fit_predict(np.c_[hex_x, hex_y])
    hex_to_super = {h: int(l) for h, l in zip(uniq_hex, h_labels)}

    df_target['Region_ID'] = pd.Series(h3_ids_full.astype(str).values).map(hex_to_super).values

else:
    df_target['Region_ID'] = h3_ids_full.astype('category').cat.codes

# ----------------------------------------------------------------------
# 5C.7 Evaluate H3 zoning family under same champion topology
# ----------------------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
    df_target,
    n_nodes=FIXED_NODES,
    radius_mi=FIXED_RADIUS,
    k_neighbors=FIXED_K,
    seq_len=SEQ_LENGTH
)

node_degrees = adj.sum(axis=1)
if not (node_degrees > 0).all():
    isolated_count = int((node_degrees == 0).sum())
    raise ValueError(f"[CRITICAL FAILURE] H3 adjacency contains {isolated_count} isolated nodes.")

pos = y_tr.sum()
neg = (y_tr.size - pos)
pos_weight_val = float(neg / max(pos, 1))

early_stop_h3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

h3_model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
h3_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop_h3],
    verbose=0
)

h3_pred = h3_model.predict(X_te, verbose=0).flatten()
h3_auc_pr = average_precision_score(y_te.flatten(), h3_pred)

# ----------------------------------------------------------------------
# 5C.8 Summarize robustness statistics
# ----------------------------------------------------------------------
from scipy.stats import percentileofscore

mean_auc = float(np.mean(zonation_auc_scores))
sd_auc = float(np.std(zonation_auc_scores))
cv_percent = (sd_auc / mean_auc) * 100 if mean_auc > 0 else 0.0
q_lo, q_hi = np.quantile(zonation_auc_scores, [0.025, 0.975])
h3_pct = percentileofscore(zonation_auc_scores, h3_auc_pr)

print("\n========================================================")
print("[*] ZONATION MAUP RESULTS")
print("========================================================")
print(f"Champion Topology Tested     : {FIXED_NODES} Nodes | {FIXED_RADIUS}mi | K={FIXED_K}")
print(f"Test-Set Prevalence Baseline : {float(test_prevalence):.4f}")
print(f"K-Means PR-AUC (n={len(zonation_auc_scores)}) : {mean_auc:.4f} ± {sd_auc:.4f}")
print(f"95% Empirical Quantile CI    : [{q_lo:.4f}, {q_hi:.4f}]")
print(f"Coefficient of Variation (CV): {cv_percent:.2f}%")
print(f"H3 Super-Region PR-AUC       : {h3_auc_pr:.4f}")
print(f"H3 Percentile within K-Means : {h3_pct:.1f}th percentile")

# ----------------------------------------------------------------------
# 5C.9 Save zonation results
# ----------------------------------------------------------------------
ZONATION_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Zonation_Results.csv"

# SCIENTIFIC FIX: Log the actual number of successful runs, not the attempted target.
zonation_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': FIXED_NODES,
    'Champion_Radius_mi': FIXED_RADIUS,
    'Champion_K': FIXED_K,
    'KMeans_Zonation_Runs': len(zonation_auc_scores),
    'KMeans_AUC_Mean': round(mean_auc, 4),
    'KMeans_AUC_Std': round(sd_auc, 4),
    'KMeans_AUC_Q2.5': round(float(q_lo), 4),
    'KMeans_AUC_Q97.5': round(float(q_hi), 4),
    'KMeans_CV_Percent': round(float(cv_percent), 2),
    'H3_AUC_PR': round(float(h3_auc_pr), 4),
    'H3_Percentile_Within_KMeans': round(float(h3_pct), 2)
}])

zonation_df.to_csv(ZONATION_RESULTS_PATH, index=False)
print(f"[SUCCESS] Zonation robustness results saved to: {ZONATION_RESULTS_PATH}")

# Cleanup
del h3_model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, h3_pred, early_stop_h3
tf.keras.backend.clear_session()
gc.collect()

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# ---------------------------------------------------------
# 1. DYNAMICALLY EXTRACT CHAMPION (Sorted by Val_AUC_PR)
# ---------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]

CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# ---------------------------------------------------------
# 2. ISOLATE THE FINAL TENSORS
# ---------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set. "
                     "Re-run Phase 5 — the arena may have selected a degenerate configuration.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)

y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# ---------------------------------------------------------
# 3. TRADITIONAL ML BASELINES (Logistic Regression & Random Forest)
# ---------------------------------------------------------
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")

X_tr_ml = X_tr.reshape(X_tr.shape[0], -1)
X_te_ml = X_te.reshape(X_te.shape[0], -1)

lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)

    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0

    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)

    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")

del X_tr_ml, X_te_ml, lr_model, rf_model
gc.collect()

# ---------------------------------------------------------
# 4. DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
# ---------------------------------------------------------
N_RUNS = 10

print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    # --- BASELINE LSTM ---
    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)

    # SCIENTIFIC FIX: Fresh callback instantiated inside the loop
    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_lstm], verbose=0)

    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)

    lstm_auc_scores.append(float(auc_lstm))
    lstm_f1_scores.append(float(f1_lstm))
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")

    del lstm_model, early_stop_lstm
    tf.keras.backend.clear_session()
    gc.collect()

    # --- CHAMPION ST-GNN ---
    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)

    # SCIENTIFIC FIX: Fresh callback instantiated inside the loop
    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_stgnn], verbose=0)

    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)

    stgnn_auc_scores.append(float(auc_stgnn))
    stgnn_f1_scores.append(float(f1_stgnn))
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")

    del stgnn_model, early_stop_stgnn
    tf.keras.backend.clear_session()
    gc.collect()

# ---------------------------------------------------------
# 5. SCIENTIFIC LOGGING & HYPOTHESIS TESTING
# ---------------------------------------------------------
lstm_auc_mean,  lstm_auc_std  = float(np.mean(lstm_auc_scores)),  float(np.std(lstm_auc_scores))
stgnn_auc_mean, stgnn_auc_std = float(np.mean(stgnn_auc_scores)), float(np.std(stgnn_auc_scores))

lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")

    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This typically occurs when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Final_Proof.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'Champion_K': CHAMP_K,
    'LR_AUC': round(float(lr_auc), 4),
    'RF_AUC': round(float(rf_auc), 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(float(lstm_ci[0]), 4),
    'LSTM_CI_Upper': round(float(lstm_ci[1]), 4),
    'STGNN_CI_Lower': round(float(stgnn_ci[0]), 4),
    'STGNN_CI_Upper': round(float(stgnn_ci[1]), 4),
    'Wilcoxon_p_value': round(float(p_value), 4) if p_value is not None else None
}])

if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)

print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')

max_y = max(auc_scores)
plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom',
                fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()

## PHASE 6B: Feature Importance & SHAP-Equivalent Analysis
**Methodology:** To determine which input features actually drive STPC predictions, the Champion Random Forest model's built-in feature importances are extracted. Since the RF operates on flattened `(Time_Steps × Nodes × Features)` vectors, importances are aggregated back across the 4 original feature dimensions to produce a per-feature contribution score.

This analysis answers a critical scientific question: does the model rely on **accident density** (Total_Accidents), **incident severity** (Mean_Severity), **road clearance time** (Mean_Distance), or **weather friction** (Mean_Humidity) to identify STPCs? The result directly informs the theoretical interpretation of the shockwave propagation mechanism.

In [ ]:
# ==============================================================================
# PHASE 6B: FEATURE IMPORTANCE ANALYSIS (Random Forest)
# ==============================================================================
print(f"[*] Extracting Feature Importances from Champion Random Forest...")

_, rf_importance_model = build_traditional_ml_baselines()
X_tr_ml_fi = X_tr.reshape(X_tr.shape[0], -1)
rf_importance_model.fit(X_tr_ml_fi, y_tr)

all_importances = np.array([est.feature_importances_ for est in rf_importance_model.estimators_])
mean_importances = all_importances.mean(axis=0)

features_list = FEATURES_LIST
n_features = len(features_list)
feature_contributions = np.zeros(n_features)
for feat_idx in range(n_features):
    feature_contributions[feat_idx] = mean_importances[feat_idx::n_features].sum()
feature_contributions /= feature_contributions.sum()

fig, ax = plt.subplots(figsize=(8, 5))
colors_fi = ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641']
bars_fi = ax.barh(features_list, feature_contributions, color=colors_fi, edgecolor='black', alpha=0.88)
ax.set_xlabel('Aggregated Feature Importance (Normalised)', fontsize=11, fontweight='bold')
ax.set_title(f'RF Feature Importance — STPC Prediction Drivers\nCity: {TARGET_CITY.upper()} | Champion Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars_fi, feature_contributions):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
FI_PATH = f"{BASE_DIR}/{TARGET_CITY}_Feature_Importance.png"
plt.savefig(FI_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] Feature Importance artifact saved to: {FI_PATH}")
plt.show()

del rf_importance_model, X_tr_ml_fi, all_importances; gc.collect()

## PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
**Methodology:** To visually validate the necessity of the Spatio-Temporal Graph Neural Network (ST-GNN), this phase maps the empirical ground truth of the "Butterfly Effect" within the target urban morphology.
By isolating the highest-density temporal window (the "Worst Day"), we plot the primary physical triggers (Primary Accidents) against the resulting friction cascades (Secondary Accidents). This visual artifact provides spatial evidence of the Champion Topology discovered in Phase 5, displaying the true physical spread of the traffic shockwave across the target urban morphology.

In [ ]:
#==============================================================================
# PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
# ==============================================================================
print(f"[*] Initializing Phase 7: Spatial Morphology Visualization for {TARGET_CITY.upper()}...")

df_map = df_target.copy()
df_map['Start_Time'] = pd.to_datetime(df_map['Start_Time'])
daily_secondary_counts = df_map[df_map[f'Is_STPC_{CHAMP_RADIUS}'] == 1].groupby(df_map['Start_Time'].dt.date).size()
worst_day = daily_secondary_counts.idxmax()
print(f"[*] Isolating Maximum Density Temporal Window: {worst_day}")

day_data = df_map[df_map['Start_Time'].dt.date == worst_day]
primary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 0]
secondary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 1]

num_primary = len(primary_accidents); num_secondary = len(secondary_accidents)
print(f"[*] Plotted Graph Nodes: {num_primary} Primary Triggers | {num_secondary} Secondary Cascades")

center_lat = day_data['Start_Lat'].mean(); center_lng = day_data['Start_Lng'].mean()
road_map = folium.Map(location=[center_lat, center_lng], zoom_start=11, control_scale=True)

for _, row in primary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=5, color='#3498db', fill=True, fill_color='#3498db', fill_opacity=0.6,
                        tooltip="Primary Accident (Isolated Trigger)").add_to(road_map)
for _, row in secondary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=8, color='black', weight=1.5, fill=True, fill_color='#e74c3c', fill_opacity=0.9,
                        tooltip=f"STPC (Proximate Collision Cascade) (Within {CHAMP_RADIUS}mi Shockwave)").add_to(road_map)

legend_html = f'''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 320px; height: auto;
    background-color: white; border: 2px solid grey; z-index: 9999;
    font-size: 14px; font-family: Arial, sans-serif;
    padding: 15px; box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
    ">
    <h4 style="margin-top: 0; margin-bottom: 5px; font-weight: bold; font-size: 15px;">
        Empirical Spatial Morphology
    </h4>
    <hr style="margin: 5px 0;">
    <b>City:</b> {TARGET_CITY.upper()}<br>
    <b>Temporal Window:</b> {worst_day}<br>
    <b>Champion Topology:</b> {CHAMP_RADIUS}mi Radius<br>
    <hr style="margin: 10px 0;">
    <div style="margin-bottom: 8px;">
        <span style="display: inline-block; width: 12px; height: 12px; background-color: #3498db; border-radius: 50%; margin-right: 8px; opacity: 0.6;"></span>
        <b>Primary Trigger</b> (n={num_primary})
    </div>
    <div>
        <span style="display: inline-block; width: 14px; height: 14px; background-color: #e74c3c; border: 2px solid black; border-radius: 50%; margin-right: 8px;"></span>
        <b>Secondary Cascade</b> (n={num_secondary})
    </div>
</div>
'''
road_map.get_root().html.add_child(Element(legend_html))
output_html = f'{BASE_DIR}/{TARGET_CITY}_Shockwave_Morphology_Map.html'
road_map.save(output_html)
print(f"\n==================================================================")
print(f"[SUCCESS] THESIS ARTIFACT GENERATED: {output_html}")
print(f"==================================================================")


from IPython.display import display
display(road_map)

##Phase 8: Temporal Drift & Calibration

In [ ]:
# ==============================================================================
# PHASE 8: TEMPORAL DRIFT & CALIBRATION (2020 Adversarial External Validation)
# ==============================================================================
print(f"[*] Initiating Phase 8: Temporal Drift Protocol for {TARGET_CITY.upper()}...")

import statsmodels.api as sm
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def build_st_graph_tensors_temporal_external(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4,
                                             cutoff=pd.Timestamp('2020-01-01')):
    """
    Fit KMeans on pre-cutoff (train/val) UTM coordinates, assign all data to regions,
    build adjacency on training cluster centers, then construct hourly tensors and
    return (X_tr, X_val) from pre-2020 and (X_te_2020) from 2020, with purge & scaling.
    """
    df = df_raw.copy().sort_values('Start_Time').reset_index(drop=True)
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]

    train_mask = df['Start_Time'] < cutoff
    test_mask  = df['Start_Time'].dt.year == 2020

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    kmeans.fit(df.loc[train_mask, ['_X','_Y']].values)
    df['Region_ID'] = kmeans.predict(df[['_X','_Y']].values)

    centers_utm = kmeans.cluster_centers_
    adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    X, y, t_end = [], [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len]); y.append(city_targets[i+seq_len]); t_end.append(full_time[i+seq_len])
    X, y, t_end = np.array(X), np.array(y), np.array(t_end)

    dev_idx = np.where(t_end < cutoff)[0]
    test_idx = np.where((t_end >= pd.Timestamp('2020-01-01')) & (t_end < pd.Timestamp('2021-01-01')))[0]
    if len(dev_idx) == 0 or len(test_idx) == 0:
        raise ValueError("[ERROR] Not enough samples in dev or 2020 windows.")

    dev_X, dev_y = X[dev_idx], y[dev_idx]
    total_dev = dev_X.shape[0]
    train_bound = int(total_dev * 0.82)  # internal dev split (train/val)
    purge = seq_len

    X_tr_raw, y_tr = dev_X[:train_bound], dev_y[:train_bound]
    X_val_raw, y_val = dev_X[train_bound + purge:], dev_y[train_bound + purge:]
    X_te_raw,  y_te  = X[test_idx], y[test_idx]

    num_features = X_tr_raw.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_tr_raw.reshape(-1, num_features)).reshape(X_tr_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te  = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, adj

X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext = build_st_graph_tensors_temporal_external(
    df_target, CHAMP_NODES, CHAMP_RADIUS, CHAMP_K, seq_len=SEQ_LENGTH, cutoff=pd.Timestamp('2020-01-01')
)
print(f"[*] Validation Era Prevalence (Pre-2020) : {float(y_val_ext.mean()):.4f}")
print(f"[*] Adversarial Era Prevalence (2020)    : {float(y_te_2020.mean()):.4f}")

pos_ext = y_tr_ext.sum(); neg_ext = (y_tr_ext.size - pos_ext)
pos_weight_ext = float(neg_ext / max(pos_ext, 1))

model_drift = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr_ext.shape[-1], adj_ext, pos_weight_ext)
model_drift.fit(X_tr_ext, y_tr_ext, validation_data=(X_val_ext, y_val_ext), epochs=15, batch_size=64, verbose=0)

y_pred_probs_2020 = model_drift.predict(X_te_2020, verbose=0).flatten()
y_true_2020 = y_te_2020.flatten()

def bootstrap_pr_auc(y_true, y_pred, n_bootstraps=500):
    scores = []
    N = len(y_pred)
    for _ in range(n_bootstraps):
        idx = np.random.randint(0, N, N)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(average_precision_score(y_true[idx], y_pred[idx]))
    if len(scores) == 0:
        return np.nan, np.nan
    return np.percentile(scores, [2.5, 97.5])

drift_auc_pr = average_precision_score(y_true_2020, y_pred_probs_2020)
boot_ci_lo, boot_ci_hi = bootstrap_pr_auc(y_true_2020, y_pred_probs_2020)
drift_brier = brier_score_loss(y_true_2020, y_pred_probs_2020)

def calibration_slope_intercept(y_true, p_hat):
    eps = 1e-6
    logit_ph = np.log(np.clip(p_hat, eps, 1-eps) / np.clip(1-p_hat, eps, 1-eps))
    X = sm.add_constant(logit_ph)  # intercept + slope
    model = sm.GLM(y_true, X, family=sm.families.Binomial())
    res = model.fit()
    return float(res.params[0]), float(res.params[1])

cal_intercept_2020, cal_slope_2020 = calibration_slope_intercept(y_true_2020, y_pred_probs_2020)

print("\n========================================================")
print("[*] TEMPORAL DRIFT & CALIBRATION ASSESSMENT (2020)")
print("========================================================")
print(f"Ranking Retention (PR-AUC) : {drift_auc_pr:.4f} (95% CI: [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}])")
print(f"Calibration Brier Score    : {drift_brier:.4f}")
print(f"Calibration Intercept      : {cal_intercept_2020:.3f} (Deviation from 0 = Risk Shift)")
print(f"Calibration Slope          : {cal_slope_2020:.3f} (Deviation from 1 = Over/Underfitting)")

prob_true, prob_pred = calibration_curve(y_true_2020, y_pred_probs_2020, n_bins=10, strategy='quantile')
bins = np.percentile(y_pred_probs_2020, np.linspace(0, 100, 11))
bin_indices = np.digitize(y_pred_probs_2020, bins[1:-1])
bin_counts = np.bincount(bin_indices, minlength=10)

print("\n[*] Reliability Curve Binning Sanity Check:")
for idx, count in enumerate(bin_counts):
    print(f"    -> Bin {idx+1}: n = {int(count)} samples")
if np.min(bin_counts) < 50:
    print("    [WARNING] Some bins have < 50 samples. Consider reducing n_bins for smoother calibration estimates.")

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='ST-GNN (2020 Adversarial Test)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.title(f"Reliability Curve: Structural Integrity vs Baseline Shift\nCity: {TARGET_CITY.upper()} (2020)")
plt.xlabel("Mean Predicted Probability"); plt.ylabel("Fraction of Actual STPCs")
plt.legend(); plt.grid(True); plt.show()

# Miami
(The Coastal Strip): A narrow, physically constrained geographic strip where traffic has very few alternative routing options.

# Experiment A ( fixed-topology transferability from LA to Miami)

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Miami"  # Options: "Los Angeles", "Chicago", "Miami", "Indianapolis", "Iowa"

# Hyperparameter Search Space (Sensitivity Analysis)
SEARCH_NODES = [100]       # MAUP scale sensitivity (node count)
SEARCH_RADII = [2.0]  # Physical interaction horizons (miles)
SEARCH_K = [3]            # Intersection routing topologies
SEQ_LENGTH = 4                  # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena...")

experiment_results = []
RESULTS_PATH = f"{BASE_DIR}/ST_GNN_Results_{TARGET_CITY}.csv"
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")
            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )
                total_pos = np.sum(y_tr); total_neg = (y_tr.size) - total_pos
                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1; continue
                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH, n_nodes=n_nodes, n_features=X_tr.shape[-1],
                    adj_matrix=adj, pos_weight=pos_weight_val
                )
                print("[*] Commencing Training...")
                history = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_val),
                    epochs=15, batch_size=64, callbacks=[early_stop], verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten(); y_pred_flat = y_pred_probs_val.flatten()
                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]; final_recall = recalls[best_idx]
                    final_f1 = 2*(final_precision*final_recall)/(final_precision+final_recall) if (final_precision+final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(auc_pr, 4),
                    'Optimal_Threshold': round(optimal_threshold, 4),
                    'Precision_at_70_Recall': round(final_precision, 4),
                    'Actual_Recall': round(final_recall, 4),
                    'Val_F1_Score': round(final_f1, 4)
                })
                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            tf.keras.backend.clear_session(); gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")


## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# 1) FROZEN LA CHAMPION TOPOLOGY (Experiment A: Transferability Test)
CHAMP_NODES = 100
CHAMP_RADIUS = 2.0
CHAMP_K = 3
print(f"[*] Frozen LA Topology Applied to {TARGET_CITY.upper()}: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
# 2) ISOLATE THE FINAL TENSORS (Champion)
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)
total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)
y_val_flat = y_val.flatten(); y_te_flat = y_te.flatten()

# 3) TRADITIONAL ML BASELINES (LR & RF) with SafeNodeClassifier
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1); X_te_ml = X_te.reshape(X_te.shape[0], -1)
lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)
    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        # SafeNodeClassifier always returns 2-col proba; guard is retained
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0
    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5
    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)
    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")
del X_tr_ml, X_te_ml, lr_model, rf_model; gc.collect()

# 4) DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
early_stop_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_final], verbose=0)
    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(auc_lstm); lstm_f1_scores.append(f1_lstm)
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_final], verbose=0)
    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(auc_stgnn); stgnn_f1_scores.append(f1_stgnn)
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

lstm_auc_mean,  lstm_auc_std  = np.mean(lstm_auc_scores),  np.std(lstm_auc_scores)
stgnn_auc_mean, stgnn_auc_std = np.mean(stgnn_auc_scores), np.std(stgnn_auc_scores)
lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")
    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This can occur when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/Thesis_Final_Proof_Matrix.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'LR_AUC': round(lr_auc, 4),
    'RF_AUC': round(rf_auc, 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(lstm_ci[0], 4),
    'LSTM_CI_Upper': round(lstm_ci[1], 4),
    'STGNN_CI_Lower': round(stgnn_ci[0], 4),
    'STGNN_CI_Upper': round(stgnn_ci[1], 4),
    'Wilcoxon_p_value': round(p_value, 4) if p_value is not None else None
}])
if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (AUC-PR)
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')
max_y = max(auc_scores); plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')
plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()


#Experiment B (The Topological Optimization)

Experiment B: Native Topology Discovery in Miami

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Miami"

# Experiment B: Unified Urban Search Space (Autonomous Discovery)
# We expand the radii and K-neighbors to allow the model to discover
SEARCH_NODES = [100, 250, 500]
SEARCH_RADII = [0.5, 1.0, 2.0]
SEARCH_K = [3, 5, 8]
SEQ_LENGTH = 4                      # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


##PHASE 4B: GRAPHSAGE COMPARATOR (STATIC KNN GRAPH)

In [ ]:
# ==============================================================================
# PHASE 4B: GRAPHSAGE COMPARATOR (STATIC KNN GRAPH)
# ==============================================================================
print("[*] Compiling GraphSAGE Comparator Layer...")

class GraphSAGELayer(Layer):
    """
    GraphSAGE-style mean aggregation layer for static macro-region graphs.

    For each node:
        1. Aggregate neighbor features by row-normalized weighted mean
        2. Transform self features
        3. Transform neighbor aggregate
        4. Concatenate both representations
        5. Project back to output_dim and apply ReLU

    Inputs:
        - inputs:     (batch, time, nodes, features)
        - adj_matrix: (nodes, nodes) static normalized adjacency matrix
    """
    def __init__(self, output_dim, **kwargs):
        super(GraphSAGELayer, self).__init__(**kwargs)
        self.output_dim = output_dim

    def build(self, input_shape):
        feature_dim = input_shape[-1]

        # Separate transforms for self and neighbor channels
        self.self_kernel = self.add_weight(
            name='sage_self_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        self.neigh_kernel = self.add_weight(
            name='sage_neigh_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        # Combine [self_repr || neigh_repr] -> output_dim
        self.combine_kernel = self.add_weight(
            name='sage_combine_kernel',
            shape=(2 * self.output_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        self.bias = self.add_weight(
            name='sage_bias',
            shape=(self.output_dim,),
            initializer='zeros',
            trainable=True
        )

        super(GraphSAGELayer, self).build(input_shape)

    def call(self, inputs, adj_matrix):
        """
        inputs:     (B, T, V, F)
        adj_matrix: (V, V)
        """
        adj_matrix = tf.cast(adj_matrix, dtype=tf.float32)

        # Remove self-loops for neighborhood-only aggregation
        adj_no_self = tf.linalg.set_diag(
            adj_matrix,
            tf.zeros(tf.shape(adj_matrix)[0], dtype=tf.float32)
        )

        # Row-normalize to create a weighted mean aggregator
        row_sum = tf.reduce_sum(adj_no_self, axis=1, keepdims=True)
        neigh_adj = tf.math.divide_no_nan(adj_no_self, row_sum)

        # Neighborhood mean aggregation: (V,V) x (B,T,V,F) -> (B,T,V,F)
        # CRITICAL FIX: Pure Python arrow, no HTML escape
        neigh_mean = tf.einsum('vw,btwf->btvf', neigh_adj, inputs)

        # Linear transforms
        self_repr = tf.matmul(inputs, self.self_kernel)        # (B,T,V,D)
        neigh_repr = tf.matmul(neigh_mean, self.neigh_kernel)  # (B,T,V,D)

        # Concatenate and combine
        combined = tf.concat([self_repr, neigh_repr], axis=-1) # (B,T,V,2D)
        output = tf.matmul(combined, self.combine_kernel) + self.bias

        return tf.nn.relu(output)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)


def build_graphsage_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    """
    GraphSAGE comparator with the SAME temporal head pattern as your ST-GNN:
        GraphSAGE -> BN -> Dropout
        GraphSAGE -> BN -> Dropout
        Residual Add
        Reshape
        GRU
        Dense
        Sigmoid Output

    This keeps the comparison fair:
        - same graph
        - same temporal sequence modeling
        - same weighted BCE loss
        - only the graph propagation rule changes
    """
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="GraphSAGE_Input")

        # GraphSAGE Block 1
        sage_1 = GraphSAGELayer(output_dim=32)(input_tensor, adj_matrix)
        sage_1 = BatchNormalization()(sage_1)
        sage_1 = Dropout(0.3)(sage_1)

        # GraphSAGE Block 2
        sage_2 = GraphSAGELayer(output_dim=32)(sage_1, adj_matrix)
        sage_2 = BatchNormalization()(sage_2)
        sage_2 = Dropout(0.3)(sage_2)

        # Residual connection (same spirit as ST-GNN)
        sage_combined = layers.Add()([sage_1, sage_2])

        # Keep the temporal head identical to ST-GNN for fair comparison
        reshape_out = Reshape((time_steps, n_nodes * 32))(sage_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out)
        gru_out = Dense(32, activation='relu')(gru_out)

        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)

        model = Model(inputs=input_tensor, outputs=output_tensor)

        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

        model.compile(
            optimizer=optimizer,
            loss=custom_loss,
            metrics=['accuracy']
        )

    return model

print("[SUCCESS] Phase 4B Complete: GraphSAGE Comparator Loaded.")

## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: EXPERIMENT B - NATIVE TOPOLOGY DISCOVERY (The Challenger Arena)
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena (Native Search)...")

experiment_results = []

# SCIENTIFIC FIX: Rename output path to explicitly protect Experiment A data
RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Arena_Results.csv"

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")

            try:
                # 1. Generate Environment
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )

                # 2. Imbalance Math
                total_pos = np.sum(y_tr)
                total_neg = (y_tr.size) - total_pos

                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1
                    continue

                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                # 3. Build & Compile Model
                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH,
                    n_nodes=n_nodes,
                    n_features=X_tr.shape[-1],
                    adj_matrix=adj,
                    pos_weight=pos_weight_val
                )

                # 4. Train Model
                print("[*] Commencing Training...")

                # SCIENTIFIC FIX: Fresh initialization per experiment to prevent memory/state leaks
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                history = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=15,
                    batch_size=64,
                    callbacks=[early_stop],
                    verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                # 5. Scientific Evaluation (AUC-PR on Validation Vault)
                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)

                y_val_flat = y_val.flatten()
                y_pred_flat = y_pred_probs_val.flatten()

                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]
                    final_recall = recalls[best_idx]
                    final_f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                # 6. Log Results
                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(float(auc_pr), 4),
                    'Optimal_Threshold': round(float(optimal_threshold), 4),
                    'Precision_at_70_Recall': round(float(final_precision), 4),
                    'Actual_Recall': round(float(final_recall), 4),
                    'Val_F1_Score': round(float(final_f1), 4)
                })

                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            # 7. Aggressive Cleanup to protect Colab RAM

            if 'early_stop' in locals(): del early_stop
            if 'model' in locals(): del model
            if 'history' in locals(): del history
            if 'y_pred_probs_val' in locals(): del y_pred_probs_val
            if 'X_tr' in locals(): del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")

## PHASE 5B: MAUP Sensitivity Analysis
**Methodology:** The Modifiable Areal Unit Problem (MAUP) states that results derived from spatially aggregated data are sensitive to the choice of areal unit — in this case, the number of K-Means macro-nodes. A result that only holds for one node count is not robust.

This section loads the Phase 5 arena results and produces a sensitivity plot comparing AUC-PR across both node configurations (100 and 500) for each radius. A stable model should show consistent relative rankings regardless of spatial granularity.

In [ ]:
# ==============================================================================
# PHASE 5B: MAUP SENSITIVITY ANALYSIS (Scale)
# ==============================================================================
print("[*] Running MAUP Sensitivity Analysis...")

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")
arena_results_df = pd.read_csv(RESULTS_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'MAUP Sensitivity Analysis — Effect of Spatial Granularity on AUC-PR\nCity: {TARGET_CITY.upper()}',
             fontsize=13, fontweight='bold')
node_colors = NODE_COLORS

for ax_idx, radius in enumerate(SEARCH_RADII):
    ax = axes[ax_idx]
    subset = arena_results_df[arena_results_df['Radius_mi'] == radius].copy()
    for n_nodes_val, grp in subset.groupby('Nodes'):
        grp_sorted = grp.sort_values('K_Neighbors')
        ax.plot(
            grp_sorted['K_Neighbors'], grp_sorted['Val_AUC_PR'],
            marker='o', linewidth=2, color=node_colors.get(n_nodes_val, 'grey'),
            label=f'{n_nodes_val} Nodes'
        )
    ax.set_title(f'Radius = {radius}mi', fontweight='bold')
    ax.set_xlabel('K Neighbors', fontweight='bold'); ax.set_ylabel('Validation AUC-PR', fontweight='bold')
    ax.set_xticks(SEARCH_K); ax.legend(title='Node Count'); ax.grid(True, alpha=0.3)

plt.tight_layout()
MAUP_PATH = f"{BASE_DIR}/{TARGET_CITY}_MAUP_Sensitivity.png"
plt.savefig(MAUP_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] MAUP Sensitivity artifact saved to: {MAUP_PATH}")
plt.show()

print("\n[*] AUC-PR by Node Count (Mean across all K and Radius configurations):")
maup_summary = arena_results_df.groupby('Nodes')['Val_AUC_PR'].agg(['mean','std','min','max'])
print(maup_summary.round(4).to_string())


##Phase 5C: The MAUP Zonation Protocol

In [ ]:
# ==============================================================================
# PHASE 5C: MAUP ZONATION SENSITIVITY (Champion-Based)
# ==============================================================================
print(f"[*] Initiating Phase 5C: MAUP Zonation Stability Protocol for {TARGET_CITY.upper()}...")

# ----------------------------------------------------------------------
# 5C.0 Load Phase 5 arena results explicitly (independent of Phase 5B)
# ----------------------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_results_df = pd.read_csv(RESULTS_PATH)
if arena_results_df.empty:
    raise ValueError("[ERROR] Arena results file is empty. No valid configurations available for Phase 5C.")

# ----------------------------------------------------------------------
# 5C.1 Anchor Phase 5C to the ACTUAL champion discovered in Phase 5
# ----------------------------------------------------------------------
champion_row = arena_results_df.sort_values('Val_AUC_PR', ascending=False).iloc[0]

FIXED_NODES = int(champion_row['Nodes'])
FIXED_RADIUS = float(champion_row['Radius_mi'])
FIXED_K = int(champion_row['K_Neighbors'])

print(f"[*] Zonation test anchored at CHAMPION topology: "
      f"N={FIXED_NODES}, Radius={FIXED_RADIUS}mi, K={FIXED_K}")

# ----------------------------------------------------------------------
# 5C.2 Chronological sort to preserve time integrity
# ----------------------------------------------------------------------
df_target['Start_Time'] = pd.to_datetime(df_target['Start_Time'])
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)

# ----------------------------------------------------------------------
# 5C.3 Dynamic UTM projection for metric-consistent zonation
# ----------------------------------------------------------------------
transformer, epsg_code = _dynamic_utm_transformer(df_target)
utm_coords = _project_to_utm(df_target, transformer)

# ----------------------------------------------------------------------
# 5C.4 Window-aligned train mask (perfectly aligned with _windows_and_split)
# ----------------------------------------------------------------------
start_min = df_target['Start_Time'].min().floor('h')
end_max   = df_target['Start_Time'].max().floor('h')
full_time = pd.date_range(start=start_min, end=end_max, freq='h')

total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)

if len(full_time) == 0 or total_windows == 0:
    raise ValueError("[ERROR] No temporal windows available for zonation protocol.")

cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]
train_mask = df_target['Start_Time'] < cutoff_time
utm_train = utm_coords[train_mask.values]

if utm_train.shape[0] < FIXED_NODES:
    raise ValueError(
        f"[ERROR] Insufficient train-era accident points ({utm_train.shape[0]}) "
        f"to fit KMeans with n_clusters={FIXED_NODES}."
    )

# ----------------------------------------------------------------------
# 5C.5 K-Means zonation stability loop
# ----------------------------------------------------------------------
ZONATION_SEEDS = 20
zonation_auc_scores = []
test_prevalence = None

print(f"[*] Executing {ZONATION_SEEDS} K-Means Spatial Partitions "
      f"(Train-Era Fitted, UTM={epsg_code})...")

for s in range(ZONATION_SEEDS):
    print(f"    -> Zonation Seed {s+1}/{ZONATION_SEEDS}")

    try:
        # Best-effort reproducibility
        random.seed(42 + s)
        np.random.seed(42 + s)
        tf.random.set_seed(42 + s)

        kmeans = KMeans(n_clusters=FIXED_NODES, random_state=s, n_init=10)
        kmeans.fit(utm_train)  # fit ONLY on train-era coordinates
        df_target['Region_ID'] = kmeans.predict(utm_coords)

        X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
            df_target,
            n_nodes=FIXED_NODES,
            radius_mi=FIXED_RADIUS,
            k_neighbors=FIXED_K,
            seq_len=SEQ_LENGTH
        )

        node_degrees = adj.sum(axis=1)
        if not (node_degrees > 0).all():
            isolated_count = int((node_degrees == 0).sum())
            raise ValueError(f"[CRITICAL FAILURE] Adjacency contains {isolated_count} isolated nodes at Seed {s}.")

        if test_prevalence is None:
            test_prevalence = float(y_te.mean())

        pos = y_tr.sum()
        neg = (y_tr.size - pos)
        pos_weight_val = float(neg / max(pos, 1))

        # SCIENTIFIC FIX: Callback initialized INSIDE the loop so it resets its memory state per run
        early_stop_zonation = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=64,
            callbacks=[early_stop_zonation],
            verbose=0
        )

        y_pred = model.predict(X_te, verbose=0).flatten()
        auc_pr = average_precision_score(y_te.flatten(), y_pred)
        zonation_auc_scores.append(float(auc_pr))

        # Cleanup
        del model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, y_pred, early_stop_zonation
        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:
        print(f"    [WARNING] Seed {s} failed: {e}")
        tf.keras.backend.clear_session()
        gc.collect()
        continue

if len(zonation_auc_scores) == 0:
    raise RuntimeError("[CRITICAL] All K-Means zonation runs failed. Cannot compute zonation robustness.")

# ----------------------------------------------------------------------
# 5C.6 Dynamic H3 method binding (Colab/runtime-safe)
# ----------------------------------------------------------------------
h3_encode = getattr(h3, "latlng_to_cell", getattr(h3, "geo_to_h3", None))
h3_decode = getattr(h3, "cell_to_latlng", getattr(h3, "h3_to_geo", None))

if h3_encode is None or h3_decode is None:
    raise ImportError("[CRITICAL FAILURE] Unrecognized H3 library version. Encode/decode methods missing.")

print("\n[*] Executing Alternate Zoning Family: H3 Hexagonal Super-Regions (Train-Era Calibrated)...")

def choose_h3_resolution(df_like, target=100, rmin=5, rmax=12):
    """
    Choose the SMALLEST H3 resolution whose occupied train-era cell count
    is >= target. This guarantees we can aggregate DOWN to exactly target.
    """
    scan_log = []

    # First search in the requested range
    for res in range(rmin, rmax + 1):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        scan_log.append((res, n))
        if n >= target:
            return res, ids, scan_log

    # If still too coarse, expand adaptively up to max H3 resolution 15
    for res in range(rmax + 1, 16):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        scan_log.append((res, n))
        if n >= target:
            return res, ids, scan_log

    raise ValueError(
        f"[CRITICAL FAILURE] Even H3 resolution 15 produced fewer than {target} occupied cells. "
        f"Scan log: {scan_log}"
    )

h3_res, _, h3_scan_log = choose_h3_resolution(df_target[train_mask], target=FIXED_NODES)
print(f"    -> H3 resolution scan (train-era): {h3_scan_log}")
print(f"    -> Selected H3 resolution: {h3_res}")

h3_ids_full = df_target.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], h3_res), axis=1)
uniq_hex = pd.Series(h3_ids_full.astype(str).values).unique()
n_hex_nodes = len(uniq_hex)

print(f"    -> H3 produced {n_hex_nodes} occupied hexes on full data at resolution {h3_res}")

if n_hex_nodes < FIXED_NODES:
    raise ValueError(
        f"[CRITICAL FAILURE] H3 produced only {n_hex_nodes} unique hexes, "
        f"which is fewer than target={FIXED_NODES}. "
        f"This should not happen if resolution was chosen on the train-era subset correctly."
    )

if n_hex_nodes == FIXED_NODES:
    print(f"    -> H3 already matches target exactly ({FIXED_NODES}). No aggregation required.")
    df_target['Region_ID'] = h3_ids_full.astype('category').cat.codes
else:
    print(f"    -> Aggregating {n_hex_nodes} H3 hexes down to exactly {FIXED_NODES} contiguous super-regions...")
    hex_latlon = np.array([h3_decode(h) for h in uniq_hex])  # (lat, lng)
    hex_x, hex_y = transformer.transform(hex_latlon[:, 1], hex_latlon[:, 0])

    # SCIENTIFIC FIX: Guard against safe_k = 0
    safe_k = max(1, min(6, n_hex_nodes - 1))

    spatial_connectivity = kneighbors_graph(
        np.c_[hex_x, hex_y], n_neighbors=safe_k, include_self=False
    )

    agg_cluster = AgglomerativeClustering(
        n_clusters=FIXED_NODES,
        connectivity=spatial_connectivity,
        linkage='ward'
    )
    h_labels = agg_cluster.fit_predict(np.c_[hex_x, hex_y])

    hex_to_super = {h: int(l) for h, l in zip(uniq_hex, h_labels)}
    df_target['Region_ID'] = pd.Series(h3_ids_full.astype(str).values).map(hex_to_super).values

# ----------------------------------------------------------------------
# 5C.7 Evaluate H3 zoning family under same champion topology
# ----------------------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
    df_target,
    n_nodes=FIXED_NODES,
    radius_mi=FIXED_RADIUS,
    k_neighbors=FIXED_K,
    seq_len=SEQ_LENGTH
)

node_degrees = adj.sum(axis=1)
if not (node_degrees > 0).all():
    isolated_count = int((node_degrees == 0).sum())
    raise ValueError(f"[CRITICAL FAILURE] H3 adjacency contains {isolated_count} isolated nodes.")

pos = y_tr.sum()
neg = (y_tr.size - pos)
pos_weight_val = float(neg / max(pos, 1))

early_stop_h3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

h3_model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
h3_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop_h3],
    verbose=0
)

h3_pred = h3_model.predict(X_te, verbose=0).flatten()
h3_auc_pr = average_precision_score(y_te.flatten(), h3_pred)

# ----------------------------------------------------------------------
# 5C.8 Summarize robustness statistics
# ----------------------------------------------------------------------
from scipy.stats import percentileofscore

mean_auc = float(np.mean(zonation_auc_scores))
sd_auc = float(np.std(zonation_auc_scores))
cv_percent = (sd_auc / mean_auc) * 100 if mean_auc > 0 else 0.0
q_lo, q_hi = np.quantile(zonation_auc_scores, [0.025, 0.975])
h3_pct = percentileofscore(zonation_auc_scores, h3_auc_pr)

print("\n========================================================")
print("[*] ZONATION MAUP RESULTS")
print("========================================================")
print(f"Champion Topology Tested     : {FIXED_NODES} Nodes | {FIXED_RADIUS}mi | K={FIXED_K}")
print(f"Test-Set Prevalence Baseline : {float(test_prevalence):.4f}")
print(f"K-Means PR-AUC (n={len(zonation_auc_scores)}) : {mean_auc:.4f} ± {sd_auc:.4f}")
print(f"95% Empirical Quantile CI    : [{q_lo:.4f}, {q_hi:.4f}]")
print(f"Coefficient of Variation (CV): {cv_percent:.2f}%")
print(f"H3 Super-Region PR-AUC       : {h3_auc_pr:.4f}")
print(f"H3 Percentile within K-Means : {h3_pct:.1f}th percentile")

# ----------------------------------------------------------------------
# 5C.9 Save zonation results
# ----------------------------------------------------------------------
ZONATION_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Zonation_Results.csv"

# SCIENTIFIC FIX: Log the actual number of successful runs, not the attempted target.
zonation_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': FIXED_NODES,
    'Champion_Radius_mi': FIXED_RADIUS,
    'Champion_K': FIXED_K,
    'KMeans_Zonation_Runs': len(zonation_auc_scores),
    'KMeans_AUC_Mean': round(mean_auc, 4),
    'KMeans_AUC_Std': round(sd_auc, 4),
    'KMeans_AUC_Q2.5': round(float(q_lo), 4),
    'KMeans_AUC_Q97.5': round(float(q_hi), 4),
    'KMeans_CV_Percent': round(float(cv_percent), 2),
    'H3_AUC_PR': round(float(h3_auc_pr), 4),
    'H3_Percentile_Within_KMeans': round(float(h3_pct), 2)
}])

zonation_df.to_csv(ZONATION_RESULTS_PATH, index=False)
print(f"[SUCCESS] Zonation robustness results saved to: {ZONATION_RESULTS_PATH}")

# Cleanup
del h3_model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, h3_pred, early_stop_h3
tf.keras.backend.clear_session()
gc.collect()

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Miami Native Champion + GraphSAGE Comparator)
# ==============================================================================
print(f"\n[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

import scipy.stats as stats

# --------------------------------------------------------
# 1. DYNAMICALLY EXTRACT CHAMPION (Sorted by Val_AUC_PR)
# --------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}")

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]
CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# --------------------------------------------------------
# 2. ISOLATE THE FINAL TENSORS
# --------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target,
    n_nodes=CHAMP_NODES,
    radius_mi=CHAMP_RADIUS,
    k_neighbors=CHAMP_K,
    seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError(
        "[ERROR] Champion topology produced zero STPC labels in training set. "
        "Re-run Phase 5 — the arena may have selected a degenerate configuration."
    )

pos_weight_val = min(float(((y_tr.size) - total_pos) / max(total_pos, 1)), 1000.0)
y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# --------------------------------------------------------
# 3. TRADITIONAL ML BASELINES (Logistic Regression & Random Forest)
# --------------------------------------------------------
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1)
X_te_ml = X_te.reshape(X_te.shape[0], -1)

lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)

    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)

    for i in range(y_te.shape[1]):
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0

    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)

    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2:
            y_val_probs[:, i] = val_preds[i][:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]

    # SCIENTIFIC FIX: Guard against IndexOutOfBounds on thresholds
    if len(valid_idx) > 0:
        best_idx = valid_idx[np.argmax(precisions[valid_idx])]
        opt_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    else:
        opt_thresh = 0.5

    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)

    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")

del X_tr_ml, X_te_ml, lr_model, rf_model
gc.collect()

# --------------------------------------------------------
# 4. DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
# --------------------------------------------------------
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
graphsage_auc_scores, graphsage_f1_scores = [], []

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]

    # SCIENTIFIC FIX: Guard against IndexOutOfBounds on thresholds
    if len(valid_idx) == 0:
        return 0.5

    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    # -------------------------------
    # BASELINE LSTM
    # -------------------------------
    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    lstm_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_lstm],
        verbose=0
    )

    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(float(auc_lstm))
    lstm_f1_scores.append(float(f1_lstm))
    print(f"[*] LSTM_Blind        -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")

    del lstm_model, early_stop_lstm
    tf.keras.backend.clear_session()
    gc.collect()

    # -------------------------------
    # ST-GNN (your custom model)
    # -------------------------------
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    stgnn_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_stgnn],
        verbose=0
    )

    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(float(auc_stgnn))
    stgnn_f1_scores.append(float(f1_stgnn))
    print(f"[*] STGNN_StaticKNN   -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")

    del stgnn_model, early_stop_stgnn
    tf.keras.backend.clear_session()
    gc.collect()

    # -------------------------------
    # GraphSAGE comparator
    # -------------------------------
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    graphsage_model = build_graphsage_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    early_stop_graphsage = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    graphsage_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_graphsage],
        verbose=0
    )

    thresh_graphsage = get_frozen_threshold(graphsage_model, X_val, y_val_flat)
    auc_graphsage, f1_graphsage = evaluate_nn(graphsage_model, X_te, y_te_flat, thresh_graphsage)
    graphsage_auc_scores.append(float(auc_graphsage))
    graphsage_f1_scores.append(float(f1_graphsage))
    print(f"[*] GraphSAGE_StaticKNN -> AUC-PR: {auc_graphsage:.4f} | F1: {f1_graphsage:.4f}")

    del graphsage_model, early_stop_graphsage
    tf.keras.backend.clear_session()
    gc.collect()

# --------------------------------------------------------
# 5. SUMMARY + HYPOTHESIS TESTING
# --------------------------------------------------------
def safe_t_interval(scores):
    scores = np.array(scores, dtype=float)
    if len(scores) < 2 or np.std(scores) == 0:
        m = float(np.mean(scores))
        return (m, m)
    return stats.t.interval(0.95, df=len(scores)-1, loc=np.mean(scores), scale=stats.sem(scores))

lstm_auc_mean, lstm_auc_std = float(np.mean(lstm_auc_scores)), float(np.std(lstm_auc_scores))
stgnn_auc_mean, stgnn_auc_std = float(np.mean(stgnn_auc_scores)), float(np.std(stgnn_auc_scores))
graphsage_auc_mean, graphsage_auc_std = float(np.mean(graphsage_auc_scores)), float(np.std(graphsage_auc_scores))

lstm_ci = safe_t_interval(lstm_auc_scores)
stgnn_ci = safe_t_interval(stgnn_auc_scores)
graphsage_ci = safe_t_interval(graphsage_auc_scores)

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()} (Graph Comparator Added)")
print("==================================================================")
print(f"Champion Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"LSTM_Blind AUC-PR          : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"STGNN_StaticKNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print(f"GraphSAGE_StaticKNN AUC-PR : {graphsage_auc_mean:.4f} ± {graphsage_auc_std:.4f}  95% CI [{graphsage_ci[0]:.4f}, {graphsage_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon, STGNN vs LSTM): p-value = {p_value:.4f}")

    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis against the temporal-only baseline.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed against the temporal-only baseline at α=0.05.")

except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This typically occurs when all paired score differences are zero.")

print(f"[*] Comparator Note: GraphSAGE mean AUC-PR = {graphsage_auc_mean:.4f} vs ST-GNN mean AUC-PR = {stgnn_auc_mean:.4f}")

# --------------------------------------------------------
# 6. SCIENTIFIC LOGGING
# --------------------------------------------------------
PROOF_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Final_Proof_with_GraphSAGE.csv"

proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'Champion_K': CHAMP_K,
    'LR_AUC': round(float(lr_auc), 4),
    'RF_AUC': round(float(rf_auc), 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'GraphSAGE_AUC_Mean': round(graphsage_auc_mean, 4),
    'GraphSAGE_AUC_Std': round(graphsage_auc_std, 4),
    'LSTM_CI_Lower': round(float(lstm_ci[0]), 4),
    'LSTM_CI_Upper': round(float(lstm_ci[1]), 4),
    'STGNN_CI_Lower': round(float(stgnn_ci[0]), 4),
    'STGNN_CI_Upper': round(float(stgnn_ci[1]), 4),
    'GraphSAGE_CI_Lower': round(float(graphsage_ci[0]), 4),
    'GraphSAGE_CI_Upper': round(float(graphsage_ci[1]), 4),
    'Wilcoxon_p_value_STGNN_vs_LSTM': round(float(p_value), 4) if p_value is not None else None
}])

proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

# --------------------------------------------------------
# 7. UPDATED VISUALIZATION ARTIFACT
# --------------------------------------------------------
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()} (with GraphSAGE)...")

model_names = [
    'Logistic\nRegression',
    'Random\nForest',
    'Spatially Blind\nLSTM',
    'ST-GNN\n(Static KNN)',
    'GraphSAGE\n(Static KNN)'
]

auc_scores = [
    lr_auc,
    rf_auc,
    lstm_auc_mean,
    stgnn_auc_mean,
    graphsage_auc_mean
]

std_devs = [
    0.0,
    0.0,
    lstm_auc_std,
    stgnn_auc_std,
    graphsage_auc_std
]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6', '#1a9641']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(
    f'Spatio-Temporal vs. Traditional Baselines (AUC-PR)\nCity: {TARGET_CITY.upper()} | Native Champion: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
    fontsize=13, fontweight='bold', pad=15
)
plt.ylabel('AUC-PR (Higher is Better)', fontsize=11, fontweight='bold')

max_y = max(auc_scores)
plt.ylim(0, max(max_y * 1.35, 0.05))

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(
        f'AUC: {height:.4f}{std_text}',
        (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
        ha='center', va='bottom',
        fontsize=10, color='black',
        xytext=(0, 8),
        textcoords='offset points',
        fontweight='bold'
    )

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Phase6_AUC_Comparison_with_GraphSAGE.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()
plt.close()

## PHASE 6B: Feature Importance & SHAP-Equivalent Analysis
**Methodology:** To determine which input features actually drive STPC predictions, the Champion Random Forest model's built-in feature importances are extracted. Since the RF operates on flattened `(Time_Steps × Nodes × Features)` vectors, importances are aggregated back across the 4 original feature dimensions to produce a per-feature contribution score.

This analysis answers a critical scientific question: does the model rely on **accident density** (Total_Accidents), **incident severity** (Mean_Severity), **road clearance time** (Mean_Distance), or **weather friction** (Mean_Humidity) to identify STPCs? The result directly informs the theoretical interpretation of the shockwave propagation mechanism.

In [ ]:
# ==============================================================================
# PHASE 6B: FEATURE IMPORTANCE ANALYSIS (Random Forest)
# ==============================================================================
print(f"[*] Extracting Feature Importances from Champion Random Forest...")

_, rf_importance_model = build_traditional_ml_baselines()
X_tr_ml_fi = X_tr.reshape(X_tr.shape[0], -1)
rf_importance_model.fit(X_tr_ml_fi, y_tr)

all_importances = np.array([est.feature_importances_ for est in rf_importance_model.estimators_])
mean_importances = all_importances.mean(axis=0)

features_list = FEATURES_LIST
n_features = len(features_list)
feature_contributions = np.zeros(n_features)
for feat_idx in range(n_features):
    feature_contributions[feat_idx] = mean_importances[feat_idx::n_features].sum()
feature_contributions /= feature_contributions.sum()

fig, ax = plt.subplots(figsize=(8, 5))
colors_fi = ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641']
bars_fi = ax.barh(features_list, feature_contributions, color=colors_fi, edgecolor='black', alpha=0.88)
ax.set_xlabel('Aggregated Feature Importance (Normalised)', fontsize=11, fontweight='bold')
ax.set_title(f'RF Feature Importance — STPC Prediction Drivers\nCity: {TARGET_CITY.upper()} | Champion Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars_fi, feature_contributions):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
FI_PATH = f"{BASE_DIR}/{TARGET_CITY}_Feature_Importance.png"
plt.savefig(FI_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] Feature Importance artifact saved to: {FI_PATH}")
plt.show()

del rf_importance_model, X_tr_ml_fi, all_importances; gc.collect()

##Phase 6C: Core Ablation Suite
(Miami Native Champion)

In [ ]:
# ==============================================================================
# PHASE 6C: CORE ABLATION SUITE (MIAMI NATIVE CHAMPION)
# ==============================================================================
print(f"\n[*] Initializing Phase 6C: Core Ablation Suite for MIAMI (Experiment B)...")

import random

# Locked Miami native champion from Phase 6
ABLATION_TARGET_CITY = "Miami"
ABLATION_EXPERIMENT_ID = "ExperimentB_UrbanNative"
ABLATION_PHASE_ID = "Phase6C_Ablation"

ABLATION_CHAMP_NODES = 100
ABLATION_CHAMP_RADIUS = 2.0
ABLATION_CHAMP_K = 5

# Full feature set (do not modify the original global FEATURES_LIST)
ABLATION_FULL_FEATURE_LIST = [
    'Total_Accidents',
    'Mean_Severity',
    'Mean_Distance',
    'Mean_Humidity'
]

# Number of repeated neural runs per ablation
ABLATION_RUNS = 5

# Additive-only output path (does not overwrite any existing Phase 6 files)
ABLATION_RESULTS_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_Results.csv"

ABLATION_CONFIGS = {
    "Full": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Humidity": {
        "feature_list": ['Total_Accidents', 'Mean_Severity', 'Mean_Distance'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Severity": {
        "feature_list": ['Total_Accidents', 'Mean_Distance', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Distance": {
        "feature_list": ['Total_Accidents', 'Mean_Severity', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_TotalAccidents": {
        "feature_list": ['Mean_Severity', 'Mean_Distance', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "SeqLen1": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 1,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "BinaryAdjacency": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 4,
        "adjacency_mode": "binary",
        "run_lstm": False,
        "run_stgnn": True
    }
}

print(f"[*] Miami ablation suite locked to champion topology: "
      f"{ABLATION_CHAMP_NODES} Nodes | {ABLATION_CHAMP_RADIUS}mi | K={ABLATION_CHAMP_K}")
print(f"[*] Ablation results will be saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C HELPER FUNCTIONS (Additive Only)
# ==============================================================================

def _build_adjacency_from_centers_utm_ablation(centers_xy, k_neighbors, adjacency_mode="inverse_distance"):
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)

    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)

    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)

            if adjacency_mode == "inverse_distance":
                w = 1.0 / d
            elif adjacency_mode == "binary":
                w = 1.0
            else:
                raise ValueError(f"[ERROR] Unknown adjacency_mode: {adjacency_mode}")

            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w

    # Symmetric normalization
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)

    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = int((deg == 0).sum())
        raise ValueError(f"[CRITICAL] Ablation adjacency contains {iso} isolated nodes.")

    return normalized_adj


def aggregate_to_hourly_ablation(df, target_col, feature_list):
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')

    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')

    multi_idx = pd.MultiIndex.from_product(
        [sorted(df['Region_ID'].unique()), full_time],
        names=['Region_ID', 'Time_Step']
    )

    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(feature_list)))

    for idx, feat in enumerate(feature_list):
        city_features[:, :, idx] = continuous_df.pivot(
            index='Time_Step',
            columns='Region_ID',
            values=feat
        ).fillna(0).values

    city_targets = continuous_df.pivot(
        index='Time_Step',
        columns='Region_ID',
        values='STPC_Count'
    ).fillna(0).values

    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time


def build_st_graph_tensors_ablation(
    df_raw,
    n_nodes,
    radius_mi,
    k_neighbors,
    seq_len=4,
    feature_list=None,
    adjacency_mode="inverse_distance"
):
    if feature_list is None:
        feature_list = ABLATION_FULL_FEATURE_LIST

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    normalized_adj = _build_adjacency_from_centers_utm_ablation(
        centers_utm,
        k_neighbors,
        adjacency_mode=adjacency_mode
    )

    city_features, city_targets, full_time = aggregate_to_hourly_ablation(
        df,
        target_col,
        feature_list
    )

    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(
        city_features,
        city_targets,
        seq_len=seq_len
    )

    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


def get_frozen_threshold_ablation(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5


def evaluate_nn_ablation(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1


# ==============================================================================
# PHASE 6C RUNNER
# ==============================================================================

def run_core_ablation_suite(
    df_raw,
    target_city,
    experiment_id,
    champ_nodes,
    champ_radius,
    champ_k,
    ablation_configs,
    n_runs=5,
    results_path=None
):
    ablation_results = []

    for ablation_name, cfg in ablation_configs.items():
        print("\n==================================================================")
        print(f"[*] ABLATION: {ablation_name}")
        print("==================================================================")

        feature_list = cfg["feature_list"]
        seq_len = cfg["seq_len"]
        adjacency_mode = cfg["adjacency_mode"]
        run_lstm = cfg["run_lstm"]
        run_stgnn = cfg["run_stgnn"]

        X_tr = X_val = X_te = y_tr = y_val = y_te = adj = None

        try:
            X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_ablation(
                df_raw=df_raw,
                n_nodes=champ_nodes,
                radius_mi=champ_radius,
                k_neighbors=champ_k,
                seq_len=seq_len,
                feature_list=feature_list,
                adjacency_mode=adjacency_mode
            )

            total_pos = np.sum(y_tr)
            if total_pos == 0:
                print(f"[WARNING] {ablation_name} yielded zero positives. Skipping.")
                continue

            pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)
            y_val_flat = y_val.flatten()
            y_te_flat = y_te.flatten()

            # ------------------------------------------------------------------
            # LSTM Blind
            # ------------------------------------------------------------------
            if run_lstm:
                lstm_auc_scores, lstm_f1_scores = [], []
                print(f"[*] Training LSTM_Blind ({n_runs} runs)...")

                for run in range(n_runs):
                    random.seed(42 + run)
                    np.random.seed(42 + run)
                    tf.random.set_seed(42 + run)

                    lstm_model = build_baseline_lstm(seq_len, champ_nodes, X_tr.shape[-1], pos_weight_val)
                    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                    lstm_model.fit(
                        X_tr, y_tr,
                        validation_data=(X_val, y_val),
                        epochs=15,
                        batch_size=64,
                        callbacks=[early_stop_lstm],
                        verbose=0
                    )

                    thresh_lstm = get_frozen_threshold_ablation(lstm_model, X_val, y_val_flat)
                    auc_lstm, f1_lstm = evaluate_nn_ablation(lstm_model, X_te, y_te_flat, thresh_lstm)

                    lstm_auc_scores.append(float(auc_lstm))
                    lstm_f1_scores.append(float(f1_lstm))

                    print(f"    [RUN {run+1}/{n_runs}] LSTM AUC-PR={auc_lstm:.4f} | F1={f1_lstm:.4f}")

                    del lstm_model, early_stop_lstm
                    tf.keras.backend.clear_session()
                    gc.collect()

                ablation_results.append({
                    "domain": target_city,
                    "experiment_id": experiment_id,
                    "phase_id": ABLATION_PHASE_ID,
                    "ablation_name": ablation_name,
                    "model_name": "LSTM_Blind",
                    "champion_nodes": champ_nodes,
                    "champion_radius_mi": champ_radius,
                    "champion_k_neighbors": champ_k,
                    "feature_list": ",".join(feature_list),
                    "seq_len": seq_len,
                    "adjacency_mode": adjacency_mode,
                    "n_runs": n_runs,
                    "test_auc_pr_mean": float(np.mean(lstm_auc_scores)),
                    "test_auc_pr_std": float(np.std(lstm_auc_scores)),
                    "test_f1_mean": float(np.mean(lstm_f1_scores)),
                    "test_f1_std": float(np.std(lstm_f1_scores)),
                })

                print(f"    -> [LSTM Summary] Mean AUC-PR: {np.mean(lstm_auc_scores):.4f} ± {np.std(lstm_auc_scores):.4f} | "
                      f"Mean F1: {np.mean(lstm_f1_scores):.4f} ± {np.std(lstm_f1_scores):.4f}")

            # ------------------------------------------------------------------
            # ST-GNN
            # ------------------------------------------------------------------
            if run_stgnn:
                stgnn_auc_scores, stgnn_f1_scores = [], []
                print(f"[*] Training STGNN_StaticKNN ({n_runs} runs)...")

                for run in range(n_runs):
                    random.seed(42 + run)
                    np.random.seed(42 + run)
                    tf.random.set_seed(42 + run)

                    stgnn_model = build_st_gnn_model(seq_len, champ_nodes, X_tr.shape[-1], adj, pos_weight_val)
                    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                    stgnn_model.fit(
                        X_tr, y_tr,
                        validation_data=(X_val, y_val),
                        epochs=15,
                        batch_size=64,
                        callbacks=[early_stop_stgnn],
                        verbose=0
                    )

                    thresh_stgnn = get_frozen_threshold_ablation(stgnn_model, X_val, y_val_flat)
                    auc_stgnn, f1_stgnn = evaluate_nn_ablation(stgnn_model, X_te, y_te_flat, thresh_stgnn)

                    stgnn_auc_scores.append(float(auc_stgnn))
                    stgnn_f1_scores.append(float(f1_stgnn))

                    print(f"    [RUN {run+1}/{n_runs}] STGNN AUC-PR={auc_stgnn:.4f} | F1={f1_stgnn:.4f}")

                    del stgnn_model, early_stop_stgnn
                    tf.keras.backend.clear_session()
                    gc.collect()

                ablation_results.append({
                    "domain": target_city,
                    "experiment_id": experiment_id,
                    "phase_id": ABLATION_PHASE_ID,
                    "ablation_name": ablation_name,
                    "model_name": "STGNN_StaticKNN",
                    "champion_nodes": champ_nodes,
                    "champion_radius_mi": champ_radius,
                    "champion_k_neighbors": champ_k,
                    "feature_list": ",".join(feature_list),
                    "seq_len": seq_len,
                    "adjacency_mode": adjacency_mode,
                    "n_runs": n_runs,
                    "test_auc_pr_mean": float(np.mean(stgnn_auc_scores)),
                    "test_auc_pr_std": float(np.std(stgnn_auc_scores)),
                    "test_f1_mean": float(np.mean(stgnn_f1_scores)),
                    "test_f1_std": float(np.std(stgnn_f1_scores)),
                })

                print(f"    -> [STGNN Summary] Mean AUC-PR: {np.mean(stgnn_auc_scores):.4f} ± {np.std(stgnn_auc_scores):.4f} | "
                      f"Mean F1: {np.mean(stgnn_f1_scores):.4f} ± {np.std(stgnn_f1_scores):.4f}")

            # Interim save for crash protection
            if results_path is not None:
                pd.DataFrame(ablation_results).to_csv(results_path, index=False)

        except Exception as e:
            print(f"[ERROR] Ablation '{ablation_name}' failed: {e}")

        finally:
            if 'X_tr' in locals() and X_tr is not None:
                del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()

    return pd.DataFrame(ablation_results)

# ==============================================================================
# RUN PHASE 6C FOR MIAMI
# ==============================================================================
miami_ablation_df = run_core_ablation_suite(
    df_raw=df_target,   # IMPORTANT: this must be the Miami-bounded df_target from Experiment B
    target_city=ABLATION_TARGET_CITY,
    experiment_id=ABLATION_EXPERIMENT_ID,
    champ_nodes=ABLATION_CHAMP_NODES,
    champ_radius=ABLATION_CHAMP_RADIUS,
    champ_k=ABLATION_CHAMP_K,
    ablation_configs=ABLATION_CONFIGS,
    n_runs=ABLATION_RUNS,
    results_path=ABLATION_RESULTS_PATH
)

print("\n[SUCCESS] Phase 6C complete.")
print(f"[*] Partial Ablation results saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C POST-PROCESSING: DELTA VS FULL
# ==============================================================================
if os.path.exists(ABLATION_RESULTS_PATH):
    ablation_df = pd.read_csv(ABLATION_RESULTS_PATH)
else:
    ablation_df = miami_ablation_df.copy()

# Sanity guard for missing "Full" reference row
full_ref = ablation_df[ablation_df["ablation_name"] == "Full"][["model_name", "test_auc_pr_mean", "test_f1_mean"]].copy()
if full_ref.empty:
    raise ValueError("[ERROR] Full ablation reference missing. Cannot compute deltas. Ensure 'Full' ablation succeeded.")

full_ref = full_ref.rename(columns={
    "test_auc_pr_mean": "full_auc_pr_mean",
    "test_f1_mean": "full_f1_mean"
})

ablation_df = ablation_df.merge(full_ref, on="model_name", how="left")
ablation_df["delta_auc_pr_vs_full"] = ablation_df["test_auc_pr_mean"] - ablation_df["full_auc_pr_mean"]
ablation_df["delta_f1_vs_full"] = ablation_df["test_f1_mean"] - ablation_df["full_f1_mean"]

ablation_df.to_csv(ABLATION_RESULTS_PATH, index=False)

print("\n[*] Ablation results with deltas vs full model:")
print(ablation_df.round(4).to_string(index=False))
print(f"\n[SUCCESS] Final ablation table with deltas saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C ARTIFACT: AUC-PR BY ABLATION
# ==============================================================================
AUC_PLOT_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_AUC.png"

plot_df = ablation_df.copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x="ablation_name",
    y="test_auc_pr_mean",
    hue="model_name"
)
plt.title(f"Core Ablation Suite — Test AUC-PR\nCity: {ABLATION_TARGET_CITY.upper()} | Champion: {ABLATION_CHAMP_NODES} Nodes, {ABLATION_CHAMP_RADIUS}mi, K={ABLATION_CHAMP_K}",
          fontweight='bold')
plt.xlabel("Ablation")
plt.ylabel("Mean Test AUC-PR")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(AUC_PLOT_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] Ablation AUC plot saved to: {AUC_PLOT_PATH}")
plt.show()

# ==============================================================================
# PHASE 6C ARTIFACT: DELTA AUC-PR VS FULL
# ==============================================================================
DELTA_PLOT_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_Delta.png"

delta_plot_df = ablation_df[ablation_df["ablation_name"] != "Full"].copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=delta_plot_df,
    x="ablation_name",
    y="delta_auc_pr_vs_full",
    hue="model_name"
)
plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
plt.title(f"Core Ablation Suite — ΔAUC-PR vs Full Model\nCity: {ABLATION_TARGET_CITY.upper()}",
          fontweight='bold')
plt.xlabel("Ablation")
plt.ylabel("Delta AUC-PR vs Full")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(DELTA_PLOT_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] Ablation delta plot saved to: {DELTA_PLOT_PATH}")
plt.show()

##PHASE 6D: ST-GNN PERMUTATION IMPORTANCE

In [ ]:
# ==============================================================================
# PHASE 6D: ST-GNN PERMUTATION IMPORTANCE (MIAMI NATIVE CHAMPION)
# ==============================================================================
print(f"\n[*] Initializing Phase 6D: ST-GNN Permutation Importance for {TARGET_CITY.upper()}...")

import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------------------------
# 1. RELOAD / LOCK THE MIAMI NATIVE CHAMPION FROM PHASE 5 ARENA RESULTS
# ------------------------------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"[ERROR] Arena results not found at {RESULTS_PATH}. "
        "Run Miami Experiment B Phase 5 first."
    )

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]
CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked for Permutation Importance: "
      f"{CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# ------------------------------------------------------------------------------
# 2. REBUILD THE CHAMPION TENSORS
# ------------------------------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target,
    n_nodes=CHAMP_NODES,
    radius_mi=CHAMP_RADIUS,
    k_neighbors=CHAMP_K,
    seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")

# Keep the sparse-domain safety cap
pos_weight_val = min(float(((y_tr.size) - total_pos) / max(total_pos, 1)), 1000.0)

y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# Feature names: use notebook FEATURES_LIST if present, otherwise fall back
if 'FEATURES_LIST' in globals():
    perm_feature_names = list(FEATURES_LIST)
else:
    perm_feature_names = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

if X_te.shape[-1] != len(perm_feature_names):
    raise ValueError(
        f"[ERROR] Feature count mismatch. X_te has {X_te.shape[-1]} channels, "
        f"but perm_feature_names has {len(perm_feature_names)} entries."
    )

PERM_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Phase6D_Permutation_Importance.csv"

# ------------------------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------------------------
def get_frozen_threshold(model, X_val, y_val_flat):
    """
    Derive the operational threshold from the validation set using the same
    recall >= 0.70 rule used elsewhere in the notebook, with a safe index guard.
    """
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)

    valid_idx = np.where(recalls >= 0.70)[0]
    if len(valid_idx) == 0:
        return 0.5

    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5


def evaluate_nn(model, X_eval, y_eval_flat, opt_thresh):
    """
    Return both AUC-PR and thresholded F1 on the supplied tensor.
    """
    y_pred_probs = model.predict(X_eval, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_eval_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_eval_flat, y_pred_binary, zero_division=0)
    return float(auc_pr), float(f1)


def permute_feature_channel(X_in, feature_idx, seed):
    """
    Permute one feature channel across the entire test tensor while preserving
    the marginal distribution of that channel.

    X_in shape: (samples, time_steps, nodes, features)
    """
    X_perm = X_in.copy()
    rng = np.random.default_rng(seed)

    # SCIENTIFIC FIX: Use .flatten() to ensure a clean, contiguous memory copy
    flat_channel = X_perm[..., feature_idx].flatten()
    rng.shuffle(flat_channel)
    X_perm[..., feature_idx] = flat_channel.reshape(X_perm[..., feature_idx].shape)

    return X_perm


# Initialize variables for the finally block
perm_model = None
early_stop_perm = None

try:
    # ------------------------------------------------------------------------------
    # 4. TRAIN ONE CHAMPION ST-GNN (FIXED SEED) FOR INTERPRETABILITY
    # ------------------------------------------------------------------------------
    print("\n[*] Training one champion ST-GNN instance for permutation importance...")
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)

    perm_model = build_st_gnn_model(
        SEQ_LENGTH,
        CHAMP_NODES,
        X_tr.shape[-1],
        adj,
        pos_weight_val
    )

    early_stop_perm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    perm_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_perm],
        verbose=0
    )

    frozen_thresh = get_frozen_threshold(perm_model, X_val, y_val_flat)
    baseline_auc, baseline_f1 = evaluate_nn(perm_model, X_te, y_te_flat, frozen_thresh)

    print(f"[BASELINE] Champion ST-GNN Test AUC-PR: {baseline_auc:.4f} | Test F1: {baseline_f1:.4f}")
    print(f"[*] Frozen threshold from validation: {frozen_thresh:.4f}")

    # ------------------------------------------------------------------------------
    # 5. PERMUTATION IMPORTANCE LOOP
    # ------------------------------------------------------------------------------
    N_PERMUTATIONS = 10
    print(f"\n[*] Running permutation importance with {N_PERMUTATIONS} shuffles per feature...")

    perm_results = []

    for feat_idx, feat_name in enumerate(perm_feature_names):
        auc_scores = []
        f1_scores = []

        print(f"\n--- [FEATURE] {feat_name} ---")

        for p in range(N_PERMUTATIONS):
            perm_seed = 1000 + (feat_idx * 100) + p
            X_te_perm = permute_feature_channel(X_te, feat_idx, seed=perm_seed)

            auc_perm, f1_perm = evaluate_nn(perm_model, X_te_perm, y_te_flat, frozen_thresh)
            auc_scores.append(auc_perm)
            f1_scores.append(f1_perm)

            print(f"    [PERM {p+1}/{N_PERMUTATIONS}] AUC-PR={auc_perm:.4f} | F1={f1_perm:.4f}")

            del X_te_perm
            gc.collect()

        auc_mean = float(np.mean(auc_scores))
        auc_std = float(np.std(auc_scores))
        f1_mean = float(np.mean(f1_scores))
        f1_std = float(np.std(f1_scores))

        delta_auc = auc_mean - baseline_auc
        delta_f1 = f1_mean - baseline_f1

        perm_results.append({
            "City": TARGET_CITY,
            "Experiment": "ExperimentB_UrbanNative",
            "Phase": "Phase6D_PermutationImportance",
            "Champion_Nodes": CHAMP_NODES,
            "Champion_Radius_mi": CHAMP_RADIUS,
            "Champion_K": CHAMP_K,
            "Feature_Name": feat_name,
            "Baseline_AUC_PR": round(baseline_auc, 4),
            "Permuted_AUC_PR_Mean": round(auc_mean, 4),
            "Permuted_AUC_PR_Std": round(auc_std, 4),
            "Delta_AUC_PR_vs_Baseline": round(delta_auc, 4),
            "Baseline_F1": round(baseline_f1, 4),
            "Permuted_F1_Mean": round(f1_mean, 4),
            "Permuted_F1_Std": round(f1_std, 4),
            "Delta_F1_vs_Baseline": round(delta_f1, 4),
            "N_Permutations": N_PERMUTATIONS
        })

        print(f"    -> [SUMMARY] {feat_name}: "
              f"AUC-PR {auc_mean:.4f} ± {auc_std:.4f} | ΔAUC={delta_auc:.4f}")

        # SCIENTIFIC FIX: Incremental saving to protect against Colab disconnects
        pd.DataFrame(perm_results).to_csv(PERM_RESULTS_PATH, index=False)

    # ------------------------------------------------------------------------------
    # 6. FINAL RESULTS DISPLAY
    # ------------------------------------------------------------------------------
    perm_df = pd.DataFrame(perm_results)
    print(f"\n[SUCCESS] Permutation-importance table saved to: {PERM_RESULTS_PATH}")
    print("\n[*] Permutation Importance Results:")
    print(perm_df.to_string(index=False))

    # ------------------------------------------------------------------------------
    # 7. PLOT ΔAUC-PR vs BASELINE
    # ------------------------------------------------------------------------------
    PERM_PLOT_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Phase6D_Permutation_Importance.png"

    plot_df = perm_df.copy()
    plot_df = plot_df.sort_values(by="Delta_AUC_PR_vs_Baseline", ascending=True)

    plt.figure(figsize=(10, 6))
    bars = plt.barh(
        plot_df["Feature_Name"],
        plot_df["Delta_AUC_PR_vs_Baseline"],
        color="#2c7bb6",
        edgecolor="black"
    )

    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.title(
        f"ST-GNN Permutation Importance (ΔAUC-PR vs Baseline)\n"
        f"City: {TARGET_CITY.upper()} | Champion: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}",
        fontweight="bold"
    )
    plt.xlabel("Delta AUC-PR vs Baseline (More Negative = More Important)")
    plt.ylabel("Feature")

    for bar, delta in zip(bars, plot_df["Delta_AUC_PR_vs_Baseline"]):
        plt.text(
            bar.get_width() + (0.001 if delta >= 0 else -0.001),
            bar.get_y() + bar.get_height() / 2,
            f"{delta:.4f}",
            va="center",
            ha="left" if delta >= 0 else "right",
            fontsize=10,
            fontweight="bold"
        )

    plt.tight_layout()
    plt.savefig(PERM_PLOT_PATH, dpi=300, bbox_inches="tight")
    print(f"[SUCCESS] Permutation-importance plot saved to: {PERM_PLOT_PATH}")
    plt.show()
    plt.close()

    # ------------------------------------------------------------------------------
    # 8. OPTIONAL ONE-LINE MORPHOLOGY-LINKED INTERPRETATION HINT
    # ------------------------------------------------------------------------------
    most_important_row = plot_df.iloc[0]   # most negative delta = biggest performance drop
    most_important_feature = most_important_row["Feature_Name"]
    most_important_delta = most_important_row["Delta_AUC_PR_vs_Baseline"]

    print("\n[*] Morphology-Linked Interpretation Hint:")
    print(
        f"Permuting '{most_important_feature}' produced the largest drop in Test AUC-PR "
        f"({most_important_delta:.4f}), suggesting that the Miami champion ST-GNN relies most strongly "
        f"on this signal when modeling cascade risk across the dense urban corridor topology."
    )

finally:
    # ------------------------------------------------------------------------------
    # 9. CLEANUP (Guaranteed Execution)
    # ------------------------------------------------------------------------------
    if perm_model is not None:
        del perm_model
    if early_stop_perm is not None:
        del early_stop_perm
    if 'X_tr' in locals() and X_tr is not None:
        del X_tr, X_val, X_te, y_tr, y_val, y_te, adj

    tf.keras.backend.clear_session()
    gc.collect()
    print("\n[SUCCESS] Phase 6D complete and memory safely cleared.")

## PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
**Methodology:** To visually validate the necessity of the Spatio-Temporal Graph Neural Network (ST-GNN), this phase maps the empirical ground truth of the "Butterfly Effect" within the target urban morphology.
By isolating the highest-density temporal window (the "Worst Day"), we plot the primary physical triggers (Primary Accidents) against the resulting friction cascades (Secondary Accidents). This visual artifact provides spatial evidence of the Champion Topology discovered in Phase 5, displaying the true physical spread of the traffic shockwave across the target urban morphology.

In [ ]:
#==============================================================================
# PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
# ==============================================================================
print(f"[*] Initializing Phase 7: Spatial Morphology Visualization for {TARGET_CITY.upper()}...")

df_map = df_target.copy()
df_map['Start_Time'] = pd.to_datetime(df_map['Start_Time'])
daily_secondary_counts = df_map[df_map[f'Is_STPC_{CHAMP_RADIUS}'] == 1].groupby(df_map['Start_Time'].dt.date).size()
worst_day = daily_secondary_counts.idxmax()
print(f"[*] Isolating Maximum Density Temporal Window: {worst_day}")

day_data = df_map[df_map['Start_Time'].dt.date == worst_day]
primary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 0]
secondary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 1]

num_primary = len(primary_accidents); num_secondary = len(secondary_accidents)
print(f"[*] Plotted Graph Nodes: {num_primary} Primary Triggers | {num_secondary} Secondary Cascades")

center_lat = day_data['Start_Lat'].mean(); center_lng = day_data['Start_Lng'].mean()
road_map = folium.Map(location=[center_lat, center_lng], zoom_start=11, control_scale=True)

for _, row in primary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=5, color='#3498db', fill=True, fill_color='#3498db', fill_opacity=0.6,
                        tooltip="Primary Accident (Isolated Trigger)").add_to(road_map)
for _, row in secondary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=8, color='black', weight=1.5, fill=True, fill_color='#e74c3c', fill_opacity=0.9,
                        tooltip=f"STPC (Proximate Collision Cascade) (Within {CHAMP_RADIUS}mi Shockwave)").add_to(road_map)

legend_html = f'''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 320px; height: auto;
    background-color: white; border: 2px solid grey; z-index: 9999;
    font-size: 14px; font-family: Arial, sans-serif;
    padding: 15px; box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
    ">
    <h4 style="margin-top: 0; margin-bottom: 5px; font-weight: bold; font-size: 15px;">
        Empirical Spatial Morphology
    </h4>
    <hr style="margin: 5px 0;">
    <b>City:</b> {TARGET_CITY.upper()}<br>
    <b>Temporal Window:</b> {worst_day}<br>
    <b>Champion Topology:</b> {CHAMP_RADIUS}mi Radius<br>
    <hr style="margin: 10px 0;">
    <div style="margin-bottom: 8px;">
        <span style="display: inline-block; width: 12px; height: 12px; background-color: #3498db; border-radius: 50%; margin-right: 8px; opacity: 0.6;"></span>
        <b>Primary Trigger</b> (n={num_primary})
    </div>
    <div>
        <span style="display: inline-block; width: 14px; height: 14px; background-color: #e74c3c; border: 2px solid black; border-radius: 50%; margin-right: 8px;"></span>
        <b>Secondary Cascade</b> (n={num_secondary})
    </div>
</div>
'''
road_map.get_root().html.add_child(Element(legend_html))
output_html = f'{BASE_DIR}/{TARGET_CITY}_Shockwave_Morphology_Map.html'
road_map.save(output_html)
print(f"\n==================================================================")
print(f"[SUCCESS] THESIS ARTIFACT GENERATED: {output_html}")
print(f"==================================================================")


from IPython.display import display
display(road_map)

##Phase 8: Temporal Drift & Calibration

In [ ]:
# ==============================================================================
# PHASE 8: TEMPORAL DRIFT & CALIBRATION (2020 Adversarial External Validation)
# ==============================================================================
print(f"[*] Initiating Phase 8: Temporal Drift Protocol for {TARGET_CITY.upper()}...")

import statsmodels.api as sm
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def build_st_graph_tensors_temporal_external(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4,
                                             cutoff=pd.Timestamp('2020-01-01')):
    """
    Fit KMeans on pre-cutoff (train/val) UTM coordinates, assign all data to regions,
    build adjacency on training cluster centers, then construct hourly tensors and
    return (X_tr, X_val) from pre-2020 and (X_te_2020) from 2020, with purge & scaling.
    """
    df = df_raw.copy().sort_values('Start_Time').reset_index(drop=True)
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]

    train_mask = df['Start_Time'] < cutoff
    test_mask  = df['Start_Time'].dt.year == 2020

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    kmeans.fit(df.loc[train_mask, ['_X','_Y']].values)
    df['Region_ID'] = kmeans.predict(df[['_X','_Y']].values)

    centers_utm = kmeans.cluster_centers_
    adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    X, y, t_end = [], [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len]); y.append(city_targets[i+seq_len]); t_end.append(full_time[i+seq_len])
    X, y, t_end = np.array(X), np.array(y), np.array(t_end)

    dev_idx = np.where(t_end < cutoff)[0]
    test_idx = np.where((t_end >= pd.Timestamp('2020-01-01')) & (t_end < pd.Timestamp('2021-01-01')))[0]
    if len(dev_idx) == 0 or len(test_idx) == 0:
        raise ValueError("[ERROR] Not enough samples in dev or 2020 windows.")

    dev_X, dev_y = X[dev_idx], y[dev_idx]
    total_dev = dev_X.shape[0]
    train_bound = int(total_dev * 0.82)  # internal dev split (train/val)
    purge = seq_len

    X_tr_raw, y_tr = dev_X[:train_bound], dev_y[:train_bound]
    X_val_raw, y_val = dev_X[train_bound + purge:], dev_y[train_bound + purge:]
    X_te_raw,  y_te  = X[test_idx], y[test_idx]

    num_features = X_tr_raw.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_tr_raw.reshape(-1, num_features)).reshape(X_tr_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te  = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, adj

X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext = build_st_graph_tensors_temporal_external(
    df_target, CHAMP_NODES, CHAMP_RADIUS, CHAMP_K, seq_len=SEQ_LENGTH, cutoff=pd.Timestamp('2020-01-01')
)
print(f"[*] Validation Era Prevalence (Pre-2020) : {float(y_val_ext.mean()):.4f}")
print(f"[*] Adversarial Era Prevalence (2020)    : {float(y_te_2020.mean()):.4f}")

pos_ext = y_tr_ext.sum(); neg_ext = (y_tr_ext.size - pos_ext)
pos_weight_ext = float(neg_ext / max(pos_ext, 1))

model_drift = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr_ext.shape[-1], adj_ext, pos_weight_ext)
model_drift.fit(X_tr_ext, y_tr_ext, validation_data=(X_val_ext, y_val_ext), epochs=15, batch_size=64, verbose=0)

y_pred_probs_2020 = model_drift.predict(X_te_2020, verbose=0).flatten()
y_true_2020 = y_te_2020.flatten()

def bootstrap_pr_auc(y_true, y_pred, n_bootstraps=500):
    scores = []
    N = len(y_pred)
    for _ in range(n_bootstraps):
        idx = np.random.randint(0, N, N)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(average_precision_score(y_true[idx], y_pred[idx]))
    if len(scores) == 0:
        return np.nan, np.nan
    return np.percentile(scores, [2.5, 97.5])

drift_auc_pr = average_precision_score(y_true_2020, y_pred_probs_2020)
boot_ci_lo, boot_ci_hi = bootstrap_pr_auc(y_true_2020, y_pred_probs_2020)
drift_brier = brier_score_loss(y_true_2020, y_pred_probs_2020)

def calibration_slope_intercept(y_true, p_hat):
    eps = 1e-6
    logit_ph = np.log(np.clip(p_hat, eps, 1-eps) / np.clip(1-p_hat, eps, 1-eps))
    X = sm.add_constant(logit_ph)  # intercept + slope
    model = sm.GLM(y_true, X, family=sm.families.Binomial())
    res = model.fit()
    return float(res.params[0]), float(res.params[1])

cal_intercept_2020, cal_slope_2020 = calibration_slope_intercept(y_true_2020, y_pred_probs_2020)

print("\n========================================================")
print("[*] TEMPORAL DRIFT & CALIBRATION ASSESSMENT (2020)")
print("========================================================")
print(f"Ranking Retention (PR-AUC) : {drift_auc_pr:.4f} (95% CI: [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}])")
print(f"Calibration Brier Score    : {drift_brier:.4f}")
print(f"Calibration Intercept      : {cal_intercept_2020:.3f} (Deviation from 0 = Risk Shift)")
print(f"Calibration Slope          : {cal_slope_2020:.3f} (Deviation from 1 = Over/Underfitting)")

prob_true, prob_pred = calibration_curve(y_true_2020, y_pred_probs_2020, n_bins=10, strategy='quantile')
bins = np.percentile(y_pred_probs_2020, np.linspace(0, 100, 11))
bin_indices = np.digitize(y_pred_probs_2020, bins[1:-1])
bin_counts = np.bincount(bin_indices, minlength=10)

print("\n[*] Reliability Curve Binning Sanity Check:")
for idx, count in enumerate(bin_counts):
    print(f"    -> Bin {idx+1}: n = {int(count)} samples")
if np.min(bin_counts) < 50:
    print("    [WARNING] Some bins have < 50 samples. Consider reducing n_bins for smoother calibration estimates.")

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='ST-GNN (2020 Adversarial Test)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.title(f"Reliability Curve: Structural Integrity vs Baseline Shift\nCity: {TARGET_CITY.upper()} (2020)")
plt.xlabel("Mean Predicted Probability"); plt.ylabel("Fraction of Actual STPCs")
plt.legend(); plt.grid(True); plt.show()

# Indianapolis
 (The Hub-and-Spoke): A centralized urban core with highways radiating outward like spokes on a wheel.

# Experiment A ( fixed-topology transferability from LA to Indianapolis)

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Indianapolis"  # Options: "Los Angeles", "Chicago", "Miami", "Indianapolis", "Iowa"

# Hyperparameter Search Space (Sensitivity Analysis)
SEARCH_NODES = [100]       # MAUP scale sensitivity (node count)
SEARCH_RADII = [2.0]  # Physical interaction horizons (miles)
SEARCH_K = [3]            # Intersection routing topologies
SEQ_LENGTH = 4                  # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena...")

experiment_results = []
RESULTS_PATH = f"{BASE_DIR}/ST_GNN_Results_{TARGET_CITY}.csv"
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")
            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )
                total_pos = np.sum(y_tr); total_neg = (y_tr.size) - total_pos
                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1; continue
                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH, n_nodes=n_nodes, n_features=X_tr.shape[-1],
                    adj_matrix=adj, pos_weight=pos_weight_val
                )
                print("[*] Commencing Training...")
                history = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_val),
                    epochs=15, batch_size=64, callbacks=[early_stop], verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten(); y_pred_flat = y_pred_probs_val.flatten()
                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]; final_recall = recalls[best_idx]
                    final_f1 = 2*(final_precision*final_recall)/(final_precision+final_recall) if (final_precision+final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(auc_pr, 4),
                    'Optimal_Threshold': round(optimal_threshold, 4),
                    'Precision_at_70_Recall': round(final_precision, 4),
                    'Actual_Recall': round(final_recall, 4),
                    'Val_F1_Score': round(final_f1, 4)
                })
                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            tf.keras.backend.clear_session(); gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")


## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# 1) FROZEN LA CHAMPION TOPOLOGY (Experiment A: Transferability Test)
CHAMP_NODES = 100
CHAMP_RADIUS = 2.0
CHAMP_K = 3
print(f"[*] Frozen LA Topology Applied to {TARGET_CITY.upper()}: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
# 2) ISOLATE THE FINAL TENSORS (Champion)
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)
total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)
y_val_flat = y_val.flatten(); y_te_flat = y_te.flatten()

# 3) TRADITIONAL ML BASELINES (LR & RF) with SafeNodeClassifier
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1); X_te_ml = X_te.reshape(X_te.shape[0], -1)
lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)
    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        # SafeNodeClassifier always returns 2-col proba; guard is retained
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0
    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5
    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)
    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")
del X_tr_ml, X_te_ml, lr_model, rf_model; gc.collect()

# 4) DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
early_stop_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_final], verbose=0)
    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(auc_lstm); lstm_f1_scores.append(f1_lstm)
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_final], verbose=0)
    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(auc_stgnn); stgnn_f1_scores.append(f1_stgnn)
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")
    tf.keras.backend.clear_session(); gc.collect()

lstm_auc_mean,  lstm_auc_std  = np.mean(lstm_auc_scores),  np.std(lstm_auc_scores)
stgnn_auc_mean, stgnn_auc_std = np.mean(stgnn_auc_scores), np.std(stgnn_auc_scores)
lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")
    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This can occur when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/Thesis_Final_Proof_Matrix.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'LR_AUC': round(lr_auc, 4),
    'RF_AUC': round(rf_auc, 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(lstm_ci[0], 4),
    'LSTM_CI_Upper': round(lstm_ci[1], 4),
    'STGNN_CI_Lower': round(stgnn_ci[0], 4),
    'STGNN_CI_Upper': round(stgnn_ci[1], 4),
    'Wilcoxon_p_value': round(p_value, 4) if p_value is not None else None
}])
if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (AUC-PR)
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')
max_y = max(auc_scores); plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')
plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()


#Experiment B (The Topological Optimization)

Experiment B: Native Topology Discovery in Indianapolis

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # (SafeNodeClassifier)

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing (scientific geometry & tessellation)
import math
import pyproj         # PROJECTION: dynamic UTM per city (metric geometry)
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target city for the current experimental run
TARGET_CITY = "Indianapolis"

# Experiment B: Unified Urban Search Space (Autonomous Discovery)
# We expand the radii and K-neighbors to allow the model to discover
SEARCH_NODES = [100, 250, 500]
SEARCH_RADII = [0.5, 1.0, 2.0]
SEARCH_K = [3, 5, 8]
SEQ_LENGTH = 4                      # Locked: 4-Hour rolling temporal window

# Feature columns used in tensor generation AND feature importance analysis.
# Defined once here — all downstream cells reference this global.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

# MAUP node color map — update if SEARCH_NODES changes
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for: {TARGET_CITY.upper()}")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")


## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

# 2.1 Automated Dataset Download (Reproducible Ingestion)
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

# Dynamically locate the CSV file inside the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, city_name):
    """
    Loads the massive national dataset and strictly filters it to the target city.
    Extracts only the critical features required for the ST-GNN tensors.
    """
    print(f"[*] Extracting raw records for {city_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (The Isolation Step)
    df_city = df_raw[df_raw['City'].str.lower() == city_name.lower()].copy()
    print(f"[*] Spatial Bounding Complete: {len(df_city)} raw accident records found in {city_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()


### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("\n[SUCCESS] EDA Complete.")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a city-specific UTM transformer based on median lon/lat (Northern/Southern hemisphere)."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)  # distances in meters (UTM)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d  # inverse meters
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w
    # Symmetric normalized adjacency (D^{-1/2} (A+I) D^{-1/2})
    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)
    # HARD GUARD: ensure no isolated nodes remain
    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    PRE-CLUSTERED GENERATOR (Zonation tests): Use preassigned Region_ID; adjacency from region centroids in UTM.
    """
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values

    print(f"[PRECLUSTER] Building adjacency on fixed regions (UTM={epsg_code})...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[PRECLUSTER] Aggregating to hourly & building windows...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

# ---- SafeNodeClassifier (NEW): prevents LR/RF failures on single-class nodes ----
class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures:
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        # Always expose a consistent 2-class interface downstream
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        proba = self.model_.predict_proba(X)
        return proba

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: EXPERIMENT B - NATIVE TOPOLOGY DISCOVERY (The Challenger Arena)
# ==============================================================================
print("[*] Initializing Phase 5: The Challenger Arena (Native Search)...")

experiment_results = []

# SCIENTIFIC FIX: Rename output path to explicitly protect Experiment A data
RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Arena_Results.csv"

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")

            try:
                # 1. Generate Environment
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )

                # 2. Imbalance Math
                total_pos = np.sum(y_tr)
                total_neg = (y_tr.size) - total_pos

                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1
                    continue

                pos_weight_val = float(total_neg / total_pos)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                # 3. Build & Compile Model
                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH,
                    n_nodes=n_nodes,
                    n_features=X_tr.shape[-1],
                    adj_matrix=adj,
                    pos_weight=pos_weight_val
                )

                # 4. Train Model
                print("[*] Commencing Training...")

                # SCIENTIFIC FIX: Fresh initialization per experiment to prevent memory/state leaks
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                history = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=15,
                    batch_size=64,
                    callbacks=[early_stop],
                    verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                # 5. Scientific Evaluation (AUC-PR on Validation Vault)
                print("[*] Evaluating Predictions on Validation Set to discover Champion...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)

                y_val_flat = y_val.flatten()
                y_pred_flat = y_pred_probs_val.flatten()

                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]
                    final_recall = recalls[best_idx]
                    final_f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                # 6. Log Results
                experiment_results.append({
                    'City': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(float(auc_pr), 4),
                    'Optimal_Threshold': round(float(optimal_threshold), 4),
                    'Precision_at_70_Recall': round(float(final_precision), 4),
                    'Actual_Recall': round(float(final_recall), 4),
                    'Val_F1_Score': round(float(final_f1), 4)
                })

                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            # 7. Aggressive Cleanup to protect Colab RAM

            if 'early_stop' in locals(): del early_stop
            if 'model' in locals(): del model
            if 'history' in locals(): del history
            if 'y_pred_probs_val' in locals(): del y_pred_probs_val
            if 'X_tr' in locals(): del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. All boundary conditions validated.")
print(f"[*] Full thesis results matrix securely saved to: {RESULTS_PATH}")

## PHASE 5B: MAUP Sensitivity Analysis
**Methodology:** The Modifiable Areal Unit Problem (MAUP) states that results derived from spatially aggregated data are sensitive to the choice of areal unit — in this case, the number of K-Means macro-nodes. A result that only holds for one node count is not robust.

This section loads the Phase 5 arena results and produces a sensitivity plot comparing AUC-PR across both node configurations (100 and 500) for each radius. A stable model should show consistent relative rankings regardless of spatial granularity.

In [ ]:
# ==============================================================================
# PHASE 5B: MAUP SENSITIVITY ANALYSIS (Scale)
# ==============================================================================
print("[*] Running MAUP Sensitivity Analysis...")

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")
arena_results_df = pd.read_csv(RESULTS_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'MAUP Sensitivity Analysis — Effect of Spatial Granularity on AUC-PR\nCity: {TARGET_CITY.upper()}',
             fontsize=13, fontweight='bold')
node_colors = NODE_COLORS

for ax_idx, radius in enumerate(SEARCH_RADII):
    ax = axes[ax_idx]
    subset = arena_results_df[arena_results_df['Radius_mi'] == radius].copy()
    for n_nodes_val, grp in subset.groupby('Nodes'):
        grp_sorted = grp.sort_values('K_Neighbors')
        ax.plot(
            grp_sorted['K_Neighbors'], grp_sorted['Val_AUC_PR'],
            marker='o', linewidth=2, color=node_colors.get(n_nodes_val, 'grey'),
            label=f'{n_nodes_val} Nodes'
        )
    ax.set_title(f'Radius = {radius}mi', fontweight='bold')
    ax.set_xlabel('K Neighbors', fontweight='bold'); ax.set_ylabel('Validation AUC-PR', fontweight='bold')
    ax.set_xticks(SEARCH_K); ax.legend(title='Node Count'); ax.grid(True, alpha=0.3)

plt.tight_layout()
MAUP_PATH = f"{BASE_DIR}/{TARGET_CITY}_MAUP_Sensitivity.png"
plt.savefig(MAUP_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] MAUP Sensitivity artifact saved to: {MAUP_PATH}")
plt.show()

print("\n[*] AUC-PR by Node Count (Mean across all K and Radius configurations):")
maup_summary = arena_results_df.groupby('Nodes')['Val_AUC_PR'].agg(['mean','std','min','max'])
print(maup_summary.round(4).to_string())


##Phase 5C: The MAUP Zonation Protocol

In [ ]:
# ==============================================================================
# PHASE 5C: MAUP ZONATION SENSITIVITY (Champion-Based)
# ==============================================================================
print(f"[*] Initiating Phase 5C: MAUP Zonation Stability Protocol for {TARGET_CITY.upper()}...")

# ----------------------------------------------------------------------
# 5C.0 Load Phase 5 arena results explicitly (independent of Phase 5B)
# ----------------------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_results_df = pd.read_csv(RESULTS_PATH)
if arena_results_df.empty:
    raise ValueError("[ERROR] Arena results file is empty. No valid configurations available for Phase 5C.")

# ----------------------------------------------------------------------
# 5C.1 Anchor Phase 5C to the ACTUAL champion discovered in Phase 5
# ----------------------------------------------------------------------
champion_row = arena_results_df.sort_values('Val_AUC_PR', ascending=False).iloc[0]

FIXED_NODES = int(champion_row['Nodes'])
FIXED_RADIUS = float(champion_row['Radius_mi'])
FIXED_K = int(champion_row['K_Neighbors'])

print(f"[*] Zonation test anchored at CHAMPION topology: "
      f"N={FIXED_NODES}, Radius={FIXED_RADIUS}mi, K={FIXED_K}")

# ----------------------------------------------------------------------
# 5C.2 Chronological sort to preserve time integrity
# ----------------------------------------------------------------------
df_target['Start_Time'] = pd.to_datetime(df_target['Start_Time'])
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)

# ----------------------------------------------------------------------
# 5C.3 Dynamic UTM projection for metric-consistent zonation
# ----------------------------------------------------------------------
transformer, epsg_code = _dynamic_utm_transformer(df_target)
utm_coords = _project_to_utm(df_target, transformer)

# ----------------------------------------------------------------------
# 5C.4 Window-aligned train mask (perfectly aligned with _windows_and_split)
# ----------------------------------------------------------------------
start_min = df_target['Start_Time'].min().floor('h')
end_max   = df_target['Start_Time'].max().floor('h')
full_time = pd.date_range(start=start_min, end=end_max, freq='h')

total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)

if len(full_time) == 0 or total_windows == 0:
    raise ValueError("[ERROR] No temporal windows available for zonation protocol.")

cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]
train_mask = df_target['Start_Time'] < cutoff_time
utm_train = utm_coords[train_mask.values]

if utm_train.shape[0] < FIXED_NODES:
    raise ValueError(
        f"[ERROR] Insufficient train-era accident points ({utm_train.shape[0]}) "
        f"to fit KMeans with n_clusters={FIXED_NODES}."
    )

# ----------------------------------------------------------------------
# 5C.5 K-Means zonation stability loop
# ----------------------------------------------------------------------
ZONATION_SEEDS = 20
zonation_auc_scores = []
test_prevalence = None

print(f"[*] Executing {ZONATION_SEEDS} K-Means Spatial Partitions "
      f"(Train-Era Fitted, UTM={epsg_code})...")

for s in range(ZONATION_SEEDS):
    print(f"    -> Zonation Seed {s+1}/{ZONATION_SEEDS}")

    try:
        # Best-effort reproducibility
        random.seed(42 + s)
        np.random.seed(42 + s)
        tf.random.set_seed(42 + s)

        kmeans = KMeans(n_clusters=FIXED_NODES, random_state=s, n_init=10)
        kmeans.fit(utm_train)  # fit ONLY on train-era coordinates
        df_target['Region_ID'] = kmeans.predict(utm_coords)

        X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
            df_target,
            n_nodes=FIXED_NODES,
            radius_mi=FIXED_RADIUS,
            k_neighbors=FIXED_K,
            seq_len=SEQ_LENGTH
        )

        node_degrees = adj.sum(axis=1)
        if not (node_degrees > 0).all():
            isolated_count = int((node_degrees == 0).sum())
            raise ValueError(f"[CRITICAL FAILURE] Adjacency contains {isolated_count} isolated nodes at Seed {s}.")

        if test_prevalence is None:
            test_prevalence = float(y_te.mean())

        pos = y_tr.sum()
        neg = (y_tr.size - pos)
        pos_weight_val = float(neg / max(pos, 1))

        # SCIENTIFIC FIX: Callback initialized INSIDE the loop so it resets its memory state per run
        early_stop_zonation = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=64,
            callbacks=[early_stop_zonation],
            verbose=0
        )

        y_pred = model.predict(X_te, verbose=0).flatten()
        auc_pr = average_precision_score(y_te.flatten(), y_pred)
        zonation_auc_scores.append(float(auc_pr))

        # Cleanup
        del model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, y_pred, early_stop_zonation
        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:
        print(f"    [WARNING] Seed {s} failed: {e}")
        tf.keras.backend.clear_session()
        gc.collect()
        continue

if len(zonation_auc_scores) == 0:
    raise RuntimeError("[CRITICAL] All K-Means zonation runs failed. Cannot compute zonation robustness.")

# ----------------------------------------------------------------------
# 5C.6 Dynamic H3 method binding (Colab/runtime-safe)
# ----------------------------------------------------------------------
h3_encode = getattr(h3, "latlng_to_cell", getattr(h3, "geo_to_h3", None))
h3_decode = getattr(h3, "cell_to_latlng", getattr(h3, "h3_to_geo", None))

if h3_encode is None or h3_decode is None:
    raise ImportError("[CRITICAL FAILURE] Unrecognized H3 library version. Encode/decode methods missing.")

print("\n[*] Executing Alternate Zoning Family: H3 Hexagonal Super-Regions (Train-Era Calibrated)...")

def choose_h3_resolution(df_like, target=100, rmin=5, rmax=12):
    """
    Choose the SMALLEST H3 resolution whose occupied train-era cell count
    is >= target. This guarantees we can aggregate DOWN to exactly target.
    """
    scan_log = []

    # First search in the requested range
    for res in range(rmin, rmax + 1):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        scan_log.append((res, n))
        if n >= target:
            return res, ids, scan_log

    # If still too coarse, expand adaptively up to max H3 resolution 15
    for res in range(rmax + 1, 16):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        scan_log.append((res, n))
        if n >= target:
            return res, ids, scan_log

    raise ValueError(
        f"[CRITICAL FAILURE] Even H3 resolution 15 produced fewer than {target} occupied cells. "
        f"Scan log: {scan_log}"
    )

h3_res, _, h3_scan_log = choose_h3_resolution(df_target[train_mask], target=FIXED_NODES)
print(f"    -> H3 resolution scan (train-era): {h3_scan_log}")
print(f"    -> Selected H3 resolution: {h3_res}")

h3_ids_full = df_target.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], h3_res), axis=1)
uniq_hex = pd.Series(h3_ids_full.astype(str).values).unique()
n_hex_nodes = len(uniq_hex)

print(f"    -> H3 produced {n_hex_nodes} occupied hexes on full data at resolution {h3_res}")

if n_hex_nodes < FIXED_NODES:
    raise ValueError(
        f"[CRITICAL FAILURE] H3 produced only {n_hex_nodes} unique hexes, "
        f"which is fewer than target={FIXED_NODES}. "
        f"This should not happen if resolution was chosen on the train-era subset correctly."
    )

if n_hex_nodes == FIXED_NODES:
    print(f"    -> H3 already matches target exactly ({FIXED_NODES}). No aggregation required.")
    df_target['Region_ID'] = h3_ids_full.astype('category').cat.codes
else:
    print(f"    -> Aggregating {n_hex_nodes} H3 hexes down to exactly {FIXED_NODES} contiguous super-regions...")
    hex_latlon = np.array([h3_decode(h) for h in uniq_hex])  # (lat, lng)
    hex_x, hex_y = transformer.transform(hex_latlon[:, 1], hex_latlon[:, 0])

    # SCIENTIFIC FIX: Guard against safe_k = 0
    safe_k = max(1, min(6, n_hex_nodes - 1))

    spatial_connectivity = kneighbors_graph(
        np.c_[hex_x, hex_y], n_neighbors=safe_k, include_self=False
    )

    agg_cluster = AgglomerativeClustering(
        n_clusters=FIXED_NODES,
        connectivity=spatial_connectivity,
        linkage='ward'
    )
    h_labels = agg_cluster.fit_predict(np.c_[hex_x, hex_y])

    hex_to_super = {h: int(l) for h, l in zip(uniq_hex, h_labels)}
    df_target['Region_ID'] = pd.Series(h3_ids_full.astype(str).values).map(hex_to_super).values

# ----------------------------------------------------------------------
# 5C.7 Evaluate H3 zoning family under same champion topology
# ----------------------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
    df_target,
    n_nodes=FIXED_NODES,
    radius_mi=FIXED_RADIUS,
    k_neighbors=FIXED_K,
    seq_len=SEQ_LENGTH
)

node_degrees = adj.sum(axis=1)
if not (node_degrees > 0).all():
    isolated_count = int((node_degrees == 0).sum())
    raise ValueError(f"[CRITICAL FAILURE] H3 adjacency contains {isolated_count} isolated nodes.")

pos = y_tr.sum()
neg = (y_tr.size - pos)
pos_weight_val = float(neg / max(pos, 1))

early_stop_h3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

h3_model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
h3_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop_h3],
    verbose=0
)

h3_pred = h3_model.predict(X_te, verbose=0).flatten()
h3_auc_pr = average_precision_score(y_te.flatten(), h3_pred)

# ----------------------------------------------------------------------
# 5C.8 Summarize robustness statistics
# ----------------------------------------------------------------------
from scipy.stats import percentileofscore

mean_auc = float(np.mean(zonation_auc_scores))
sd_auc = float(np.std(zonation_auc_scores))
cv_percent = (sd_auc / mean_auc) * 100 if mean_auc > 0 else 0.0
q_lo, q_hi = np.quantile(zonation_auc_scores, [0.025, 0.975])
h3_pct = percentileofscore(zonation_auc_scores, h3_auc_pr)

print("\n========================================================")
print("[*] ZONATION MAUP RESULTS")
print("========================================================")
print(f"Champion Topology Tested     : {FIXED_NODES} Nodes | {FIXED_RADIUS}mi | K={FIXED_K}")
print(f"Test-Set Prevalence Baseline : {float(test_prevalence):.4f}")
print(f"K-Means PR-AUC (n={len(zonation_auc_scores)}) : {mean_auc:.4f} ± {sd_auc:.4f}")
print(f"95% Empirical Quantile CI    : [{q_lo:.4f}, {q_hi:.4f}]")
print(f"Coefficient of Variation (CV): {cv_percent:.2f}%")
print(f"H3 Super-Region PR-AUC       : {h3_auc_pr:.4f}")
print(f"H3 Percentile within K-Means : {h3_pct:.1f}th percentile")

# ----------------------------------------------------------------------
# 5C.9 Save zonation results
# ----------------------------------------------------------------------
ZONATION_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Zonation_Results.csv"

# SCIENTIFIC FIX: Log the actual number of successful runs, not the attempted target.
zonation_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': FIXED_NODES,
    'Champion_Radius_mi': FIXED_RADIUS,
    'Champion_K': FIXED_K,
    'KMeans_Zonation_Runs': len(zonation_auc_scores),
    'KMeans_AUC_Mean': round(mean_auc, 4),
    'KMeans_AUC_Std': round(sd_auc, 4),
    'KMeans_AUC_Q2.5': round(float(q_lo), 4),
    'KMeans_AUC_Q97.5': round(float(q_hi), 4),
    'KMeans_CV_Percent': round(float(cv_percent), 2),
    'H3_AUC_PR': round(float(h3_auc_pr), 4),
    'H3_Percentile_Within_KMeans': round(float(h3_pct), 2)
}])

zonation_df.to_csv(ZONATION_RESULTS_PATH, index=False)
print(f"[SUCCESS] Zonation robustness results saved to: {ZONATION_RESULTS_PATH}")

# Cleanup
del h3_model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, h3_pred, early_stop_h3
tf.keras.backend.clear_session()
gc.collect()

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Statistical Variance & Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# ---------------------------------------------------------
# 1. DYNAMICALLY EXTRACT CHAMPION (Sorted by Val_AUC_PR)
# ---------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]

CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# ---------------------------------------------------------
# 2. ISOLATE THE FINAL TENSORS
# ---------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target, n_nodes=CHAMP_NODES, radius_mi=CHAMP_RADIUS, k_neighbors=CHAMP_K, seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set. "
                     "Re-run Phase 5 — the arena may have selected a degenerate configuration.")
pos_weight_val = float(((y_tr.size) - total_pos) / total_pos)

y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# ---------------------------------------------------------
# 3. TRADITIONAL ML BASELINES (Logistic Regression & Random Forest)
# ---------------------------------------------------------
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")

X_tr_ml = X_tr.reshape(X_tr.shape[0], -1)
X_te_ml = X_te.reshape(X_te.shape[0], -1)

lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)

    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)
    for i in range(y_te.shape[1]):
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0

    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)
    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2: y_val_probs[:, i] = val_preds[i][:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)

    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")

del X_tr_ml, X_te_ml, lr_model, rf_model
gc.collect()

# ---------------------------------------------------------
# 4. DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
# ---------------------------------------------------------
N_RUNS = 10

print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    # --- BASELINE LSTM ---
    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)

    # SCIENTIFIC FIX: Fresh callback instantiated inside the loop
    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    lstm_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                   callbacks=[early_stop_lstm], verbose=0)

    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)

    lstm_auc_scores.append(float(auc_lstm))
    lstm_f1_scores.append(float(f1_lstm))
    print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")

    del lstm_model, early_stop_lstm
    tf.keras.backend.clear_session()
    gc.collect()

    # --- CHAMPION ST-GNN ---
    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)

    # SCIENTIFIC FIX: Fresh callback instantiated inside the loop
    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    stgnn_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64,
                    callbacks=[early_stop_stgnn], verbose=0)

    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)

    stgnn_auc_scores.append(float(auc_stgnn))
    stgnn_f1_scores.append(float(f1_stgnn))
    print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")

    del stgnn_model, early_stop_stgnn
    tf.keras.backend.clear_session()
    gc.collect()

# ---------------------------------------------------------
# 5. SCIENTIFIC LOGGING & HYPOTHESIS TESTING
# ---------------------------------------------------------
lstm_auc_mean,  lstm_auc_std  = float(np.mean(lstm_auc_scores)),  float(np.std(lstm_auc_scores))
stgnn_auc_mean, stgnn_auc_std = float(np.mean(stgnn_auc_scores)), float(np.std(stgnn_auc_scores))

lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()}")
print("==================================================================")
print(f"Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Champion ST-GNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")

    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This typically occurs when all paired score differences are zero.")

PROOF_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Final_Proof.csv"
proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'Champion_K': CHAMP_K,
    'LR_AUC': round(float(lr_auc), 4),
    'RF_AUC': round(float(rf_auc), 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(float(lstm_ci[0]), 4),
    'LSTM_CI_Upper': round(float(lstm_ci[1]), 4),
    'STGNN_CI_Lower': round(float(stgnn_ci[0]), 4),
    'STGNN_CI_Upper': round(float(stgnn_ci[1]), 4),
    'Wilcoxon_p_value': round(float(p_value), 4) if p_value is not None else None
}])

if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)

print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Topological\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(f'Spatio-Temporal vs. Traditional Baselines (Area Under PR Curve)\nCity: {TARGET_CITY.upper()} | Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')

max_y = max(auc_scores)
plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(f'AUC: {height:.4f}{std_text}',
                (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
                ha='center', va='bottom',
                fontsize=11, color='black', xytext=(0, 8),
                textcoords='offset points', fontweight='bold')

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentB_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()

## PHASE 6B: Feature Importance & SHAP-Equivalent Analysis
**Methodology:** To determine which input features actually drive STPC predictions, the Champion Random Forest model's built-in feature importances are extracted. Since the RF operates on flattened `(Time_Steps × Nodes × Features)` vectors, importances are aggregated back across the 4 original feature dimensions to produce a per-feature contribution score.

This analysis answers a critical scientific question: does the model rely on **accident density** (Total_Accidents), **incident severity** (Mean_Severity), **road clearance time** (Mean_Distance), or **weather friction** (Mean_Humidity) to identify STPCs? The result directly informs the theoretical interpretation of the shockwave propagation mechanism.

In [ ]:
# ==============================================================================
# PHASE 6B: FEATURE IMPORTANCE ANALYSIS (Random Forest)
# ==============================================================================
print(f"[*] Extracting Feature Importances from Champion Random Forest...")

_, rf_importance_model = build_traditional_ml_baselines()
X_tr_ml_fi = X_tr.reshape(X_tr.shape[0], -1)
rf_importance_model.fit(X_tr_ml_fi, y_tr)

all_importances = np.array([est.feature_importances_ for est in rf_importance_model.estimators_])
mean_importances = all_importances.mean(axis=0)

features_list = FEATURES_LIST
n_features = len(features_list)
feature_contributions = np.zeros(n_features)
for feat_idx in range(n_features):
    feature_contributions[feat_idx] = mean_importances[feat_idx::n_features].sum()
feature_contributions /= feature_contributions.sum()

fig, ax = plt.subplots(figsize=(8, 5))
colors_fi = ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641']
bars_fi = ax.barh(features_list, feature_contributions, color=colors_fi, edgecolor='black', alpha=0.88)
ax.set_xlabel('Aggregated Feature Importance (Normalised)', fontsize=11, fontweight='bold')
ax.set_title(f'RF Feature Importance — STPC Prediction Drivers\nCity: {TARGET_CITY.upper()} | Champion Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars_fi, feature_contributions):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
FI_PATH = f"{BASE_DIR}/{TARGET_CITY}_Feature_Importance.png"
plt.savefig(FI_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] Feature Importance artifact saved to: {FI_PATH}")
plt.show()

del rf_importance_model, X_tr_ml_fi, all_importances; gc.collect()

## PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
**Methodology:** To visually validate the necessity of the Spatio-Temporal Graph Neural Network (ST-GNN), this phase maps the empirical ground truth of the "Butterfly Effect" within the target urban morphology.
By isolating the highest-density temporal window (the "Worst Day"), we plot the primary physical triggers (Primary Accidents) against the resulting friction cascades (Secondary Accidents). This visual artifact provides spatial evidence of the Champion Topology discovered in Phase 5, displaying the true physical spread of the traffic shockwave across the target urban morphology.

In [ ]:
#==============================================================================
# PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
# ==============================================================================
print(f"[*] Initializing Phase 7: Spatial Morphology Visualization for {TARGET_CITY.upper()}...")

df_map = df_target.copy()
df_map['Start_Time'] = pd.to_datetime(df_map['Start_Time'])
daily_secondary_counts = df_map[df_map[f'Is_STPC_{CHAMP_RADIUS}'] == 1].groupby(df_map['Start_Time'].dt.date).size()
worst_day = daily_secondary_counts.idxmax()
print(f"[*] Isolating Maximum Density Temporal Window: {worst_day}")

day_data = df_map[df_map['Start_Time'].dt.date == worst_day]
primary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 0]
secondary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 1]

num_primary = len(primary_accidents); num_secondary = len(secondary_accidents)
print(f"[*] Plotted Graph Nodes: {num_primary} Primary Triggers | {num_secondary} Secondary Cascades")

center_lat = day_data['Start_Lat'].mean(); center_lng = day_data['Start_Lng'].mean()
road_map = folium.Map(location=[center_lat, center_lng], zoom_start=11, control_scale=True)

for _, row in primary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=5, color='#3498db', fill=True, fill_color='#3498db', fill_opacity=0.6,
                        tooltip="Primary Accident (Isolated Trigger)").add_to(road_map)
for _, row in secondary_accidents.iterrows():
    folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                        radius=8, color='black', weight=1.5, fill=True, fill_color='#e74c3c', fill_opacity=0.9,
                        tooltip=f"STPC (Proximate Collision Cascade) (Within {CHAMP_RADIUS}mi Shockwave)").add_to(road_map)

legend_html = f'''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 320px; height: auto;
    background-color: white; border: 2px solid grey; z-index: 9999;
    font-size: 14px; font-family: Arial, sans-serif;
    padding: 15px; box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
    ">
    <h4 style="margin-top: 0; margin-bottom: 5px; font-weight: bold; font-size: 15px;">
        Empirical Spatial Morphology
    </h4>
    <hr style="margin: 5px 0;">
    <b>City:</b> {TARGET_CITY.upper()}<br>
    <b>Temporal Window:</b> {worst_day}<br>
    <b>Champion Topology:</b> {CHAMP_RADIUS}mi Radius<br>
    <hr style="margin: 10px 0;">
    <div style="margin-bottom: 8px;">
        <span style="display: inline-block; width: 12px; height: 12px; background-color: #3498db; border-radius: 50%; margin-right: 8px; opacity: 0.6;"></span>
        <b>Primary Trigger</b> (n={num_primary})
    </div>
    <div>
        <span style="display: inline-block; width: 14px; height: 14px; background-color: #e74c3c; border: 2px solid black; border-radius: 50%; margin-right: 8px;"></span>
        <b>Secondary Cascade</b> (n={num_secondary})
    </div>
</div>
'''
road_map.get_root().html.add_child(Element(legend_html))
output_html = f'{BASE_DIR}/{TARGET_CITY}_Shockwave_Morphology_Map.html'
road_map.save(output_html)
print(f"\n==================================================================")
print(f"[SUCCESS] THESIS ARTIFACT GENERATED: {output_html}")
print(f"==================================================================")


from IPython.display import display
display(road_map)

##Phase 8: Temporal Drift & Calibration

In [ ]:
# ==============================================================================
# PHASE 8: TEMPORAL DRIFT & CALIBRATION (2020 Adversarial External Validation)
# ==============================================================================
print(f"[*] Initiating Phase 8: Temporal Drift Protocol for {TARGET_CITY.upper()}...")

import statsmodels.api as sm
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def build_st_graph_tensors_temporal_external(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4,
                                             cutoff=pd.Timestamp('2020-01-01')):
    """
    Fit KMeans on pre-cutoff (train/val) UTM coordinates, assign all data to regions,
    build adjacency on training cluster centers, then construct hourly tensors and
    return (X_tr, X_val) from pre-2020 and (X_te_2020) from 2020, with purge & scaling.
    """
    df = df_raw.copy().sort_values('Start_Time').reset_index(drop=True)
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]

    train_mask = df['Start_Time'] < cutoff
    test_mask  = df['Start_Time'].dt.year == 2020

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    kmeans.fit(df.loc[train_mask, ['_X','_Y']].values)
    df['Region_ID'] = kmeans.predict(df[['_X','_Y']].values)

    centers_utm = kmeans.cluster_centers_
    adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    X, y, t_end = [], [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len]); y.append(city_targets[i+seq_len]); t_end.append(full_time[i+seq_len])
    X, y, t_end = np.array(X), np.array(y), np.array(t_end)

    dev_idx = np.where(t_end < cutoff)[0]
    test_idx = np.where((t_end >= pd.Timestamp('2020-01-01')) & (t_end < pd.Timestamp('2021-01-01')))[0]
    if len(dev_idx) == 0 or len(test_idx) == 0:
        raise ValueError("[ERROR] Not enough samples in dev or 2020 windows.")

    dev_X, dev_y = X[dev_idx], y[dev_idx]
    total_dev = dev_X.shape[0]
    train_bound = int(total_dev * 0.82)  # internal dev split (train/val)
    purge = seq_len

    X_tr_raw, y_tr = dev_X[:train_bound], dev_y[:train_bound]
    X_val_raw, y_val = dev_X[train_bound + purge:], dev_y[train_bound + purge:]
    X_te_raw,  y_te  = X[test_idx], y[test_idx]

    num_features = X_tr_raw.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_tr_raw.reshape(-1, num_features)).reshape(X_tr_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te  = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, adj

X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext = build_st_graph_tensors_temporal_external(
    df_target, CHAMP_NODES, CHAMP_RADIUS, CHAMP_K, seq_len=SEQ_LENGTH, cutoff=pd.Timestamp('2020-01-01')
)
print(f"[*] Validation Era Prevalence (Pre-2020) : {float(y_val_ext.mean()):.4f}")
print(f"[*] Adversarial Era Prevalence (2020)    : {float(y_te_2020.mean()):.4f}")

pos_ext = y_tr_ext.sum(); neg_ext = (y_tr_ext.size - pos_ext)
pos_weight_ext = float(neg_ext / max(pos_ext, 1))

model_drift = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr_ext.shape[-1], adj_ext, pos_weight_ext)
model_drift.fit(X_tr_ext, y_tr_ext, validation_data=(X_val_ext, y_val_ext), epochs=15, batch_size=64, verbose=0)

y_pred_probs_2020 = model_drift.predict(X_te_2020, verbose=0).flatten()
y_true_2020 = y_te_2020.flatten()

def bootstrap_pr_auc(y_true, y_pred, n_bootstraps=500):
    scores = []
    N = len(y_pred)
    for _ in range(n_bootstraps):
        idx = np.random.randint(0, N, N)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(average_precision_score(y_true[idx], y_pred[idx]))
    if len(scores) == 0:
        return np.nan, np.nan
    return np.percentile(scores, [2.5, 97.5])

drift_auc_pr = average_precision_score(y_true_2020, y_pred_probs_2020)
boot_ci_lo, boot_ci_hi = bootstrap_pr_auc(y_true_2020, y_pred_probs_2020)
drift_brier = brier_score_loss(y_true_2020, y_pred_probs_2020)

def calibration_slope_intercept(y_true, p_hat):
    eps = 1e-6
    logit_ph = np.log(np.clip(p_hat, eps, 1-eps) / np.clip(1-p_hat, eps, 1-eps))
    X = sm.add_constant(logit_ph)  # intercept + slope
    model = sm.GLM(y_true, X, family=sm.families.Binomial())
    res = model.fit()
    return float(res.params[0]), float(res.params[1])

cal_intercept_2020, cal_slope_2020 = calibration_slope_intercept(y_true_2020, y_pred_probs_2020)

print("\n========================================================")
print("[*] TEMPORAL DRIFT & CALIBRATION ASSESSMENT (2020)")
print("========================================================")
print(f"Ranking Retention (PR-AUC) : {drift_auc_pr:.4f} (95% CI: [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}])")
print(f"Calibration Brier Score    : {drift_brier:.4f}")
print(f"Calibration Intercept      : {cal_intercept_2020:.3f} (Deviation from 0 = Risk Shift)")
print(f"Calibration Slope          : {cal_slope_2020:.3f} (Deviation from 1 = Over/Underfitting)")

prob_true, prob_pred = calibration_curve(y_true_2020, y_pred_probs_2020, n_bins=10, strategy='quantile')
bins = np.percentile(y_pred_probs_2020, np.linspace(0, 100, 11))
bin_indices = np.digitize(y_pred_probs_2020, bins[1:-1])
bin_counts = np.bincount(bin_indices, minlength=10)

print("\n[*] Reliability Curve Binning Sanity Check:")
for idx, count in enumerate(bin_counts):
    print(f"    -> Bin {idx+1}: n = {int(count)} samples")
if np.min(bin_counts) < 50:
    print("    [WARNING] Some bins have < 50 samples. Consider reducing n_bins for smoother calibration estimates.")

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='ST-GNN (2020 Adversarial Test)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.title(f"Reliability Curve: Structural Integrity vs Baseline Shift\nCity: {TARGET_CITY.upper()} (2020)")
plt.xlabel("Mean Predicted Probability"); plt.ylabel("Fraction of Actual STPCs")
plt.legend(); plt.grid(True); plt.show()

#Iowa
(The Rural Control): A vast, empty rural state with massive distances between nodes, serving as the ultimate spatial control group to test how the model handles extreme sparsity.

#Experiment C — Rural External Control Domain

#Experiment C1 — Urban-to-Rural Transfer Stress Test
 Frozen LA Topology -> Iowa (Statewide External Control Domain)

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
 * Null Hypothesis (H0): Traffic shockwaves are purely temporal. Spatially blind models
   (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather)
  are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition
   of a graph-based spatial Adjacency Matrix provides no statistically significant improvement.

 * Alternative Hypothesis (H1): A topology discovered in a dense urban system (Los Angeles)
   retains predictive value when transferred unchanged into a sparse rural external domain
   (Iowa). The ST-GNN integrating the frozen LA topology will achieve a statistically
   significant improvement in AUC-PR over spatially blind baselines (α = 0.05), as verified
   by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENTAL INITIALIZATION & HARDWARE CONFIGURATION
# ==============================================================================
# REPRODUCIBILITY NOTE — Tested library versions:
# tensorflow>=2.13, scikit-learn>=1.3, pandas>=2.0, numpy>=1.24,
# scipy>=1.11, folium>=0.14, seaborn>=0.12, matplotlib>=3.7
# Python 3.10+ recommended. Run on Google Colab with T4 High-RAM or GPU T4/A100.
# ==============================================================================

# 1. System, OS, Memory & Time Utilities
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for a clean output log

# 2. Core Data Manipulation & Math Engine
import numpy as np
import pandas as pd

# 3. Visualization & Geospatial Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

# 4. Scientific & Statistical Libraries
from scipy.stats import wilcoxon
import scipy.stats as stats

# 5. Scikit-Learn (EDA, Baselines, Clustering, & Metrics)
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # SafeNodeClassifier

# 6. Deep Learning (TensorFlow & Keras - Explicit Class Imports)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout,
    Reshape, Layer, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# 7. Geodesy & Indexing
import math
import pyproj

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    """Locks all random states to ensure experimental reproducibility across runs."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
# Target domain for Experiment C1
# NOTE: Iowa is handled as a STATE-LEVEL extraction (State == 'IA'), not a city.
TARGET_CITY = "Iowa"

# Frozen LA topology (Experiment C1: transfer stress test)
SEARCH_NODES = [100]
SEARCH_RADII = [2.0]
SEARCH_K = [3]
SEQ_LENGTH = 4

# Feature columns used in tensor generation AND feature importance analysis.
FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'])}

print(f"[*] Master Pipeline Configured for EXPERIMENT C1: {TARGET_CITY.upper()} (Statewide Rural External Control)")

# ==============================================================================
# 1.3 HARDWARE ACCELERATOR INITIALIZATION (Google TPU v5e or GPU fallback)
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"[*] Network TPU Address Found: {os.environ['COLAB_TPU_ADDR']}")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"[SUCCESS] Distributed Strategy initialized on TPU: {tpu.master()}")
    print(f"[SUCCESS] Number of TPU accelerators: {strategy.num_replicas_in_sync}")
except Exception:
    print(f"[*] TPU Initialization Failed. Checking for High-Performance GPUs...")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
        except RuntimeError as memory_error:
            print(f"[ERROR] GPU Memory Error: {memory_error}")
            strategy = tf.distribute.get_strategy()
    else:
        print("[WARNING] No TPU or GPU detected. Running on standard CPU.")
        strategy = tf.distribute.get_strategy()

# ==============================================================================
# 1.4 LOCAL STORAGE CONFIGURATION (High-Speed I/O)
# ==============================================================================
BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")

## PHASE 2: Universal Data Engineering Pipeline
**Methodology:**

The raw national dataset must undergo strict spatial bounding. For Experiment C1,
 Iowa is not treated as a city; it is a state-level rural external control domain.
 Therefore, filtering is performed using State == 'IA'.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================

print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")

print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')

csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, target_name):
    """
    Loads the massive national dataset and strictly filters it to the target domain.
    For Iowa, bounding is performed at the STATE level (State == 'IA').
    For urban experiments, bounding can still be performed at the CITY level.
    """
    print(f"[*] Extracting raw records for {target_name} (This may take a minute)...")
    usecols = [
        'ID', 'City', 'State', 'Start_Time', 'End_Time',
        'Start_Lat', 'Start_Lng', 'Severity',
        'Distance(mi)', 'Humidity(%)'
    ]
    df_raw = pd.read_csv(file_path, usecols=usecols)

    # 1. Spatial Bounding (State-level for Iowa, city-level otherwise)
    if str(target_name).strip().lower() == "iowa":
        df_city = df_raw[df_raw['State'].astype(str).str.upper() == 'IA'].copy()
        print(f"[*] State Bounding Complete: {len(df_city)} raw accident records found in Iowa (State == 'IA').")
    else:
        df_city = df_raw[df_raw['City'].fillna('').str.lower() == str(target_name).strip().lower()].copy()
        print(f"[*] City Bounding Complete: {len(df_city)} raw accident records found in {target_name}.")

    # 2. Temporal Standardization
    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    # 3. Continuous Feature Cleaning (Imputation strategy)
    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)

    # 4. Final verification of required spatial coordinates
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])

    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files
gc.collect()


### 2.1B Exploratory Data Analysis (EDA)


In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"[*] Running EDA for {TARGET_CITY.upper()}...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()}', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)')
axes[0].set_ylabel('Accident Count')
axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#d7191c', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level')
axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")

df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore')
gc.collect()
print("\n[SUCCESS] EDA Complete.")


### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)


In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING (Causation & Leakage Fix)
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")

EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"\n[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad'])
gc.collect()
print("\n[SUCCESS] Phase 2.2 Complete: Multi-Radius Targets Engineered Successfully.")

### 2.2B Class Imbalance Report


In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f'[*] Class Imbalance Report for {TARGET_CITY.upper()}:')
print(f'    Total records: {len(df_target):,}')
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target)
        ratio = neg / pos if pos > 0 else float('inf')
        print(f'    Radius {r}mi | STPCs: {pos:,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1')
print('[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.')

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)


In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR (GEODESY FIX APPLIED)
# ==============================================================================
print("[*] Initializing Master Tensor Generator with metric geometry (UTM)...")

def _dynamic_utm_transformer(df_like):
    """Create a domain-specific UTM transformer based on median lon/lat."""
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    """Project (lon, lat) -> (x, y) in meters using the provided transformer."""
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    """Build KNN adjacency on UTM centers with inverse-distance weights (meters)."""
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w

    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)

    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    """Aggregate to hourly time steps and pivot to dense (time, region, features)."""
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(FEATURES_LIST)))
    for idx, feat in enumerate(FEATURES_LIST):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)

    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    """Build sliding windows; split chronologically with overlap purge; scale features on train only."""
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                    y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound], y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound + purge :],             y[val_bound + purge :]

    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError(f"[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    """
    MASTER GENERATOR (GEODESY FIX): KMeans on dynamic UTM; adjacency on UTM centers (inverse meters).
    """
    print(f"\n========================================================")
    print(f"[*] GENERATING PIPELINE: {n_nodes} Nodes | {radius_mi}mi Radius | K={k_neighbors}")
    print(f"========================================================")

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    print(f"[1/5] Clustering coordinates into {n_nodes} Macro-Regions (UTM={epsg_code})...")
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    print(f"[2/5] Building Adjacency Matrix (K={k_neighbors}) with inverse-meters weights...")
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)

    print("[3/5] Aggregating timestamps into hourly steps...")
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    print(f"[4/5] Pivoting into 4D Sliding Windows (Seq_Len={seq_len})...")
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)

    print(f"[5/5] Final X_tr shape: {X_tr.shape}")
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE (Null Hypotheses)
# ==============================================================================
print("[*] Compiling Spatially-Blind Baseline Architectures...")

def get_weighted_bce(pos_weight_val):
    """Applies a heavy asymmetric penalty to False Negatives (minority class)."""
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    """Spatially blind LSTM: flattens nodes to prevent topological learning."""
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    """
    Catches empty geographic nodes to prevent Scikit-Learn math failures.
    If a node has only one class in training (all 0s or all 1s), the wrapped
    estimator would fail. We short-circuit to a degenerate predictor.
    """
    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        self.classes_ = np.array([0, 1], dtype=int)

        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self

    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)

    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        return self.model_.predict_proba(X)

    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    """
    Deterministic traditional ML baselines wrapped with SafeNodeClassifier
    so that empty/single-class nodes never crash the pipeline.
    """
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)
    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

print("[SUCCESS] Phase 3 Complete: Baseline Suite (LR, RF, LSTM) Loaded (SafeNodeClassifier active).")


## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
print("[*] Compiling Custom Graph Convolutional Layer...")

class SpatioTemporalGCNLayer(Layer):
    """GCN Layer: normalized adjacency message passing across macro-nodes."""
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(
            name='gcn_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        super(SpatioTemporalGCNLayer, self).build(input_shape)

    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1)
        gcn_1 = Dropout(0.3)(gcn_1)

        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2)
        gcn_2 = Dropout(0.3)(gcn_2)

        gcn_combined = layers.Add()([gcn_1, gcn_2])

        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out)
        gru_out = Dense(32, activation='relu')(gru_out)

        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)

        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model


## PHASE 5: Frozen-Topology Validation Pass (Single Configuration)
**Methodology:** In Experiment C1, topology discovery is NOT performed. The LA champion topology
 is frozen by design. This phase executes a single validation pass on Iowa under
 the transferred LA topology to produce a validation reference before the final proof.

In [ ]:
# ==============================================================================
# PHASE 5: Frozen-Topology Validation Pass (Single Configuration)
# ==============================================================================
print("[*] Initializing Phase 5: Frozen LA Topology Validation Pass...")

experiment_results = []

# SCIENTIFIC FIX: Silo Experiment C1 outputs to prevent cross-experiment matrix corruption
RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C1_Validation.csv"

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: Frozen LA Topology -> {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")

            model = None
            X_tr = X_val = X_te = y_tr = y_val = y_te = adj = None
            early_stop = None

            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target,
                    n_nodes=n_nodes,
                    radius_mi=radius_mi,
                    k_neighbors=k_neighbors,
                    seq_len=SEQ_LENGTH
                )

                total_pos = np.sum(y_tr)
                total_neg = (y_tr.size) - total_pos

                if total_pos == 0:
                    print(f"[WARNING] Topology yielded 0 STPCs. Skipping configuration.")
                    current_exp += 1
                    continue

                # SCIENTIFIC FIX: Guard against scientifically meaningless sparse-positive training
                if total_pos < 10:
                    print(f"[WARNING] Topology yielded only {int(total_pos)} positive STPC labels in training. Skipping configuration as scientifically unstable.")
                    current_exp += 1
                    continue

                # SCIENTIFIC FIX: Cap extreme imbalance weights to prevent exploding gradients
                raw_weight = float(total_neg / total_pos)
                pos_weight_val = min(raw_weight, 1000.0)

                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg} | Capped Pos_Weight: {pos_weight_val:.2f}")

                model = build_st_gnn_model(
                    time_steps=SEQ_LENGTH,
                    n_nodes=n_nodes,
                    n_features=X_tr.shape[-1],
                    adj_matrix=adj,
                    pos_weight=pos_weight_val
                )

                print("[*] Commencing Training...")

                # Fresh callback per experiment
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                history = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=15,
                    batch_size=64,
                    callbacks=[early_stop],
                    verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                print("[*] Evaluating Predictions on Validation Set...")
                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten()
                y_pred_flat = y_pred_probs_val.flatten()

                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]
                    final_recall = recalls[best_idx]
                    final_f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'City': TARGET_CITY,
                    'Experiment': 'C1',
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(float(pos_weight_val), 2),
                    'Val_AUC_PR': round(float(auc_pr), 4),
                    'Optimal_Threshold': round(float(optimal_threshold), 4),
                    'Precision_at_70_Recall': round(float(final_precision), 4),
                    'Actual_Recall': round(float(final_recall), 4),
                    'Val_F1_Score': round(float(final_f1), 4)
                })

                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            finally:
                # SCIENTIFIC FIX: Safe cleanup (no unsafe del of undefined variables)
                if model is not None:
                    del model
                if early_stop is not None:
                    del early_stop
                if X_tr is not None:
                    del X_tr, X_val, X_te, y_tr, y_val, y_te, adj

                tf.keras.backend.clear_session()
                gc.collect()
                current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. Frozen topology validated on Iowa.")
print(f"[*] Validation results securely saved to: {RESULTS_PATH}")

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
# ==============================================================================
print(f"[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

# 1) FROZEN LA CHAMPION TOPOLOGY (Experiment C1: Urban-to-Rural Transfer Stress Test)
CHAMP_NODES = SEARCH_NODES[0]
CHAMP_RADIUS = SEARCH_RADII[0]
CHAMP_K = SEARCH_K[0]
print(f"[*] Frozen LA Topology Applied to {TARGET_CITY.upper()}: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# 2) ISOLATE THE FINAL TENSORS
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target,
    n_nodes=CHAMP_NODES,
    radius_mi=CHAMP_RADIUS,
    k_neighbors=CHAMP_K,
    seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Frozen LA topology produced zero STPC labels in Iowa training set.")

# SCIENTIFIC FIX: guard against scientifically meaningless extremely sparse training
if total_pos < 10:
    raise ValueError(f"[ERROR] Frozen LA topology produced only {int(total_pos)} positive STPC labels in Iowa training set. Experiment C1 is not scientifically stable under this support level.")

# SCIENTIFIC FIX: cap pos_weight to prevent exploding gradients
raw_weight = float(((y_tr.size) - total_pos) / total_pos)
pos_weight_val = min(raw_weight, 1000.0)

y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# 3) TRADITIONAL ML BASELINES (LR & RF)
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")

X_tr_ml = X_tr.reshape(X_tr.shape[0], -1)
X_te_ml = X_te.reshape(X_te.shape[0], -1)

lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)

    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)

    # Safe probability extraction per node
    for i in range(y_te.shape[1]):
        if isinstance(preds, list):
            node_preds = preds[i]
        else:
            node_preds = preds[:, i, :]

        if node_preds.shape[1] >= 2:
            y_pred_probs[:, i] = node_preds[:, 1]
        else:
            y_pred_probs[:, i] = 0.0

    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)

    for i in range(y_val.shape[1]):
        if isinstance(val_preds, list):
            node_val_preds = val_preds[i]
        else:
            node_val_preds = val_preds[:, i, :]

        if node_val_preds.shape[1] >= 2:
            y_val_probs[:, i] = node_val_preds[:, 1]
        else:
            y_val_probs[:, i] = 0.0

    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]
    opt_thresh = thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5

    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)

    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return float(auc_pr), float(f1)

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")

del X_tr_ml, X_te_ml, lr_model, rf_model
gc.collect()

# 4) DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return float(auc_pr), float(f1)

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return float(thresholds[valid_idx[np.argmax(precisions[valid_idx])]]) if len(valid_idx) > 0 else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")
    tf.random.set_seed(42 + run)

    lstm_model = None
    stgnn_model = None
    early_stop_lstm = None
    early_stop_stgnn = None

    try:
        # LSTM baseline
        lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
        early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        lstm_model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=64,
            callbacks=[early_stop_lstm],
            verbose=0
        )

        thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
        auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
        lstm_auc_scores.append(auc_lstm)
        lstm_f1_scores.append(f1_lstm)
        print(f"[*] LSTM   -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")

    finally:
        if lstm_model is not None:
            del lstm_model
        if early_stop_lstm is not None:
            del early_stop_lstm
        tf.keras.backend.clear_session()
        gc.collect()

    try:
        # Frozen-topology ST-GNN
        stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
        early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        stgnn_model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=64,
            callbacks=[early_stop_stgnn],
            verbose=0
        )

        thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
        auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
        stgnn_auc_scores.append(auc_stgnn)
        stgnn_f1_scores.append(f1_stgnn)
        print(f"[*] ST-GNN -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")

    finally:
        if stgnn_model is not None:
            del stgnn_model
        if early_stop_stgnn is not None:
            del early_stop_stgnn
        tf.keras.backend.clear_session()
        gc.collect()

# 5) Scientific logging & hypothesis testing
lstm_auc_mean  = float(np.mean(lstm_auc_scores))
lstm_auc_std   = float(np.std(lstm_auc_scores))
stgnn_auc_mean = float(np.mean(stgnn_auc_scores))
stgnn_auc_std  = float(np.std(stgnn_auc_scores))

lstm_ci  = stats.t.interval(0.95, df=N_RUNS-1, loc=lstm_auc_mean,  scale=stats.sem(lstm_auc_scores))
stgnn_ci = stats.t.interval(0.95, df=N_RUNS-1, loc=stgnn_auc_mean, scale=stats.sem(stgnn_auc_scores))

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()} (Experiment C1)")
print("==================================================================")
print(f"Transferred Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"Baseline LSTM AUC-PR       : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"Transferred ST-GNN AUC-PR  : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon): p-value = {p_value:.4f}")
    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] The transferred LA topology retains statistically significant predictive value in Iowa.")
    else:
        print("[CONCLUSION] The transferred LA topology does not achieve statistical significance over the temporal baseline in Iowa at α=0.05.")
except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This can occur when all paired score differences are zero.")

# SCIENTIFIC FIX: Silo Experiment C1 proof output
PROOF_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C1_Final_Proof.csv"
proof_df = pd.DataFrame([{
    'Domain': TARGET_CITY,
    'Experiment': 'C1',
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'Champion_K': CHAMP_K,
    'LR_AUC': round(lr_auc, 4),
    'RF_AUC': round(rf_auc, 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'LSTM_CI_Lower': round(float(lstm_ci[0]), 4),
    'LSTM_CI_Upper': round(float(lstm_ci[1]), 4),
    'STGNN_CI_Lower': round(float(stgnn_ci[0]), 4),
    'STGNN_CI_Upper': round(float(stgnn_ci[1]), 4),
    'Wilcoxon_p_value': round(float(p_value), 4) if p_value is not None else None
}])

if os.path.exists(PROOF_RESULTS_PATH):
    proof_df.to_csv(PROOF_RESULTS_PATH, mode='a', header=False, index=False)
else:
    proof_df.to_csv(PROOF_RESULTS_PATH, index=False)

print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")



In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (AUC-PR)
# ==============================================================================
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Transferred\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(
    f'Urban-to-Rural Transfer Stress Test (AUC-PR)\nDomain: {TARGET_CITY.upper()} | Frozen LA Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
    fontsize=13, fontweight='bold', pad=15
)
plt.ylabel('AUC-PR (Higher is Better for Extreme Imbalance)', fontsize=11, fontweight='bold')
max_y = max(auc_scores)
plt.ylim(0, max_y * 1.35)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(
        f'AUC: {height:.4f}{std_text}',
        (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
        ha='center', va='bottom',
        fontsize=11, color='black', xytext=(0, 8),
        textcoords='offset points', fontweight='bold'
    )

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C1_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()

#Diagnostic Step C1.5 — Iowa Spatial Feasibility and Search-Space Justification

In [ ]:
# ==============================================================================
# STEP C1.5: RURAL DOMAIN GEOMETRY DIAGNOSTIC (IOWA STATEWIDE)
# The output of this script empirically justifies the C2 Hyperparameter Search Space.
# ==============================================================================
import os
import glob
import math
import numpy as np
import pandas as pd
import pyproj
import kagglehub
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

print("[*] Initiating Step C1.5: Rural Domain Geometry Diagnostic (Iowa)")

# --- CONFIGURATION (Aligned with Master Pipeline) ---
TARGET_STATE = 'IA'
SEQ_LENGTH = 4
SEARCH_NODES = [50, 100]
SEARCH_RADII = [5.0, 10.0, 15.0]
SEARCH_K = [2, 3, 4]
MAX_SEARCH_RADIUS_MILES = 50.0
EARTH_RADIUS_MILES = 3958.8

# ==============================================================================
# 1. DATA EXTRACTION & LEAKAGE PREVENTION
# ==============================================================================
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')
csv_path = glob.glob(os.path.join(dataset_dir, '*.csv'))[0]

usecols = ['ID', 'State', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng']
df_raw = pd.read_csv(csv_path, usecols=usecols)

# Extract Iowa explicitly as a state-level domain
df_ia = df_raw[df_raw['State'].astype(str).str.upper() == TARGET_STATE].copy()
df_ia['Start_Time'] = pd.to_datetime(df_ia['Start_Time'], errors='coerce')
df_ia = df_ia.dropna(subset=['Start_Time', 'Start_Lat', 'Start_Lng'])
df_ia = df_ia.sort_values('Start_Time').reset_index(drop=True)

# Replicate the tensor generator's hourly timeline to find the exact 70% boundary
df_ia['Time_Step'] = df_ia['Start_Time'].dt.floor('h')
start_min = df_ia['Time_Step'].min()
end_max = df_ia['Time_Step'].max()
full_time = pd.date_range(start=start_min, end=end_max, freq='h')

total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)
cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]

df_train = df_ia[df_ia['Start_Time'] < cutoff_time].copy().reset_index(drop=True)
print(f"[*] Temporal Guard Active: Analyzing ONLY the {len(df_train)} training-era records (before {cutoff_time}).\n")

# --- RUNTIME HYGIENE GUARDS ---
if len(df_train) == 0:
    raise ValueError("[ERROR] Training-era Iowa subset is empty under the current cutoff.")
if len(df_train) < max(SEARCH_NODES):
    raise ValueError(
        f"[ERROR] Training-era Iowa subset has only {len(df_train)} records, "
        f"which is smaller than max SEARCH_NODES={max(SEARCH_NODES)}."
    )

# ==============================================================================
# 2. C1.5A: RADIUS PHYSICS (Temporally Eligible Spatial Spacing)
# ==============================================================================
df_train['Lat_rad'] = np.radians(df_train['Start_Lat'])
df_train['Lng_rad'] = np.radians(df_train['Start_Lng'])
timestamps = df_train['Start_Time'].values.astype('datetime64[ns]')
coords_rad = df_train[['Lat_rad', 'Lng_rad']].values

print(f"[*] Constructing Haversine BallTree (Querying up to {MAX_SEARCH_RADIUS_MILES} miles)...")
tree = BallTree(coords_rad, metric='haversine')
indices_within_max, distances_within_max = tree.query_radius(
    coords_rad, r=(MAX_SEARCH_RADIUS_MILES / EARTH_RADIUS_MILES), return_distance=True
)

nearest_eligible_distances = []
coverage_count = 0

# Store event-level STPC flags for aggregation later
event_stpc_flags = {r: np.zeros(len(df_train), dtype=int) for r in SEARCH_RADII}

for i, (neighbors, dists) in enumerate(zip(indices_within_max, distances_within_max)):
    # Look strictly backward in time
    past_mask = neighbors < i
    past_neighbors = neighbors[past_mask]
    past_dists_miles = dists[past_mask] * EARTH_RADIUS_MILES

    if len(past_neighbors) > 0:
        time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
        # Enforce the strict 2-hour STPC eligibility logic
        eligible_mask = (time_diffs_hours > 0) & (time_diffs_hours <= 2.0)

        if np.any(eligible_mask):
            coverage_count += 1
            eligible_dists = past_dists_miles[eligible_mask]
            nearest_eligible_distances.append(np.min(eligible_dists))

            for r in SEARCH_RADII:
                if np.any(eligible_dists <= r):
                    event_stpc_flags[r][i] = 1

nearest_eligible_distances = np.array(nearest_eligible_distances)
for r in SEARCH_RADII:
    df_train[f'STPC_{r}'] = event_stpc_flags[r]

# Safe division guard
coverage_pct = (coverage_count / len(df_train) * 100) if len(df_train) > 0 else 0.0

# ==============================================================================
# 3. C1.5B & C1.5C: GRAPH REALISM & OBSERVED LABEL SUPPORT
# ==============================================================================
print("[*] Projecting to UTM and evaluating Graph Connectivity & Target Prevalence...")

lon0, lat0 = df_train['Start_Lng'].median(), df_train['Start_Lat'].median()
zone = int(math.floor((lon0 + 180)/6) + 1)
epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
x, y = transformer.transform(df_train['Start_Lng'].values, df_train['Start_Lat'].values)
coords_utm = np.c_[x, y]

connectivity_results = []
aggregated_support_results = []

for n_nodes in SEARCH_NODES:
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df_train['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    # C1.5C: Observed Target Support (Pre-tensor aggregation)
    agg_df = df_train.groupby(['Region_ID', 'Time_Step']).agg({
        f'STPC_{r}': 'sum' for r in SEARCH_RADII
    }).reset_index()

    for r in SEARCH_RADII:
        # A region-hour is positive if it has AT LEAST ONE STPC
        observed_positives = (agg_df[f'STPC_{r}'] > 0).sum()
        aggregated_support_results.append({
            'Nodes': n_nodes,
            'Radius_mi': r,
            'Raw_Event_STPCs': df_train[f'STPC_{r}'].sum(),
            'Observed_Positive_RegionHours': observed_positives
        })

    # C1.5B: Spatial-Proximity Graph Realism (Edge Forensics)
    for k in SEARCH_K:
        knn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
        knn.fit(centers_utm)
        distances, indices = knn.kneighbors(centers_utm)

        # Convert UTM meters to miles for readability
        distances_miles = distances / 1609.34

        edge_lengths = []
        adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
        for i in range(n_nodes):
            for j in range(1, k + 1):
                nb = indices[i, j]
                adj_matrix[i, nb] = 1.0
                adj_matrix[nb, i] = 1.0
                edge_lengths.append(distances_miles[i, j])

        edge_lengths = np.array(edge_lengths)
        graph_csr = csr_matrix(adj_matrix)
        n_components, _ = connected_components(csgraph=graph_csr, directed=False, return_labels=True)

        connectivity_results.append({
            'Nodes': n_nodes,
            'K': k,
            'Components': n_components,
            'Med_Edge_mi': round(np.median(edge_lengths), 1),
            '90th_Edge_mi': round(np.percentile(edge_lengths, 90), 1),
            'Max_Edge_mi': round(np.max(edge_lengths), 1)
        })

# ==============================================================================
# 4. SCIENTIFIC OUTPUT LOG (Directly supports Methodology Chapter)
# ==============================================================================
print("\n==================================================================")
print("[*] STEP C1.5: IOWA RURAL GEOMETRY & FEASIBILITY REPORT")
print("==================================================================")

print("--- [C1.5A] TEMPORALLY ELIGIBLE SPATIAL SPACING ---")
print(f"Temporal-Spatial Coverage Rate (≤2h, ≤50mi): {coverage_pct:.2f}%")
if len(nearest_eligible_distances) > 0:
    print(f"  -> Median (50th %ile) : {np.median(nearest_eligible_distances):.2f} miles")
    print(f"  -> 75th Percentile    : {np.percentile(nearest_eligible_distances, 75):.2f} miles")
    print(f"  -> 90th Percentile    : {np.percentile(nearest_eligible_distances, 90):.2f} miles")
else:
    print("[!] Zero temporally eligible pairs found within 50 miles.")

print("\n--- [C1.5C] OBSERVED LABEL SUPPORT (Prevalence Check) ---")
support_df = pd.DataFrame(aggregated_support_results)
print(support_df.to_string(index=False))
print("\n*(Note: Configurations with very low observed positive region-hours (e.g., <20)")
print("  should be treated as statistically unstable and unsuitable for reliable neural training.)*")

print("\n--- [C1.5B] SPATIAL-PROXIMITY GRAPH REALISM ---")
conn_df = pd.DataFrame(connectivity_results)
print(conn_df.to_string(index=False))
print("\n*(Note: High 90th percentile edge lengths indicate artificial spatial-proximity connections)*")
print("==================================================================")

#Experiment C2: Native Rural Topology Discovery

## PHASE 1: Experimental Initialization & Hardware Configuration
To ensure strict scientific reproducibility, global random seeds are locked. The compute environment is initialized for distributed training on a Google Cloud T4 High-RAM.

**Scientific Hypotheses:**
* **Null Hypothesis (H0):** Traffic shockwaves are purely temporal. Spatially blind models (Logistic Regression, Random Forest, and LSTM — analyzing only time, distance, and weather) are sufficient to predict Spatio-Temporal Proximate Collisions (STPCs). The addition of a graph-based spatial Adjacency Matrix provides no statistically significant improvement in predictive power.
* **Alternative Hypothesis (H1):** Secondary collisions are governed by physical road topology. A Spatio-Temporal Graph Neural Network (ST-GNN) integrating a symmetrically normalized Adjacency Matrix will achieve a statistically significant improvement in Area Under the Precision-Recall Curve (AUC-PR) over all spatially blind baselines (α = 0.05), as verified by a Wilcoxon signed-rank test over n=10 independent paired runs.

In [ ]:
# ==============================================================================
# Experiment C2: Native Rural Topology Discovery (Statewide External Control)
# ==============================================================================

## PHASE 1: Experimental Initialization & Hardware Configuration
import os
import kagglehub
import glob
import gc
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import Element

from scipy.stats import wilcoxon
import scipy.stats as stats

from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import BallTree, NearestNeighbors, kneighbors_graph
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Reshape, Layer, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

import math
import pyproj
import h3

# ==============================================================================
# 1.1 REPRODUCIBILITY CONTROLS
# ==============================================================================
def set_global_determinism(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[*] Global random seeds locked at {seed} for reproducibility.")

set_global_determinism()

# ==============================================================================
# 1.2 MASTER EXPERIMENT CONTROL PANEL
# ==============================================================================
TARGET_CITY = "Iowa"

# Experiment C2: Rural-Specific Sparse Topology Discovery
SEARCH_NODES = [50, 100]
SEARCH_RADII = [5.0, 10.0, 15.0]
SEARCH_K = [2, 3, 4]
SEQ_LENGTH = 4

FEATURES_LIST = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']
NODE_COLORS = {n: c for n, c in zip(SEARCH_NODES, ['#1a9641', '#2c7bb6'])}

print(f"[*] Master Pipeline Configured for EXPERIMENT C2: {TARGET_CITY.upper()} (Statewide)")

# ==============================================================================
# 1.3 HARDWARE & STORAGE INITIALIZATION
# ==============================================================================
print("\n[*] Initializing Hardware Strategy...")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        strategy = tf.distribute.MirroredStrategy()
        print(f"[SUCCESS] Distributed Strategy initialized on {len(gpus)} GPU(s).")
    except RuntimeError as memory_error:
        print(f"[ERROR] GPU Memory Error: {memory_error}")
        strategy = tf.distribute.get_strategy()
else:
    print("[WARNING] No GPU detected. Running on standard CPU.")
    strategy = tf.distribute.get_strategy()

BASE_DIR = '/content/Traffic_Research_Artifacts'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"[*] Local High-Speed Storage Initialized: {BASE_DIR}")

## PHASE 2: Universal Data Engineering Pipeline
**Methodology:** To accurately model the "Butterfly Effect", the raw national dataset must undergo strict spatial bounding. Filtering by the target environment (e.g., Los Angeles) prevents the Spatio-Temporal Graph Neural Network (ST-GNN) from attempting to draw physically impossible topological connections across state lines.

### 2.1 Spatial Bounding & Feature Extraction
The dataset is constrained to the designated `TARGET_CITY`. Continuous environmental and spatial features are extracted, and temporal data is standardized for the sliding window sequences.

In [ ]:
# ==============================================================================
# PHASE 2: Universal Data Engineering Pipeline
# ==============================================================================
print(f"[*] Initiating Data Pipeline for: {TARGET_CITY}")
print("[*] Downloading US Accidents Dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download('sobhanmoosavi/us-accidents')
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
if not csv_files:
    raise FileNotFoundError("[ERROR] No CSV file found in the Kaggle download.")
CSV_PATH = csv_files[0]
print(f"[SUCCESS] Dataset secured at: {CSV_PATH}")

def load_and_bound_data(file_path, target_name):
    print(f"[*] Extracting raw records for {target_name}...")
    usecols = ['ID', 'City', 'State', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Severity', 'Distance(mi)', 'Humidity(%)']
    df_raw = pd.read_csv(file_path, usecols=usecols)

    if str(target_name).strip().lower() == "iowa":
        df_city = df_raw[df_raw['State'].astype(str).str.upper() == 'IA'].copy()
        print(f"[*] State Bounding Complete: {len(df_city)} raw accident records found in Iowa (State == 'IA').")
    else:
        df_city = df_raw[df_raw['City'].fillna('').str.lower() == str(target_name).strip().lower()].copy()
        print(f"[*] City Bounding Complete: {len(df_city)} raw accident records found in {target_name}.")

    print("[*] Standardizing Temporal Features...")
    df_city['Start_Time'] = pd.to_datetime(df_city['Start_Time'], errors='coerce')
    df_city['End_Time'] = pd.to_datetime(df_city['End_Time'], errors='coerce')
    df_city = df_city.dropna(subset=['Start_Time', 'End_Time'])
    df_city = df_city.sort_values('Start_Time').reset_index(drop=True)

    median_humidity = df_city['Humidity(%)'].median()
    df_city['Humidity(%)'] = df_city['Humidity(%)'].fillna(median_humidity)
    df_city['Severity'] = pd.to_numeric(df_city['Severity'], errors='coerce').fillna(2.0)
    df_city['Distance(mi)'] = pd.to_numeric(df_city['Distance(mi)'], errors='coerce').fillna(0.0)
    df_city = df_city.dropna(subset=['Start_Lat', 'Start_Lng'])
    print(f"[SUCCESS] Data Engineering Phase 2.1 Complete. Final shape: {df_city.shape}")
    return df_city

df_target = load_and_bound_data(CSV_PATH, TARGET_CITY)
del csv_files; gc.collect()

### 2.1B Exploratory Data Analysis (EDA)
**Methodology:** Before engineering any targets, the raw dataset is profiled to establish: record counts, class imbalance ratios across all three radii, temporal distribution of accidents by hour and month, and spatial density. This section produces reproducible artefacts that characterise the dataset and justify downstream modelling decisions.

In [ ]:
# ==============================================================================
# 2.1B EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
print(f"\n[*] Running EDA for {TARGET_CITY.upper()} (Experiment C2)...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Exploratory Data Analysis — {TARGET_CITY.upper()} (Statewide Rural Control)', fontsize=14, fontweight='bold')

df_target['Hour'] = df_target['Start_Time'].dt.hour
hourly_counts = df_target.groupby('Hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='#1a9641', edgecolor='black', alpha=0.85) # Rural Green
axes[0].set_title('Accident Frequency by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0–23)'); axes[0].set_ylabel('Accident Count'); axes[0].set_xticks(range(0, 24, 2))

df_target['Month'] = df_target['Start_Time'].dt.month
monthly_counts = df_target.groupby('Month').size()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_counts.index, monthly_counts.values, color='#2c7bb6', edgecolor='black', alpha=0.85)
axes[1].set_title('Accident Frequency by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Accident Count')
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(month_labels, rotation=45)

severity_counts = df_target['Severity'].value_counts().sort_index()
axes[2].bar(severity_counts.index.astype(str), severity_counts.values, color='#fdae61', edgecolor='black', alpha=0.85)
axes[2].set_title('Severity Distribution (1=Minor, 4=Critical)', fontweight='bold')
axes[2].set_xlabel('Severity Level'); axes[2].set_ylabel('Count')

plt.tight_layout()
EDA_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_EDA_Overview.png"
plt.savefig(EDA_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] EDA artifact saved to: {EDA_PATH}")
plt.show()

print("\n[*] Missing Data Summary (post-cleaning):")
missing = df_target[['Severity', 'Distance(mi)', 'Humidity(%)', 'Start_Lat', 'Start_Lng']].isnull().sum()
for col, cnt in missing.items():
    print(f"    {col:<20}: {cnt} missing values")
df_target = df_target.drop(columns=['Hour', 'Month'], errors='ignore'); gc.collect()
print("[SUCCESS] EDA Complete.\n")

### 2.2 Spatio-Temporal Target Engineering (The Haversine BallTree)
**Methodology:** The target variable (`Is_STPC`) (Spatio-Temporal Proximate Collision) is not native to the dataset; it must be mathematically derived. Critically, this label makes **no causal claim** — it identifies collisions that occur within a strict temporal window (2 hours) and strict physical radius of a prior collision, without asserting that the prior collision caused the later one. This is an intentional epistemological boundary: establishing causality would require a third data source (e.g., loop detector speed data) to confirm congestion propagation between the two points, which is beyond the scope of this pipeline.

To eliminate data leakage, the labeling logic is **strictly backward-looking**: for each accident at index `i`, only accidents at index `< i` (i.e., strictly in the past after chronological sorting) are eligible as prior triggers. This ensures no future information contaminates any label.

To prevent computationally prohibitive $O(N^2)$ distance calculations across all records in the target city, a `BallTree` spatial index is utilized with the Haversine distance metric (accounting for Earth's curvature). Targets are pre-computed across the entire Search Space boundaries (0.5mi, 1.0mi, 2.0mi) to optimize downstream tensor generation.

In [ ]:
# ==============================================================================
# 2.2 SPATIO-TEMPORAL TARGET ENGINEERING
# ==============================================================================
print(f"[*] Initiating Target Engineering for {len(df_target)} records...")
EARTH_RADIUS_MILES = 3958.8
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)
df_target['Lat_rad'] = np.radians(df_target['Start_Lat'])
df_target['Lng_rad'] = np.radians(df_target['Start_Lng'])

print("[*] Constructing Haversine BallTree spatial index...")
tree = BallTree(df_target[['Lat_rad', 'Lng_rad']].values, metric='haversine')
timestamps = df_target['Start_Time'].values.astype('datetime64[ns]')

for radius_mi in SEARCH_RADII:
    col_name = f'Is_STPC_{radius_mi}'
    target_array = np.zeros(len(df_target), dtype=int)
    print(f"[*] Calculating STPC (Proximate Collisions) for radius: {radius_mi} miles...")
    radius_rad = radius_mi / EARTH_RADIUS_MILES
    indices_within_radius = tree.query_radius(df_target[['Lat_rad', 'Lng_rad']].values, r=radius_rad)

    for i, neighbors in enumerate(indices_within_radius):
        past_neighbors = neighbors[neighbors < i]
        if len(past_neighbors) > 0:
            time_diffs_hours = (timestamps[i] - timestamps[past_neighbors]) / np.timedelta64(1, 'h')
            if np.any((time_diffs_hours > 0) & (time_diffs_hours <= 2.0)):
                target_array[i] = 1

    df_target[col_name] = target_array
    print(f"[+] Radius {radius_mi}mi complete. Discovered {int(target_array.sum())} STPC labels.")

df_target = df_target.drop(columns=['Lat_rad', 'Lng_rad']); gc.collect()

### 2.2B Class Imbalance Report
**Methodology:** Now that STPC labels have been computed across all three radii, the class distribution is reported. This quantifies the severity of imbalance that justifies the use of AUC-PR over accuracy and the weighted BCE loss function.

In [ ]:
# ==============================================================================
# 2.2B CLASS IMBALANCE REPORT
# ==============================================================================
print(f"\n[*] Class Imbalance Report for {TARGET_CITY.upper()} (Experiment C2):")
print(f"    Total records: {len(df_target):,}")
for r in SEARCH_RADII:
    col = f'Is_STPC_{r}'
    if col in df_target.columns:
        pos = int(df_target[col].sum())
        neg = len(df_target) - pos
        pct = 100 * pos / len(df_target) if len(df_target) > 0 else 0
        ratio = neg / pos if pos > 0 else float('inf')
        print(f"    Radius {r:>4.1f}mi | STPCs: {pos:<5,} ({pct:.2f}%) | Neg:Pos ratio = {ratio:.1f}:1")
print("[SUCCESS] Class imbalance quantified. AUC-PR and weighted BCE are justified.\n")

### 2.3 Topological Construction & Strict Tensor Scaling (The Generator)
**Methodology:** To construct the Graph Convolutional network, independent GPS coordinates must be unified into a contiguous graph.
1. **Macro-Node Generation:** K-Means clustering is applied to group raw coordinates into macro-regions (`n_nodes`), solving coordinate sparsity. Note: Standard K-Means uses Euclidean distance on unprojected GPS coordinates. At mid-latitudes (e.g., Los Angeles at ~34°N), one degree of longitude is physically ~17% shorter than one degree of latitude, introducing mild cluster distortion. For city-scale bounding boxes this distortion is negligible for macro-region mapping but is formally acknowledged as a limitation.
2. **Adjacency Topology:** K-Nearest Neighbors (KNN) maps the physical highway connections (`k_neighbors`) between these macro-regions using **inverse Haversine distance weights** (`w = 1 / (d + ε)`). Unlike binary adjacency, weighted edges encode spatial proximity strength — closer macro-regions propagate stronger signals through the GCN, reflecting the physical reality that accidents nearer to a primary event are more likely to be causally related.
3. **Temporal Aggregation:** Data is compressed into discrete 1-hour time steps.
4. **Strict Isolation & Scaling:** To prevent Data Leakage, the 4D Spatio-Temporal tensors are split chronologically (70/15/15 — Train/Validation/Test) *prior* to MinMax Scaling. The scaler is fitted exclusively on the training distribution to mimic real-world deployment.
5. **Temporal Overlap Purge:** Because windows slide with stride=1 and `seq_len=4`, adjacent splits share `seq_len−1 = 3` overlapping input hours at each boundary. A purge gap of `seq_len` steps is removed from the start of both Validation and Test sets, ensuring a clean temporal break with zero input overlap. The data loss is 2 × `seq_len` samples — negligible relative to tens of thousands of hourly windows.

In [ ]:
# ==============================================================================
# 2.3 MASTER SPATIO-TEMPORAL TENSOR GENERATOR
# ==============================================================================
def _dynamic_utm_transformer(df_like):
    lon0, lat0 = df_like['Start_Lng'].median(), df_like['Start_Lat'].median()
    zone = int(math.floor((lon0 + 180)/6) + 1)
    epsg_code = f"epsg:{32600 + zone}" if lat0 >= 0 else f"epsg:{32700 + zone}"
    transformer = pyproj.Transformer.from_crs("epsg:4326", epsg_code, always_xy=True)
    return transformer, epsg_code

def _project_to_utm(df_like, transformer):
    x, y = transformer.transform(df_like['Start_Lng'].values, df_like['Start_Lat'].values)
    return np.c_[x, y]

def _build_adjacency_from_centers_utm(centers_xy, k_neighbors):
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)
    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)
            w = 1.0 / d
            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w

    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)

    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = (deg == 0).sum()
        raise ValueError(f"[CRITICAL] Adjacency contains {iso} isolated nodes. Increase K or adjust topology.")
    return normalized_adj

def _aggregate_to_hourly(df, target_col):
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')
    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')
    multi_idx = pd.MultiIndex.from_product([sorted(df['Region_ID'].unique()), full_time],
                                           names=['Region_ID', 'Time_Step'])
    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    features_list = FEATURES_LIST
    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(features_list)))
    for idx, feat in enumerate(features_list):
        city_features[:, :, idx] = continuous_df.pivot(index='Time_Step', columns='Region_ID', values=feat).fillna(0).values

    city_targets = continuous_df.pivot(index='Time_Step', columns='Region_ID', values='STPC_Count').fillna(0).values
    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time

def _windows_and_split(city_features, city_targets, seq_len=4, purge=4, train_frac=0.70, val_frac=0.15):
    X, y = [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
    X, y = np.array(X), np.array(y)

    total_samples = X.shape[0]
    train_bound = int(total_samples * train_frac)
    val_bound   = int(total_samples * (train_frac + val_frac))

    X_train_raw, y_tr  = X[:train_bound],                      y[:train_bound]
    X_val_raw,   y_val = X[train_bound + purge : val_bound],   y[train_bound + purge : val_bound]
    X_te_raw,    y_te  = X[val_bound   + purge :],             y[val_bound   + purge :]

    # SCIENTIFIC FIX 3: Empty Split Guard
    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError("[ERROR] Temporal purge emptied val or test. Reduce seq_len or use more data.")

    num_features = X.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_train_raw.reshape(-1, num_features)).reshape(X_train_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

def build_st_graph_tensors(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

def build_st_graph_tensors_preclustered(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4):
    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'
    if 'Region_ID' not in df.columns:
        raise ValueError("[ERROR] Region_ID missing for preclustered pipeline.")
    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]
    centers_utm = df.groupby('Region_ID')[['_X','_Y']].mean().sort_index().values
    normalized_adj = _build_adjacency_from_centers_utm(centers_utm, k_neighbors)
    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)
    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(city_features, city_targets, seq_len=seq_len)
    df.drop(columns=['_X','_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj

## PHASE 3: The Baseline Benchmark Suite (Null Hypotheses)
**Methodology:** To scientifically prove that physical road topology provides a measurable predictive advantage, a multi-tier control group is established across four architectures of increasing complexity:

* **Logistic Regression (Linear Control):** A `MultiOutputClassifier`-wrapped logistic model that operates on fully flattened feature vectors. Establishes whether the task is solvable by a linear decision boundary at all. If LR matches ST-GNN, deep learning is unnecessary.
* **Random Forest (Ensemble Control):** A `MultiOutputClassifier`-wrapped ensemble of 100 decision trees. Establishes whether non-linear feature interactions are exploitable without temporal sequencing or spatial topology. If RF matches ST-GNN, neither sequence modelling nor graph structure adds value.
* **Spatially Blind LSTM (Temporal Control):** A standard Long Short-Term Memory network that receives the exact same 4D tensors `(Samples, Time_Steps, Nodes, Features)` as the ST-GNN, but with the spatial dimension **flattened**. It is deliberately not provided the Adjacency Matrix, forcing it to treat the city as a disconnected list of coordinates rather than a physical highway network. Establishes whether temporal modelling alone is sufficient.
* **ST-GNN (Proposed Architecture):** The only model that receives the symmetrically normalized Adjacency Matrix, enabling physical shockwave propagation across the road topology.

All four models are evaluated on the same held-out test set using Area Under the Precision-Recall Curve (AUC-PR) as the primary metric, chosen for its robustness under severe class imbalance (< 5% positive class). Neural network results are reported as mean ± standard deviation over 10 independent runs with Wilcoxon signed-rank significance testing.

**Imbalance Handling Note:** Neural networks are governed by a globally weighted Binary Cross-Entropy loss (`pos_weight = total_neg / total_pos` across all nodes). Traditional baselines use Scikit-Learn's `class_weight='balanced'`, which computes weights independently per output node. Both approaches address the minority class penalty but are not mathematically identical — the neural network weight is a global aggregate while the ML baseline weight is node-specific. This difference is formally acknowledged but does not invalidate the comparison, as both methods are standard practice for their respective model families.

In [ ]:
# ==============================================================================
# PHASE 3: THE BASELINE BENCHMARK SUITE
# ==============================================================================
def get_weighted_bce(pos_weight_val):
    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        bce = - (pos_weight_val * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        return tf.reduce_mean(bce)
    return weighted_bce

def build_baseline_lstm(time_steps, n_nodes, n_features, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Temporal_Input")
        flattened_input = Reshape((time_steps, n_nodes * n_features))(input_tensor)
        lstm_out = LSTM(64, return_sequences=False, activation='tanh')(flattened_input)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(lstm_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

class SafeNodeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None):
        self.estimator = estimator
    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        unique_classes = np.unique(y)
        self.classes_ = np.array([0, 1], dtype=int)
        if len(unique_classes) < 2:
            self.is_single_class_ = True
            self.single_class_value_ = int(unique_classes[0])
            self.model_ = None
        else:
            self.is_single_class_ = False
            self.model_ = clone(self.estimator)
            self.model_.fit(X, y)
        return self
    def predict(self, X):
        if self.is_single_class_:
            return np.full(X.shape[0], self.single_class_value_, dtype=int)
        return self.model_.predict(X)
    def predict_proba(self, X):
        if self.is_single_class_:
            probs = np.zeros((X.shape[0], 2), dtype=float)
            probs[:, self.single_class_value_] = 1.0
            return probs
        return self.model_.predict_proba(X)
    @property
    def feature_importances_(self):
        if self.is_single_class_:
            return np.zeros(self.n_features_in_, dtype=float)
        return getattr(self.model_, 'feature_importances_', np.zeros(self.n_features_in_, dtype=float))

def build_traditional_ml_baselines():
    lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_base = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
    lr_model = MultiOutputClassifier(SafeNodeClassifier(lr_base))
    rf_model = MultiOutputClassifier(SafeNodeClassifier(rf_base))
    return lr_model, rf_model

## PHASE 4: Proposed Architecture (ST-GNN Pipeline)
**Methodology:** Standard deep learning models (like the LSTM baseline) fail to capture physical road topology. To solve this and test the Alternative Hypothesis, a Spatio-Temporal Graph Neural Network (ST-GNN) is engineered.

1. **Graph Convolutional Network (GCN):** A custom neural layer performs message passing using the **symmetrically normalized Laplacian** (`D⁻¹/²ÃD⁻¹/²`, where `Ã = A + I`). This normalization prevents high-degree hub nodes from dominating gradient propagation, ensuring stable and balanced signal flow across the highway topology. Two GCN hops are stacked with a residual connection to model multi-hop shockwave propagation.
2. **Gated Recurrent Unit (GRU):** Processes the temporal sequencing (4-hour rolling window) to capture the build-up and dissipation of traffic friction. The architecture deliberately flattens the spatial dimension before the GRU, allowing it to act as a **global temporal forecaster** that captures macro-level city phenomena (e.g., city-wide rush hour) transcending localized intersections. The GCN handles physically constrained local propagation; the GRU handles city-wide temporal dynamics. This hybrid design follows the precedent of Graph WaveNet (Wu et al., 2019) and is intentional rather than incidental.
3. **Shared Evaluation:** This architecture utilizes the exact same custom Weighted Binary Cross-Entropy (BCE) loss function as the baseline to ensure a mathematically identical penalty for False Negatives.

In [ ]:
# ==============================================================================
# PHASE 4: ST-GNN (Proposed Architecture)
# ==============================================================================
class SpatioTemporalGCNLayer(Layer):
    def __init__(self, output_dim, **kwargs):
        super(SpatioTemporalGCNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.kernel = self.add_weight(name='gcn_kernel', shape=(feature_dim, self.output_dim), initializer='glorot_uniform', trainable=True)
        super(SpatioTemporalGCNLayer, self).build(input_shape)
    def call(self, inputs, adj_matrix):
        x_w = tf.matmul(inputs, self.kernel)
        adj_matrix_cast = tf.cast(adj_matrix, dtype=tf.float32)
        spatial_output = tf.einsum('vw,btwf->btvf', adj_matrix_cast, x_w)
        return tf.nn.relu(spatial_output)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)

def build_st_gnn_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="Spatio_Temporal_Input")
        gcn_1 = SpatioTemporalGCNLayer(output_dim=32)(input_tensor, adj_matrix)
        gcn_1 = BatchNormalization()(gcn_1); gcn_1 = Dropout(0.3)(gcn_1)
        gcn_2 = SpatioTemporalGCNLayer(output_dim=32)(gcn_1, adj_matrix)
        gcn_2 = BatchNormalization()(gcn_2); gcn_2 = Dropout(0.3)(gcn_2)
        gcn_combined = layers.Add()([gcn_1, gcn_2])
        reshape_out = Reshape((time_steps, n_nodes * 32))(gcn_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out); gru_out = Dense(32, activation='relu')(gru_out)
        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)
        model = Model(inputs=input_tensor, outputs=output_tensor)
        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=custom_loss, metrics=['accuracy'])
    return model

##PHASE 4B: GRAPHSAGE COMPARATOR (STATIC KNN GRAPH)

In [ ]:
# ==============================================================================
# PHASE 4B: GRAPHSAGE COMPARATOR (STATIC KNN GRAPH)
# ==============================================================================
print("[*] Compiling GraphSAGE Comparator Layer...")

from tensorflow.keras.layers import Layer, Input, BatchNormalization, Dropout, Reshape, GRU, Dense
from tensorflow.keras.models import Model
import tensorflow as tf

class GraphSAGELayer(Layer):
    """
    GraphSAGE-style mean aggregation layer for static macro-region graphs.

    For each node:
        1. Aggregate neighbor features by row-normalized weighted mean
        2. Transform self features
        3. Transform neighbor aggregate
        4. Concatenate both representations
        5. Project back to output_dim and apply ReLU

    Inputs:
        - inputs:     (batch, time, nodes, features)
        - adj_matrix: (nodes, nodes) static normalized adjacency matrix
    """
    def __init__(self, output_dim, **kwargs):
        super(GraphSAGELayer, self).__init__(**kwargs)
        self.output_dim = output_dim

    def build(self, input_shape):
        feature_dim = input_shape[-1]

        # Separate transforms for self and neighbor channels
        self.self_kernel = self.add_weight(
            name='sage_self_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        self.neigh_kernel = self.add_weight(
            name='sage_neigh_kernel',
            shape=(feature_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        # Combine [self_repr || neigh_repr] -> output_dim
        self.combine_kernel = self.add_weight(
            name='sage_combine_kernel',
            shape=(2 * self.output_dim, self.output_dim),
            initializer='glorot_uniform',
            trainable=True
        )

        self.bias = self.add_weight(
            name='sage_bias',
            shape=(self.output_dim,),
            initializer='zeros',
            trainable=True
        )

        super(GraphSAGELayer, self).build(input_shape)

    def call(self, inputs, adj_matrix):
        """
        inputs:     (B, T, V, F)
        adj_matrix: (V, V)
        """
        adj_matrix = tf.cast(adj_matrix, dtype=tf.float32)

        # Remove self-loops for neighborhood-only aggregation
        adj_no_self = tf.linalg.set_diag(
            adj_matrix,
            tf.zeros(tf.shape(adj_matrix)[0], dtype=tf.float32)
        )

        # Row-normalize to create a weighted mean aggregator
        row_sum = tf.reduce_sum(adj_no_self, axis=1, keepdims=True)
        neigh_adj = tf.math.divide_no_nan(adj_no_self, row_sum)

        # Neighborhood mean aggregation: (V,V) x (B,T,V,F) -> (B,T,V,F)
        neigh_mean = tf.einsum('vw,btwf->btvf', neigh_adj, inputs)

        # Linear transforms
        self_repr = tf.matmul(inputs, self.self_kernel)         # (B,T,V,D)
        neigh_repr = tf.matmul(neigh_mean, self.neigh_kernel)   # (B,T,V,D)

        # Concatenate and combine
        combined = tf.concat([self_repr, neigh_repr], axis=-1)  # (B,T,V,2D)
        output = tf.matmul(combined, self.combine_kernel) + self.bias

        return tf.nn.relu(output)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.output_dim)


def build_graphsage_model(time_steps, n_nodes, n_features, adj_matrix, pos_weight):
    """
    GraphSAGE comparator with the SAME temporal head pattern as your ST-GNN:
        GraphSAGE -> BN -> Dropout
        GraphSAGE -> BN -> Dropout
        Residual Add
        Reshape
        GRU
        Dense
        Sigmoid Output

    This keeps the comparison fair:
        - same graph
        - same temporal sequence modeling
        - same weighted BCE loss
        - only the graph propagation rule changes
    """
    with strategy.scope():
        input_tensor = Input(shape=(time_steps, n_nodes, n_features), name="GraphSAGE_Input")

        # GraphSAGE Block 1
        sage_1 = GraphSAGELayer(output_dim=32)(input_tensor, adj_matrix)
        sage_1 = BatchNormalization()(sage_1)
        sage_1 = Dropout(0.3)(sage_1)

        # GraphSAGE Block 2
        sage_2 = GraphSAGELayer(output_dim=32)(sage_1, adj_matrix)
        sage_2 = BatchNormalization()(sage_2)
        sage_2 = Dropout(0.3)(sage_2)

        # Residual connection
        sage_combined = tf.keras.layers.Add()([sage_1, sage_2])

        # Same temporal head as ST-GNN
        reshape_out = Reshape((time_steps, n_nodes * 32))(sage_combined)
        gru_out = GRU(64, return_sequences=False, activation='tanh')(reshape_out)
        gru_out = Dropout(0.3)(gru_out)
        gru_out = Dense(32, activation='relu')(gru_out)

        output_tensor = Dense(n_nodes, activation='sigmoid', name="Risk_Prediction")(gru_out)

        model = Model(inputs=input_tensor, outputs=output_tensor)

        custom_loss = get_weighted_bce(pos_weight)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

        model.compile(
            optimizer=optimizer,
            loss=custom_loss,
            metrics=['accuracy']
        )

    return model

print("[SUCCESS] Phase 4B Complete: GraphSAGE Comparator Loaded.")

## PHASE 5: Experimental Setup & Sensitivity Analysis ("Challenger Arena")
**Methodology:** A full combinatorial Grid Search is executed across the defined spatial boundaries. Because traffic datasets feature extreme class imbalance (STPCs are < 5% of all data), standard Accuracy is a scientifically invalid metric.

The primary selection metric for the Champion topology is the **Area Under the Precision-Recall Curve (AUC-PR)**, computed via `average_precision_score`. Unlike F1-Score at a single threshold, AUC-PR evaluates model performance across every possible operating threshold, making it mathematically immune to threshold bias.

A secondary **Dynamic Thresholding Protocol** is also computed for operational reference:
1. The model outputs continuous probability distributions (0.0 to 1.0).
2. The Precision-Recall curve is calculated.
3. The operational threshold is dynamically shifted until **Recall is locked at ≥ 70%** (ensuring 7 out of 10 STPCs are detected to satisfy public safety requirements).
4. The resulting Precision and F1 at that threshold are logged as secondary indicators.

The **Champion topology is selected exclusively by highest Validation AUC-PR** and is never evaluated on the test set during this phase.

In [ ]:
# ==============================================================================
# PHASE 5: EXPERIMENT C2 - NATIVE RURAL TOPOLOGY DISCOVERY (The Challenger Arena)
# ==============================================================================
print("\n[*] Initializing Phase 5: The Challenger Arena (Native Rural Search)...")

experiment_results = []
RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_Arena_Results.csv"

total_experiments = len(SEARCH_NODES) * len(SEARCH_RADII) * len(SEARCH_K)
current_exp = 1

for n_nodes in SEARCH_NODES:
    for radius_mi in SEARCH_RADII:
        for k_neighbors in SEARCH_K:
            print(f"\n==================================================================")
            print(f"[*] EXPERIMENT {current_exp}/{total_experiments}: {n_nodes} Nodes | {radius_mi}mi | K={k_neighbors}")
            print(f"==================================================================")

            # Explicitly initialize variables for safe cleanup
            model = history = y_pred_probs_val = early_stop = None
            X_tr = X_val = X_te = y_tr = y_val = y_te = adj = None

            try:
                X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
                    df_raw=df_target, n_nodes=n_nodes, radius_mi=radius_mi, k_neighbors=k_neighbors, seq_len=SEQ_LENGTH
                )

                total_pos = np.sum(y_tr)
                total_neg = (y_tr.size) - total_pos

                if total_pos < 10:
                    print(f"[WARNING] Topology yielded only {total_pos} STPCs. Statistically unstable. Skipping.")
                    current_exp += 1
                    if X_tr is not None: del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
                    gc.collect()
                    continue

                pos_weight_val = min(float(total_neg / total_pos), 1000.0)
                print(f"[*] Imbalance Ratio -> Positives: {total_pos} | Negatives: {total_neg}")

                model = build_st_gnn_model(SEQ_LENGTH, n_nodes, X_tr.shape[-1], adj, pos_weight_val)

                print("[*] Commencing Training...")
                early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                history = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64, callbacks=[early_stop], verbose=0
                )
                print(f"[+] Training Converged at Epoch {len(history.history['loss'])}")

                y_pred_probs_val = model.predict(X_val, batch_size=64, verbose=0)
                y_val_flat = y_val.flatten()
                y_pred_flat = y_pred_probs_val.flatten()

                precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_pred_flat)
                auc_pr = average_precision_score(y_val_flat, y_pred_flat)

                valid_idx = np.where(recalls >= 0.70)[0]
                if len(valid_idx) > 0:
                    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
                    optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
                    final_precision = precisions[best_idx]
                    final_recall = recalls[best_idx]
                    final_f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0
                else:
                    optimal_threshold, final_precision, final_recall, final_f1 = 0.5, 0.0, 0.0, 0.0

                print(f"[+] Validation Yield -> AUC-PR: {auc_pr:.4f} | F1: {final_f1:.4f} | Thresh: {optimal_threshold:.3f}")

                experiment_results.append({
                    'Domain': TARGET_CITY,
                    'Nodes': n_nodes,
                    'Radius_mi': radius_mi,
                    'K_Neighbors': k_neighbors,
                    'Pos_Weight': round(pos_weight_val, 2),
                    'Val_AUC_PR': round(float(auc_pr), 4),
                    'Optimal_Threshold': round(float(optimal_threshold), 4),
                    'Precision_at_70_Recall': round(float(final_precision), 4),
                    'Actual_Recall': round(float(final_recall), 4),
                    'Val_F1_Score': round(float(final_f1), 4)
                })

                pd.DataFrame(experiment_results).to_csv(RESULTS_PATH, index=False)

            except Exception as e:
                print(f"[ERROR] Experiment failed: {str(e)}")

            if early_stop is not None: del early_stop
            if model is not None: del model
            if history is not None: del history
            if y_pred_probs_val is not None: del y_pred_probs_val
            if X_tr is not None: del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()
            current_exp += 1

print("\n[SUCCESS] PHASE 5 COMPLETE. Rural native topology discovery finished.")

## PHASE 5B: MAUP Sensitivity Analysis
**Methodology:** The Modifiable Areal Unit Problem (MAUP) states that results derived from spatially aggregated data are sensitive to the choice of areal unit — in this case, the number of K-Means macro-nodes. A result that only holds for one node count is not robust.

This section loads the Phase 5 arena results and produces a sensitivity plot comparing AUC-PR across both node configurations (100 and 500) for each radius. A stable model should show consistent relative rankings regardless of spatial granularity.

In [ ]:
# ==============================================================================
# PHASE 5B: MAUP SENSITIVITY ANALYSIS (Scale)
# ==============================================================================
print("\n[*] Running MAUP Sensitivity Analysis (Experiment C2)...")

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}. Run Phase 5 first.")

arena_results_df = pd.read_csv(RESULTS_PATH)
if arena_results_df.empty:
    raise ValueError("[ERROR] Arena results file is empty. All Phase 5 configurations were skipped due to extreme sparsity.")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'MAUP Sensitivity Analysis — Effect of Spatial Granularity on AUC-PR\nDomain: {TARGET_CITY.upper()} (Experiment C2)', fontsize=13, fontweight='bold')

for ax_idx, radius in enumerate(SEARCH_RADII):
    ax = axes[ax_idx]
    subset = arena_results_df[arena_results_df['Radius_mi'] == radius].copy()
    for n_nodes_val, grp in subset.groupby('Nodes'):
        grp_sorted = grp.sort_values('K_Neighbors')
        ax.plot(grp_sorted['K_Neighbors'], grp_sorted['Val_AUC_PR'], marker='o', linewidth=2, color=NODE_COLORS.get(n_nodes_val, 'grey'), label=f'{n_nodes_val} Nodes')
    ax.set_title(f'Radius = {radius}mi', fontweight='bold')
    ax.set_xlabel('K Neighbors', fontweight='bold'); ax.set_ylabel('Validation AUC-PR', fontweight='bold')
    ax.set_xticks(SEARCH_K); ax.legend(title='Node Count'); ax.grid(True, alpha=0.3)

plt.tight_layout()
MAUP_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_MAUP_Sensitivity.png"
plt.savefig(MAUP_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] MAUP Sensitivity artifact saved to: {MAUP_PATH}")
plt.show()

##Phase 5C: The MAUP Zonation Protocol

In [ ]:
# ==============================================================================
# PHASE 5C: MAUP ZONATION SENSITIVITY (Champion-Based)
# ==============================================================================
print(f"\n[*] Initiating Phase 5C: MAUP Zonation Stability Protocol for {TARGET_CITY.upper()}...")

champion_row = arena_results_df.sort_values('Val_AUC_PR', ascending=False).iloc[0]
FIXED_NODES = int(champion_row['Nodes'])
FIXED_RADIUS = float(champion_row['Radius_mi'])
FIXED_K = int(champion_row['K_Neighbors'])

print(f"[*] Zonation test anchored at CHAMPION topology: N={FIXED_NODES}, Radius={FIXED_RADIUS}mi, K={FIXED_K}")

df_target['Start_Time'] = pd.to_datetime(df_target['Start_Time'])
df_target = df_target.sort_values('Start_Time').reset_index(drop=True)

transformer, epsg_code = _dynamic_utm_transformer(df_target)
utm_coords = _project_to_utm(df_target, transformer)

start_min = df_target['Start_Time'].min().floor('h')
end_max   = df_target['Start_Time'].max().floor('h')
full_time = pd.date_range(start=start_min, end=end_max, freq='h')

total_windows = max(0, len(full_time) - SEQ_LENGTH)
train_windows = int(total_windows * 0.70)
cutoff_time = full_time[train_windows] if train_windows < len(full_time) else full_time[-1]
train_mask = df_target['Start_Time'] < cutoff_time
utm_train = utm_coords[train_mask.values]

ZONATION_SEEDS = 20
zonation_auc_scores = []
test_prevalence = None

print(f"[*] Executing {ZONATION_SEEDS} K-Means Spatial Partitions (Train-Era Fitted, UTM={epsg_code})...")

for s in range(ZONATION_SEEDS):
    try:
        random.seed(42 + s)
        np.random.seed(42 + s)
        tf.random.set_seed(42 + s)

        kmeans = KMeans(n_clusters=FIXED_NODES, random_state=s, n_init=10)
        kmeans.fit(utm_train)
        df_target['Region_ID'] = kmeans.predict(utm_coords)

        X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
            df_target, n_nodes=FIXED_NODES, radius_mi=FIXED_RADIUS, k_neighbors=FIXED_K, seq_len=SEQ_LENGTH
        )

        if test_prevalence is None: test_prevalence = float(y_te.mean())

        pos = y_tr.sum()
        if pos < 10:
            print(f"    [WARNING] Seed {s} produced only {int(pos)} positives. Skipping unstable zonation run.")
            del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session(); gc.collect()
            continue

        neg = (y_tr.size - pos)
        pos_weight_val = min(float(neg / max(pos, 1)), 1000.0)

        early_stop_zonation = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
        model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64, callbacks=[early_stop_zonation], verbose=0)

        y_pred = model.predict(X_te, verbose=0).flatten()
        auc_pr = average_precision_score(y_te.flatten(), y_pred)
        zonation_auc_scores.append(float(auc_pr))

        del model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, y_pred, early_stop_zonation
        tf.keras.backend.clear_session(); gc.collect()

    except Exception as e:
        print(f"    [WARNING] Seed {s} failed: {e}")
        tf.keras.backend.clear_session(); gc.collect()
        continue

# SCIENTIFIC FIX 4B: Guard if all K-Means zonations failed
if len(zonation_auc_scores) == 0:
    raise RuntimeError("[CRITICAL] All K-Means zonation runs were skipped or failed. Cannot compute zonation robustness.")

h3_encode = getattr(h3, "latlng_to_cell", getattr(h3, "geo_to_h3", None))
h3_decode = getattr(h3, "cell_to_latlng", getattr(h3, "h3_to_geo", None))

# SCIENTIFIC FIX 4A: Check if H3 API is viable
if h3_encode is None or h3_decode is None:
    raise ImportError("[CRITICAL FAILURE] Unrecognized H3 library version. Encode/decode methods missing.")

print("\n[*] Executing Alternate Zoning Family: H3 Hexagonal Super-Regions (Train-Era Calibrated)...")

def choose_h3_resolution(df_like, target=100, tol=5, rmin=5, rmax=10):
    best = None
    for res in range(rmin, rmax + 1):
        ids = df_like.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], res), axis=1)
        n = ids.nunique()
        if abs(n - target) <= tol: return res, ids
        if best is None or abs(n - target) < abs(best[0] - target): best = (n, res, ids)
    return best[1], best[2]

try:
    h3_res, _ = choose_h3_resolution(df_target[train_mask], target=FIXED_NODES)
    h3_ids_full = df_target.apply(lambda r: h3_encode(r['Start_Lat'], r['Start_Lng'], h3_res), axis=1)
    uniq_hex = pd.Series(h3_ids_full.astype(str).values).unique()
    n_hex_nodes = len(uniq_hex)

    if n_hex_nodes < FIXED_NODES:
        raise ValueError(f"H3 produced only {n_hex_nodes} hexes; cannot aggregate into {FIXED_NODES} super-regions.")

    hex_latlon = np.array([h3_decode(h) for h in uniq_hex])
    hex_x, hex_y = transformer.transform(hex_latlon[:, 1], hex_latlon[:, 0])

    spatial_connectivity = kneighbors_graph(np.c_[hex_x, hex_y], n_neighbors=min(6, n_hex_nodes - 1), include_self=False)
    agg_cluster = AgglomerativeClustering(n_clusters=FIXED_NODES, connectivity=spatial_connectivity, linkage='ward')
    h_labels = agg_cluster.fit_predict(np.c_[hex_x, hex_y])
    hex_to_super = {h: int(l) for h, l in zip(uniq_hex, h_labels)}
    df_target['Region_ID'] = pd.Series(h3_ids_full.astype(str).values).map(hex_to_super).values

    X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_preclustered(
        df_target, n_nodes=FIXED_NODES, radius_mi=FIXED_RADIUS, k_neighbors=FIXED_K, seq_len=SEQ_LENGTH
    )

    pos = y_tr.sum()
    if pos < 10:
        raise ValueError(f"[ERROR] H3 zonation produced only {int(pos)} positives. Statistically unstable for training.")

    neg = (y_tr.size - pos)
    pos_weight_val = min(float(neg / max(pos, 1)), 1000.0)

    early_stop_h3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    h3_model = build_st_gnn_model(SEQ_LENGTH, FIXED_NODES, X_tr.shape[-1], adj, pos_weight_val)
    h3_model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=15, batch_size=64, callbacks=[early_stop_h3], verbose=0)

    h3_pred = h3_model.predict(X_te, verbose=0).flatten()
    h3_auc_pr = average_precision_score(y_te.flatten(), h3_pred)

    del h3_model, X_tr, X_val, X_te, y_tr, y_val, y_te, adj, h3_pred, early_stop_h3
    tf.keras.backend.clear_session(); gc.collect()

    from scipy.stats import percentileofscore
    mean_auc = float(np.mean(zonation_auc_scores))
    sd_auc = float(np.std(zonation_auc_scores))
    cv_percent = (sd_auc / mean_auc) * 100 if mean_auc > 0 else 0.0
    q_lo, q_hi = np.quantile(zonation_auc_scores, [0.025, 0.975])
    h3_pct = percentileofscore(zonation_auc_scores, h3_auc_pr)

    print("\n========================================================")
    print("[*] ZONATION MAUP RESULTS")
    print("========================================================")
    print(f"K-Means PR-AUC (n={len(zonation_auc_scores)}) : {mean_auc:.4f} ± {sd_auc:.4f}")
    print(f"H3 Super-Region PR-AUC       : {h3_auc_pr:.4f}")

    ZONATION_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_Zonation_Results.csv"
    zonation_df = pd.DataFrame([{
        'City': TARGET_CITY, 'Champion_Nodes': FIXED_NODES, 'Champion_Radius_mi': FIXED_RADIUS, 'Champion_K': FIXED_K,
        'KMeans_Zonation_Runs': len(zonation_auc_scores), 'KMeans_AUC_Mean': round(mean_auc, 4), 'KMeans_AUC_Std': round(sd_auc, 4),
        'KMeans_AUC_Q2.5': round(float(q_lo), 4), 'KMeans_AUC_Q97.5': round(float(q_hi), 4),
        'KMeans_CV_Percent': round(float(cv_percent), 2), 'H3_AUC_PR': round(float(h3_auc_pr), 4), 'H3_Percentile_Within_KMeans': round(float(h3_pct), 2)
    }])
    zonation_df.to_csv(ZONATION_RESULTS_PATH, index=False)

except Exception as e:
    print(f"[WARNING] H3 Evaluation skipped or failed: {e}")

## PHASE 6: The Final Proof (Statistical Hypothesis Testing)
**Methodology:** This phase executes the formal statistical test of the Alternative Hypothesis against all four Null Hypothesis baselines.

Using the empirically discovered Champion topology from Phase 5 (selected by highest Validation AUC-PR), the following sequential protocol is executed:

1. **Traditional ML Baselines (LR & RF):** Logistic Regression and Random Forest are trained on flattened 2D feature matrices and evaluated on the held-out test set. Both are deterministic (fixed `random_state=42`) and run once.
2. **Statistical Variance Loop (n=10):** Both the Spatially Blind LSTM and the Champion ST-GNN are trained and evaluated independently across 10 runs, each with a different random seed (`42` through `51`). This produces a distribution of AUC-PR scores rather than a single point estimate.
3. **Frozen Threshold Protocol:** For each neural network run, the optimal decision threshold is derived exclusively from the validation set at ≥70% recall, then applied to the test set without modification. The test set is never used for threshold selection.
4. **Wilcoxon Signed-Rank Test:** A one-sided Wilcoxon test is applied to the 10 paired (ST-GNN, LSTM) AUC-PR scores to determine if the ST-GNN improvement is statistically significant at α=0.05. With n=10 paired observations, the test is mathematically capable of achieving p < 0.05.
5. **Honest Conclusion:** The hypothesis conclusion is determined solely by the p-value and direction of effect — it is never hardcoded.

In [ ]:
# ==============================================================================
# PHASE 6: THE FINAL PROOF (Iowa Native Champion + GraphSAGE Comparator)
# ==============================================================================
print(f"\n[*] Initializing Phase 6: The Final Proof for {TARGET_CITY.upper()}...")

import scipy.stats as stats
from scipy.stats import wilcoxon
import random
import gc
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# --------------------------------------------------------
# 1. DYNAMICALLY EXTRACT CHAMPION (Sorted by Val_AUC_PR)
# --------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(f"[ERROR] Arena results not found at {RESULTS_PATH}")

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]
CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# --------------------------------------------------------
# 2. ISOLATE THE FINAL TENSORS
# --------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target,
    n_nodes=CHAMP_NODES,
    radius_mi=CHAMP_RADIUS,
    k_neighbors=CHAMP_K,
    seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError(
        "[ERROR] Champion topology produced zero STPC labels in training set. "
        "Re-run Phase 5 — the arena may have selected a degenerate configuration."
    )

# Safety cap for sparse domains
pos_weight_val = min(float(((y_tr.size) - total_pos) / max(total_pos, 1)), 1000.0)
y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# --------------------------------------------------------
# 3. TRADITIONAL ML BASELINES (Logistic Regression & Random Forest)
# --------------------------------------------------------
print("\n[*] Evaluating Traditional ML Baselines (LR & RF)...")
X_tr_ml = X_tr.reshape(X_tr.shape[0], -1)
X_te_ml = X_te.reshape(X_te.shape[0], -1)

lr_model, rf_model = build_traditional_ml_baselines()

def evaluate_ml_baseline(model, name):
    print(f"[*] Training {name}...")
    model.fit(X_tr_ml, y_tr)

    preds = model.predict_proba(X_te_ml)
    y_pred_probs = np.zeros(y_te.shape)

    for i in range(y_te.shape[1]):
        if preds[i].shape[1] >= 2:
            y_pred_probs[:, i] = preds[i][:, 1]
        else:
            y_pred_probs[:, i] = 0.0

    y_pred_flat = y_pred_probs.flatten()
    auc_pr = average_precision_score(y_te_flat, y_pred_flat)

    val_preds = model.predict_proba(X_val.reshape(X_val.shape[0], -1))
    y_val_probs = np.zeros(y_val.shape)

    for i in range(y_val.shape[1]):
        if val_preds[i].shape[1] >= 2:
            y_val_probs[:, i] = val_preds[i][:, 1]
        else:
            y_val_probs[:, i] = 0.0

    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs.flatten())
    valid_idx = np.where(recalls >= 0.70)[0]

    # SCIENTIFIC FIX: Safe threshold index guard
    if len(valid_idx) > 0:
        best_idx = valid_idx[np.argmax(precisions[valid_idx])]
        opt_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    else:
        opt_thresh = 0.5

    y_te_binary = (y_pred_flat >= opt_thresh).astype(int)
    f1 = f1_score(y_te_flat, y_te_binary, zero_division=0)

    print(f"[+] {name} Yield -> Test AUC-PR: {auc_pr:.4f} | Test F1: {f1:.4f}")
    return auc_pr, f1

lr_auc, lr_f1 = evaluate_ml_baseline(lr_model, "Logistic Regression")
rf_auc, rf_f1 = evaluate_ml_baseline(rf_model, "Random Forest")

del X_tr_ml, X_te_ml, lr_model, rf_model
gc.collect()

# --------------------------------------------------------
# 4. DEEP LEARNING STATISTICAL VARIANCE LOOP (n=10)
# --------------------------------------------------------
N_RUNS = 10
print(f"\n[*] Commencing Statistical Variance Testing (n={N_RUNS} Runs) for Neural Networks...")

lstm_auc_scores, lstm_f1_scores = [], []
stgnn_auc_scores, stgnn_f1_scores = [], []
graphsage_auc_scores, graphsage_f1_scores = [], []

def evaluate_nn(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1

def get_frozen_threshold(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]

    # SCIENTIFIC FIX: Safe threshold index guard
    if len(valid_idx) == 0:
        return 0.5

    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5

for run in range(N_RUNS):
    print(f"\n--- [RUN {run+1}/{N_RUNS}] ---")

    # SCIENTIFIC FIX: Triple-seed reset
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    # -------------------------------
    # BASELINE LSTM
    # -------------------------------
    lstm_model = build_baseline_lstm(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], pos_weight_val)
    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    lstm_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_lstm],
        verbose=0
    )

    thresh_lstm = get_frozen_threshold(lstm_model, X_val, y_val_flat)
    auc_lstm, f1_lstm = evaluate_nn(lstm_model, X_te, y_te_flat, thresh_lstm)
    lstm_auc_scores.append(float(auc_lstm))
    lstm_f1_scores.append(float(f1_lstm))
    print(f"[*] LSTM_Blind           -> AUC-PR: {auc_lstm:.4f} | F1: {f1_lstm:.4f}")

    del lstm_model, early_stop_lstm
    tf.keras.backend.clear_session()
    gc.collect()

    # -------------------------------
    # ST-GNN (custom model)
    # -------------------------------
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    stgnn_model = build_st_gnn_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    stgnn_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_stgnn],
        verbose=0
    )

    thresh_stgnn = get_frozen_threshold(stgnn_model, X_val, y_val_flat)
    auc_stgnn, f1_stgnn = evaluate_nn(stgnn_model, X_te, y_te_flat, thresh_stgnn)
    stgnn_auc_scores.append(float(auc_stgnn))
    stgnn_f1_scores.append(float(f1_stgnn))
    print(f"[*] STGNN_StaticKNN      -> AUC-PR: {auc_stgnn:.4f} | F1: {f1_stgnn:.4f}")

    del stgnn_model, early_stop_stgnn
    tf.keras.backend.clear_session()
    gc.collect()

    # -------------------------------
    # GraphSAGE comparator
    # -------------------------------
    random.seed(42 + run)
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)

    graphsage_model = build_graphsage_model(SEQ_LENGTH, CHAMP_NODES, X_tr.shape[-1], adj, pos_weight_val)
    early_stop_graphsage = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    graphsage_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_graphsage],
        verbose=0
    )

    thresh_graphsage = get_frozen_threshold(graphsage_model, X_val, y_val_flat)
    auc_graphsage, f1_graphsage = evaluate_nn(graphsage_model, X_te, y_te_flat, thresh_graphsage)
    graphsage_auc_scores.append(float(auc_graphsage))
    graphsage_f1_scores.append(float(f1_graphsage))
    print(f"[*] GraphSAGE_StaticKNN  -> AUC-PR: {auc_graphsage:.4f} | F1: {f1_graphsage:.4f}")

    del graphsage_model, early_stop_graphsage
    tf.keras.backend.clear_session()
    gc.collect()

# --------------------------------------------------------
# 5. SUMMARY + HYPOTHESIS TESTING
# --------------------------------------------------------
def safe_t_interval(scores):
    scores = np.array(scores, dtype=float)
    if len(scores) < 2 or np.std(scores) == 0:
        m = float(np.mean(scores))
        return (m, m)
    return stats.t.interval(0.95, df=len(scores)-1, loc=np.mean(scores), scale=stats.sem(scores))

lstm_auc_mean, lstm_auc_std = float(np.mean(lstm_auc_scores)), float(np.std(lstm_auc_scores))
stgnn_auc_mean, stgnn_auc_std = float(np.mean(stgnn_auc_scores)), float(np.std(stgnn_auc_scores))
graphsage_auc_mean, graphsage_auc_std = float(np.mean(graphsage_auc_scores)), float(np.std(graphsage_auc_scores))

lstm_ci = safe_t_interval(lstm_auc_scores)
stgnn_ci = safe_t_interval(stgnn_auc_scores)
graphsage_ci = safe_t_interval(graphsage_auc_scores)

print("\n==================================================================")
print(f"[*] FINAL THESIS BENCHMARK: {TARGET_CITY.upper()} (Graph Comparator Added)")
print("==================================================================")
print(f"Champion Topology: {CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")
print("--- Baselines ---")
print(f"Logistic Regression AUC-PR : {lr_auc:.4f}")
print(f"Random Forest AUC-PR       : {rf_auc:.4f}")
print(f"--- Deep Learning (Mean ± Std over {N_RUNS} runs) ---")
print(f"LSTM_Blind AUC-PR          : {lstm_auc_mean:.4f} ± {lstm_auc_std:.4f}  95% CI [{lstm_ci[0]:.4f}, {lstm_ci[1]:.4f}]")
print(f"STGNN_StaticKNN AUC-PR     : {stgnn_auc_mean:.4f} ± {stgnn_auc_std:.4f}  95% CI [{stgnn_ci[0]:.4f}, {stgnn_ci[1]:.4f}]")
print(f"GraphSAGE_StaticKNN AUC-PR : {graphsage_auc_mean:.4f} ± {graphsage_auc_std:.4f}  95% CI [{graphsage_ci[0]:.4f}, {graphsage_ci[1]:.4f}]")
print("==================================================================")

p_value = None
try:
    stat, p_value = wilcoxon(stgnn_auc_scores, lstm_auc_scores, alternative='greater')
    print(f"[*] Statistical Test (Wilcoxon, STGNN vs LSTM): p-value = {p_value:.4f}")

    if p_value < 0.05 and stgnn_auc_mean > lstm_auc_mean:
        print("[CONCLUSION] Strong empirical evidence supports the Alternative Hypothesis against the temporal-only baseline.")
    else:
        print("[CONCLUSION] The Alternative Hypothesis cannot be definitively confirmed against the temporal-only baseline at α=0.05.")

except Exception as e:
    print(f"[WARNING] Wilcoxon test failed: {e}")
    print("[INFO] This typically occurs when all paired score differences are zero.")

print(f"[*] Comparator Note: GraphSAGE mean AUC-PR = {graphsage_auc_mean:.4f} vs ST-GNN mean AUC-PR = {stgnn_auc_mean:.4f}")

# --------------------------------------------------------
# 6. SCIENTIFIC LOGGING
# --------------------------------------------------------
PROOF_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentC2_Final_Proof_with_GraphSAGE.csv"

proof_df = pd.DataFrame([{
    'City': TARGET_CITY,
    'Champion_Nodes': CHAMP_NODES,
    'Champion_Radius_mi': CHAMP_RADIUS,
    'Champion_K': CHAMP_K,
    'LR_AUC': round(float(lr_auc), 4),
    'RF_AUC': round(float(rf_auc), 4),
    'LSTM_AUC_Mean': round(lstm_auc_mean, 4),
    'LSTM_AUC_Std': round(lstm_auc_std, 4),
    'STGNN_AUC_Mean': round(stgnn_auc_mean, 4),
    'STGNN_AUC_Std': round(stgnn_auc_std, 4),
    'GraphSAGE_AUC_Mean': round(graphsage_auc_mean, 4),
    'GraphSAGE_AUC_Std': round(graphsage_auc_std, 4),
    'LSTM_CI_Lower': round(float(lstm_ci[0]), 4),
    'LSTM_CI_Upper': round(float(lstm_ci[1]), 4),
    'STGNN_CI_Lower': round(float(stgnn_ci[0]), 4),
    'STGNN_CI_Upper': round(float(stgnn_ci[1]), 4),
    'GraphSAGE_CI_Lower': round(float(graphsage_ci[0]), 4),
    'GraphSAGE_CI_Upper': round(float(graphsage_ci[1]), 4),
    'Wilcoxon_p_value_STGNN_vs_LSTM': round(float(p_value), 4) if p_value is not None else None
}])

proof_df.to_csv(PROOF_RESULTS_PATH, index=False)
print(f"\n[*] Scientific conclusion successfully logged to: {PROOF_RESULTS_PATH}")

# --------------------------------------------------------
# 7. UPDATED VISUALIZATION ARTIFACT
# --------------------------------------------------------
print(f"[*] Generating Scientific Artifact for {TARGET_CITY.upper()} (with GraphSAGE)...")

model_names = [
    'Logistic\nRegression',
    'Random\nForest',
    'Spatially Blind\nLSTM',
    'ST-GNN\n(Static KNN)',
    'GraphSAGE\n(Static KNN)'
]

auc_scores = [
    lr_auc,
    rf_auc,
    lstm_auc_mean,
    stgnn_auc_mean,
    graphsage_auc_mean
]

std_devs = [
    0.0,
    0.0,
    lstm_auc_std,
    stgnn_auc_std,
    graphsage_auc_std
]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#2c7bb6', '#1a9641']
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(
    f'Spatio-Temporal vs. Traditional Baselines (AUC-PR)\nDomain: {TARGET_CITY.upper()} | Native Champion: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
    fontsize=13, fontweight='bold', pad=15
)
plt.ylabel('AUC-PR (Higher is Better)', fontsize=11, fontweight='bold')

max_y = max(auc_scores)
plt.ylim(0, max(max_y * 1.35, 0.05))

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(
        f'AUC: {height:.4f}{std_text}',
        (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
        ha='center', va='bottom',
        fontsize=10, color='black',
        xytext=(0, 8),
        textcoords='offset points',
        fontweight='bold'
    )

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentC2_Phase6_AUC_Comparison_with_GraphSAGE.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# THESIS ARTIFACT: MULTI-BASELINE VISUALIZATION (EXPERIMENT C2)
# ==============================================================================
print(f"\n[*] Generating Scientific Artifact for {TARGET_CITY.upper()}...")

model_names = ['Logistic\nRegression', 'Random\nForest', 'Spatially Blind\nLSTM', 'Native Rural\nST-GNN']
auc_scores = [lr_auc, rf_auc, lstm_auc_mean, stgnn_auc_mean]
std_devs = [0.0, 0.0, lstm_auc_std, stgnn_auc_std]

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#d3d3d3', '#a9a9a9', '#8c8c8c', '#1a9641']  # Rural Green for Champion
bars = ax.bar(model_names, auc_scores, yerr=std_devs, capsize=5, color=colors, edgecolor='black', alpha=0.9)

plt.title(
    f'Rural Topology Discovery (AUC-PR)\nDomain: {TARGET_CITY.upper()} | Native Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}',
    fontsize=13, fontweight='bold', pad=15
)
plt.ylabel('AUC-PR (Higher is Better)', fontsize=11, fontweight='bold')

max_y = max(auc_scores)
upper = max(max_y * 1.35, 0.05)
plt.ylim(0, upper)

for i, bar in enumerate(bars):
    height = bar.get_height()
    std_text = f" ± {std_devs[i]:.4f}" if std_devs[i] > 0 else ""
    ax.annotate(
        f'AUC: {height:.4f}{std_text}',
        (bar.get_x() + bar.get_width() / 2., height + std_devs[i]),
        ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 8),
        textcoords='offset points', fontweight='bold'
    )

plt.tight_layout()
VIS_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_Phase6_AUC_Comparison.png"
plt.savefig(VIS_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-Resolution Artifact saved to: {VIS_PATH}")
plt.show()


## PHASE 6B: Feature Importance & SHAP-Equivalent Analysis
**Methodology:** To determine which input features actually drive STPC predictions, the Champion Random Forest model's built-in feature importances are extracted. Since the RF operates on flattened `(Time_Steps × Nodes × Features)` vectors, importances are aggregated back across the 4 original feature dimensions to produce a per-feature contribution score.

This analysis answers a critical scientific question: does the model rely on **accident density** (Total_Accidents), **incident severity** (Mean_Severity), **road clearance time** (Mean_Distance), or **weather friction** (Mean_Humidity) to identify STPCs? The result directly informs the theoretical interpretation of the shockwave propagation mechanism.

In [ ]:
# ==============================================================================
# PHASE 6B: AGGREGATED FEATURE IMPORTANCE ANALYSIS (Random Forest Proxy)
# ==============================================================================
print(f"\n[*] Extracting Aggregated Feature Importances from Champion Random Forest...")

_, rf_importance_model = build_traditional_ml_baselines()
X_tr_ml_fi = X_tr.reshape(X_tr.shape[0], -1)
rf_importance_model.fit(X_tr_ml_fi, y_tr)

all_importances = np.array([est.feature_importances_ for est in rf_importance_model.estimators_])
mean_importances = all_importances.mean(axis=0)

feature_contributions = np.zeros(len(FEATURES_LIST))
for feat_idx in range(len(FEATURES_LIST)):
    feature_contributions[feat_idx] = mean_importances[feat_idx::len(FEATURES_LIST)].sum()
if feature_contributions.sum() > 0: feature_contributions /= feature_contributions.sum()

fig, ax = plt.subplots(figsize=(8, 5))
bars_fi = ax.barh(FEATURES_LIST, feature_contributions, color=['#2c7bb6', '#d7191c', '#fdae61', '#1a9641'], edgecolor='black', alpha=0.88)
ax.set_xlabel('Aggregated Feature Importance (Normalised)', fontsize=11, fontweight='bold')
ax.set_title(f'RF Aggregated Feature Importance — STPC Prediction Drivers\nDomain: {TARGET_CITY.upper()} | Champion Topology: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi', fontsize=12, fontweight='bold')
for bar, val in zip(bars_fi, feature_contributions):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
FI_PATH = f"{BASE_DIR}/{TARGET_CITY}_Experiment_C2_Feature_Importance.png"
plt.savefig(FI_PATH, dpi=300, bbox_inches='tight'); print(f"[SUCCESS] Feature Importance artifact saved to: {FI_PATH}")
plt.show()

del rf_importance_model, X_tr_ml_fi, all_importances; gc.collect()

##PHASE 6C: CORE ABLATION SUITE

In [ ]:
# ==============================================================================
# PHASE 6C: CORE ABLATION SUITE (IOWA NATIVE CHAMPION)
# ==============================================================================
print(f"\n[*] Initializing Phase 6C: Core Ablation Suite for IOWA (Experiment C2)...")

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import gc
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# Locked Iowa native champion from Experiment C2 Phase 6
ABLATION_TARGET_CITY = "Iowa"
ABLATION_EXPERIMENT_ID = "ExperimentC2_RuralNative"
ABLATION_PHASE_ID = "Phase6C_Ablation"

ABLATION_CHAMP_NODES = 50
ABLATION_CHAMP_RADIUS = 15.0
ABLATION_CHAMP_K = 3

# Full feature set (do not modify the original global FEATURES_LIST)
ABLATION_FULL_FEATURE_LIST = [
    'Total_Accidents',
    'Mean_Severity',
    'Mean_Distance',
    'Mean_Humidity'
]

# Number of repeated neural runs per ablation
ABLATION_RUNS = 5

# Additive-only output path
ABLATION_RESULTS_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_Results.csv"

ABLATION_CONFIGS = {
    "Full": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Humidity": {
        "feature_list": ['Total_Accidents', 'Mean_Severity', 'Mean_Distance'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Severity": {
        "feature_list": ['Total_Accidents', 'Mean_Distance', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_Distance": {
        "feature_list": ['Total_Accidents', 'Mean_Severity', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "No_TotalAccidents": {
        "feature_list": ['Mean_Severity', 'Mean_Distance', 'Mean_Humidity'],
        "seq_len": 4,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "SeqLen1": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 1,
        "adjacency_mode": "inverse_distance",
        "run_lstm": True,
        "run_stgnn": True
    },
    "BinaryAdjacency": {
        "feature_list": ABLATION_FULL_FEATURE_LIST,
        "seq_len": 4,
        "adjacency_mode": "binary",
        "run_lstm": False,
        "run_stgnn": True
    }
}

print(f"[*] Iowa ablation suite locked to champion topology: "
      f"{ABLATION_CHAMP_NODES} Nodes | {ABLATION_CHAMP_RADIUS}mi | K={ABLATION_CHAMP_K}")
print(f"[*] Ablation results will be saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C HELPER FUNCTIONS (Additive Only)
# ==============================================================================

def _build_adjacency_from_centers_utm_ablation(centers_xy, k_neighbors, adjacency_mode="inverse_distance"):
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    knn.fit(centers_xy)
    distances, indices = knn.kneighbors(centers_xy)

    n_nodes = centers_xy.shape[0]
    adj_matrix = np.zeros((n_nodes, n_nodes), dtype=float)

    for i in range(n_nodes):
        for j in range(1, k_neighbors + 1):
            nb = indices[i, j]
            d = max(distances[i, j], 1e-6)

            if adjacency_mode == "inverse_distance":
                w = 1.0 / d
            elif adjacency_mode == "binary":
                w = 1.0
            else:
                raise ValueError(f"[ERROR] Unknown adjacency_mode: {adjacency_mode}")

            adj_matrix[i, nb] = w
            adj_matrix[nb, i] = w

    adj_tilde = adj_matrix + np.eye(n_nodes)
    rowsum = np.array(adj_tilde.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt)

    deg = normalized_adj.sum(axis=1)
    if not (deg > 0).all():
        iso = int((deg == 0).sum())
        raise ValueError(f"[CRITICAL] Ablation adjacency contains {iso} isolated nodes.")

    return normalized_adj


def aggregate_to_hourly_ablation(df, target_col, feature_list):
    df = df.copy()
    df['Time_Step'] = df['Start_Time'].dt.floor('h')

    agg_df = df.groupby(['Region_ID', 'Time_Step']).agg(
        Total_Accidents=(target_col, 'count'),
        STPC_Count=(target_col, 'sum'),
        Mean_Severity=('Severity', 'mean'),
        Mean_Distance=('Distance(mi)', 'mean'),
        Mean_Humidity=('Humidity(%)', 'mean')
    ).reset_index()

    full_time = pd.date_range(start=df['Time_Step'].min(), end=df['Time_Step'].max(), freq='h')

    multi_idx = pd.MultiIndex.from_product(
        [sorted(df['Region_ID'].unique()), full_time],
        names=['Region_ID', 'Time_Step']
    )

    agg_df.set_index(['Region_ID', 'Time_Step'], inplace=True)
    continuous_df = agg_df.reindex(multi_idx, fill_value=0).reset_index()

    n_nodes = len(sorted(df['Region_ID'].unique()))
    city_features = np.zeros((len(full_time), n_nodes, len(feature_list)))

    for idx, feat in enumerate(feature_list):
        city_features[:, :, idx] = continuous_df.pivot(
            index='Time_Step', columns='Region_ID', values=feat
        ).fillna(0).values

    city_targets = continuous_df.pivot(
        index='Time_Step', columns='Region_ID', values='STPC_Count'
    ).fillna(0).values

    city_targets = (city_targets > 0).astype(int)
    return city_features, city_targets, full_time


def build_st_graph_tensors_ablation(df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4, feature_list=None, adjacency_mode="inverse_distance"):
    if feature_list is None:
        feature_list = ABLATION_FULL_FEATURE_LIST

    df = df_raw.copy()
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    df['Region_ID'] = kmeans.fit_predict(coords_utm)
    centers_utm = kmeans.cluster_centers_

    normalized_adj = _build_adjacency_from_centers_utm_ablation(
        centers_utm, k_neighbors, adjacency_mode=adjacency_mode
    )

    city_features, city_targets, full_time = aggregate_to_hourly_ablation(
        df, target_col, feature_list
    )

    X_tr, X_val, X_te, y_tr, y_val, y_te = _windows_and_split(
        city_features, city_targets, seq_len=seq_len
    )

    return X_tr, X_val, X_te, y_tr, y_val, y_te, normalized_adj


def get_frozen_threshold_ablation(model, X_val, y_val_flat):
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)
    valid_idx = np.where(recalls >= 0.70)[0]
    return thresholds[valid_idx[np.argmax(precisions[valid_idx])]] if len(valid_idx) > 0 else 0.5


def evaluate_nn_ablation(model, X_test, y_test_flat, opt_thresh):
    y_pred_probs = model.predict(X_test, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_test_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_test_flat, y_pred_binary, zero_division=0)
    return auc_pr, f1


# ==============================================================================
# PHASE 6C RUNNER
# ==============================================================================

def run_core_ablation_suite(df_raw, target_city, experiment_id, champ_nodes, champ_radius, champ_k, ablation_configs, n_runs=5, results_path=None):
    ablation_results = []

    for ablation_name, cfg in ablation_configs.items():
        print("\n==================================================================")
        print(f"[*] ABLATION: {ablation_name}")
        print("==================================================================")

        feature_list = cfg["feature_list"]
        seq_len = cfg["seq_len"]
        adjacency_mode = cfg["adjacency_mode"]
        run_lstm = cfg["run_lstm"]
        run_stgnn = cfg["run_stgnn"]

        X_tr = X_val = X_te = y_tr = y_val = y_te = adj = None

        try:
            X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors_ablation(
                df_raw=df_raw,
                n_nodes=champ_nodes,
                radius_mi=champ_radius,
                k_neighbors=champ_k,
                seq_len=seq_len,
                feature_list=feature_list,
                adjacency_mode=adjacency_mode
            )

            total_pos = np.sum(y_tr)
            if total_pos == 0:
                print(f"[WARNING] {ablation_name} yielded zero positives. Skipping.")
                continue

            pos_weight_val = min(float(((y_tr.size) - total_pos) / max(total_pos, 1)), 1000.0)
            y_val_flat = y_val.flatten()
            y_te_flat = y_te.flatten()

            # ------------------------------------------------------------------
            # LSTM Blind
            # ------------------------------------------------------------------
            if run_lstm:
                lstm_auc_scores, lstm_f1_scores = [], []
                print(f"[*] Training LSTM_Blind ({n_runs} runs)...")

                for run in range(n_runs):
                    random.seed(42 + run)
                    np.random.seed(42 + run)
                    tf.random.set_seed(42 + run)

                    lstm_model = build_baseline_lstm(seq_len, champ_nodes, X_tr.shape[-1], pos_weight_val)
                    early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                    lstm_model.fit(
                        X_tr, y_tr, validation_data=(X_val, y_val),
                        epochs=15, batch_size=64, callbacks=[early_stop_lstm], verbose=0
                    )

                    thresh_lstm = get_frozen_threshold_ablation(lstm_model, X_val, y_val_flat)
                    auc_lstm, f1_lstm = evaluate_nn_ablation(lstm_model, X_te, y_te_flat, thresh_lstm)

                    lstm_auc_scores.append(float(auc_lstm))
                    lstm_f1_scores.append(float(f1_lstm))

                    print(f"    [RUN {run+1}/{n_runs}] LSTM AUC-PR={auc_lstm:.4f} | F1={f1_lstm:.4f}")

                    del lstm_model, early_stop_lstm
                    tf.keras.backend.clear_session()
                    gc.collect()

                ablation_results.append({
                    "domain": target_city, "experiment_id": experiment_id, "phase_id": ABLATION_PHASE_ID, "ablation_name": ablation_name,
                    "model_name": "LSTM_Blind", "champion_nodes": champ_nodes, "champion_radius_mi": champ_radius, "champion_k_neighbors": champ_k,
                    "feature_list": ",".join(feature_list), "seq_len": seq_len, "adjacency_mode": adjacency_mode, "n_runs": n_runs,
                    "test_auc_pr_mean": float(np.mean(lstm_auc_scores)), "test_auc_pr_std": float(np.std(lstm_auc_scores)),
                    "test_f1_mean": float(np.mean(lstm_f1_scores)), "test_f1_std": float(np.std(lstm_f1_scores)),
                })
                print(f"    -> [LSTM Summary] Mean AUC-PR: {np.mean(lstm_auc_scores):.4f} ± {np.std(lstm_auc_scores):.4f} | Mean F1: {np.mean(lstm_f1_scores):.4f} ± {np.std(lstm_f1_scores):.4f}")

            # ------------------------------------------------------------------
            # ST-GNN
            # ------------------------------------------------------------------
            if run_stgnn:
                stgnn_auc_scores, stgnn_f1_scores = [], []
                print(f"[*] Training STGNN_StaticKNN ({n_runs} runs)...")

                for run in range(n_runs):
                    random.seed(42 + run)
                    np.random.seed(42 + run)
                    tf.random.set_seed(42 + run)

                    stgnn_model = build_st_gnn_model(seq_len, champ_nodes, X_tr.shape[-1], adj, pos_weight_val)
                    early_stop_stgnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

                    stgnn_model.fit(
                        X_tr, y_tr, validation_data=(X_val, y_val),
                        epochs=15, batch_size=64, callbacks=[early_stop_stgnn], verbose=0
                    )

                    thresh_stgnn = get_frozen_threshold_ablation(stgnn_model, X_val, y_val_flat)
                    auc_stgnn, f1_stgnn = evaluate_nn_ablation(stgnn_model, X_te, y_te_flat, thresh_stgnn)

                    stgnn_auc_scores.append(float(auc_stgnn))
                    stgnn_f1_scores.append(float(f1_stgnn))

                    print(f"    [RUN {run+1}/{n_runs}] STGNN AUC-PR={auc_stgnn:.4f} | F1={f1_stgnn:.4f}")

                    del stgnn_model, early_stop_stgnn
                    tf.keras.backend.clear_session()
                    gc.collect()

                ablation_results.append({
                    "domain": target_city, "experiment_id": experiment_id, "phase_id": ABLATION_PHASE_ID, "ablation_name": ablation_name,
                    "model_name": "STGNN_StaticKNN", "champion_nodes": champ_nodes, "champion_radius_mi": champ_radius, "champion_k_neighbors": champ_k,
                    "feature_list": ",".join(feature_list), "seq_len": seq_len, "adjacency_mode": adjacency_mode, "n_runs": n_runs,
                    "test_auc_pr_mean": float(np.mean(stgnn_auc_scores)), "test_auc_pr_std": float(np.std(stgnn_auc_scores)),
                    "test_f1_mean": float(np.mean(stgnn_f1_scores)), "test_f1_std": float(np.std(stgnn_f1_scores)),
                })
                print(f"    -> [STGNN Summary] Mean AUC-PR: {np.mean(stgnn_auc_scores):.4f} ± {np.std(stgnn_auc_scores):.4f} | Mean F1: {np.mean(stgnn_f1_scores):.4f} ± {np.std(stgnn_f1_scores):.4f}")

            # Interim save
            if results_path is not None:
                pd.DataFrame(ablation_results).to_csv(results_path, index=False)

        except Exception as e:
            print(f"[ERROR] Ablation '{ablation_name}' failed: {e}")

        finally:
            if 'X_tr' in locals() and X_tr is not None:
                del X_tr, X_val, X_te, y_tr, y_val, y_te, adj
            tf.keras.backend.clear_session()
            gc.collect()

    return pd.DataFrame(ablation_results)

# ==============================================================================
# RUN PHASE 6C FOR IOWA
# ==============================================================================
import os

iowa_ablation_df = run_core_ablation_suite(
    df_raw=df_target,
    target_city=ABLATION_TARGET_CITY,
    experiment_id=ABLATION_EXPERIMENT_ID,
    champ_nodes=ABLATION_CHAMP_NODES,
    champ_radius=ABLATION_CHAMP_RADIUS,
    champ_k=ABLATION_CHAMP_K,
    ablation_configs=ABLATION_CONFIGS,
    n_runs=ABLATION_RUNS,
    results_path=ABLATION_RESULTS_PATH
)

print("\n[SUCCESS] Phase 6C complete.")
print(f"[*] Partial ablation results saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C POST-PROCESSING: DELTA VS FULL
# ==============================================================================
if os.path.exists(ABLATION_RESULTS_PATH):
    ablation_df = pd.read_csv(ABLATION_RESULTS_PATH)
else:
    ablation_df = iowa_ablation_df.copy()

full_ref = ablation_df[ablation_df["ablation_name"] == "Full"][["model_name", "test_auc_pr_mean", "test_f1_mean"]].copy()
if full_ref.empty:
    raise ValueError("[ERROR] Full ablation reference missing. Cannot compute deltas. Ensure 'Full' ablation succeeded.")

full_ref = full_ref.rename(columns={
    "test_auc_pr_mean": "full_auc_pr_mean",
    "test_f1_mean": "full_f1_mean"
})

ablation_df = ablation_df.merge(full_ref, on="model_name", how="left")
ablation_df["delta_auc_pr_vs_full"] = ablation_df["test_auc_pr_mean"] - ablation_df["full_auc_pr_mean"]
ablation_df["delta_f1_vs_full"] = ablation_df["test_f1_mean"] - ablation_df["full_f1_mean"]

ablation_df.to_csv(ABLATION_RESULTS_PATH, index=False)

print("\n[*] Ablation results with deltas vs full model:")
print(ablation_df.round(4).to_string(index=False))
print(f"\n[SUCCESS] Final ablation table with deltas saved to: {ABLATION_RESULTS_PATH}")

# ==============================================================================
# PHASE 6C ARTIFACT: AUC-PR BY ABLATION
# ==============================================================================
AUC_PLOT_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_AUC.png"
plot_df = ablation_df.copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x="ablation_name",
    y="test_auc_pr_mean",
    hue="model_name"
)
plt.title(
    f"Core Ablation Suite — Test AUC-PR\nDomain: {ABLATION_TARGET_CITY.upper()} | "
    f"Champion: {ABLATION_CHAMP_NODES} Nodes, {ABLATION_CHAMP_RADIUS}mi, K={ABLATION_CHAMP_K}",
    fontweight='bold'
)
plt.xlabel("Ablation")
plt.ylabel("Mean Test AUC-PR")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(AUC_PLOT_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] Ablation AUC plot saved to: {AUC_PLOT_PATH}")
plt.show()
plt.close()

# ==============================================================================
# PHASE 6C ARTIFACT: DELTA AUC-PR VS FULL
# ==============================================================================
DELTA_PLOT_PATH = f"{BASE_DIR}/{ABLATION_TARGET_CITY}_{ABLATION_EXPERIMENT_ID}_Phase6C_Ablation_Delta.png"
delta_plot_df = ablation_df[ablation_df["ablation_name"] != "Full"].copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=delta_plot_df,
    x="ablation_name",
    y="delta_auc_pr_vs_full",
    hue="model_name"
)
plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
plt.title(f"Core Ablation Suite — ΔAUC-PR vs Full Model\nDomain: {ABLATION_TARGET_CITY.upper()}", fontweight='bold')
plt.xlabel("Ablation")
plt.ylabel("Delta AUC-PR vs Full")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(DELTA_PLOT_PATH, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] Ablation delta plot saved to: {DELTA_PLOT_PATH}")
plt.show()
plt.close()

##PHASE 6D: ST-GNN PERMUTATION IMPORTANCE

In [ ]:
# ==============================================================================
# PHASE 6D: ST-GNN PERMUTATION IMPORTANCE (IOWA NATIVE CHAMPION)
# ==============================================================================
print(f"\n[*] Initializing Phase 6D: ST-GNN Permutation Importance for {TARGET_CITY.upper()}...")

import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------------------------
# 1. RELOAD / LOCK THE IOWA NATIVE CHAMPION FROM PHASE 5 ARENA RESULTS
# ------------------------------------------------------------------------------
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"[ERROR] Arena results not found at {RESULTS_PATH}. "
        "Run Iowa Experiment C2 Phase 5 first."
    )

arena_df = pd.read_csv(RESULTS_PATH)
if arena_df.empty:
    raise ValueError("[ERROR] Arena results file is empty.")

# Dynamically lock the best Iowa topology
champion_row = arena_df.sort_values(by='Val_AUC_PR', ascending=False).iloc[0]
CHAMP_NODES = int(champion_row['Nodes'])
CHAMP_RADIUS = float(champion_row['Radius_mi'])
CHAMP_K = int(champion_row['K_Neighbors'])

print(f"[*] Champion Topology Locked for Permutation Importance: "
      f"{CHAMP_NODES} Nodes | {CHAMP_RADIUS}mi | K={CHAMP_K}")

# ------------------------------------------------------------------------------
# 2. REBUILD THE CHAMPION TENSORS
# ------------------------------------------------------------------------------
X_tr, X_val, X_te, y_tr, y_val, y_te, adj = build_st_graph_tensors(
    df_raw=df_target,
    n_nodes=CHAMP_NODES,
    radius_mi=CHAMP_RADIUS,
    k_neighbors=CHAMP_K,
    seq_len=SEQ_LENGTH
)

total_pos = np.sum(y_tr)
if total_pos == 0:
    raise ValueError("[ERROR] Champion topology produced zero STPC labels in training set.")

# Strict sparse-domain safety cap for Iowa
pos_weight_val = min(float(((y_tr.size) - total_pos) / max(total_pos, 1)), 1000.0)

y_val_flat = y_val.flatten()
y_te_flat = y_te.flatten()

# Feature names: use notebook FEATURES_LIST if present, otherwise fall back
if 'FEATURES_LIST' in globals():
    perm_feature_names = list(FEATURES_LIST)
else:
    perm_feature_names = ['Total_Accidents', 'Mean_Severity', 'Mean_Distance', 'Mean_Humidity']

if X_te.shape[-1] != len(perm_feature_names):
    raise ValueError(
        f"[ERROR] Feature count mismatch. X_te has {X_te.shape[-1]} channels, "
        f"but perm_feature_names has {len(perm_feature_names)} entries."
    )

PERM_RESULTS_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentC2_Phase6D_Permutation_Importance.csv"

# ------------------------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------------------------
def get_frozen_threshold(model, X_val, y_val_flat):
    """
    Derive the operational threshold from the validation set using the same
    recall >= 0.70 rule used elsewhere in the notebook, with a safe index guard.
    """
    y_val_probs = model.predict(X_val, batch_size=64, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val_flat, y_val_probs)

    valid_idx = np.where(recalls >= 0.70)[0]
    if len(valid_idx) == 0:
        return 0.5

    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5


def evaluate_nn(model, X_eval, y_eval_flat, opt_thresh):
    """
    Return both AUC-PR and thresholded F1 on the supplied tensor.
    """
    y_pred_probs = model.predict(X_eval, batch_size=64, verbose=0).flatten()
    auc_pr = average_precision_score(y_eval_flat, y_pred_probs)
    y_pred_binary = (y_pred_probs >= opt_thresh).astype(int)
    f1 = f1_score(y_eval_flat, y_pred_binary, zero_division=0)
    return float(auc_pr), float(f1)


def permute_feature_channel(X_in, feature_idx, seed):
    """
    Permute one feature channel across the entire test tensor while preserving
    the marginal distribution of that channel.

    X_in shape: (samples, time_steps, nodes, features)
    """
    X_perm = X_in.copy()
    rng = np.random.default_rng(seed)

    # SCIENTIFIC FIX: Use .flatten() to ensure a clean, contiguous memory copy
    flat_channel = X_perm[..., feature_idx].flatten()
    rng.shuffle(flat_channel)
    X_perm[..., feature_idx] = flat_channel.reshape(X_perm[..., feature_idx].shape)

    return X_perm


# Initialize variables for the finally block
perm_model = None
early_stop_perm = None

try:
    # ------------------------------------------------------------------------------
    # 4. TRAIN ONE CHAMPION ST-GNN (FIXED SEED) FOR INTERPRETABILITY
    # ------------------------------------------------------------------------------
    print("\n[*] Training one champion ST-GNN instance for permutation importance...")
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)

    perm_model = build_st_gnn_model(
        SEQ_LENGTH,
        CHAMP_NODES,
        X_tr.shape[-1],
        adj,
        pos_weight_val
    )

    early_stop_perm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    perm_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[early_stop_perm],
        verbose=0
    )

    frozen_thresh = get_frozen_threshold(perm_model, X_val, y_val_flat)
    baseline_auc, baseline_f1 = evaluate_nn(perm_model, X_te, y_te_flat, frozen_thresh)

    print(f"[BASELINE] Champion ST-GNN Test AUC-PR: {baseline_auc:.4f} | Test F1: {baseline_f1:.4f}")
    print(f"[*] Frozen threshold from validation: {frozen_thresh:.4f}")

    # ------------------------------------------------------------------------------
    # 5. PERMUTATION IMPORTANCE LOOP
    # ------------------------------------------------------------------------------
    N_PERMUTATIONS = 10
    print(f"\n[*] Running permutation importance with {N_PERMUTATIONS} shuffles per feature...")

    perm_results = []

    for feat_idx, feat_name in enumerate(perm_feature_names):
        auc_scores = []
        f1_scores = []

        print(f"\n--- [FEATURE] {feat_name} ---")

        for p in range(N_PERMUTATIONS):
            perm_seed = 1000 + (feat_idx * 100) + p
            X_te_perm = permute_feature_channel(X_te, feat_idx, seed=perm_seed)

            auc_perm, f1_perm = evaluate_nn(perm_model, X_te_perm, y_te_flat, frozen_thresh)
            auc_scores.append(auc_perm)
            f1_scores.append(f1_perm)

            print(f"    [PERM {p+1}/{N_PERMUTATIONS}] AUC-PR={auc_perm:.4f} | F1={f1_perm:.4f}")

            del X_te_perm
            gc.collect()

        auc_mean = float(np.mean(auc_scores))
        auc_std = float(np.std(auc_scores))
        f1_mean = float(np.mean(f1_scores))
        f1_std = float(np.std(f1_scores))

        delta_auc = auc_mean - baseline_auc
        delta_f1 = f1_mean - baseline_f1

        perm_results.append({
            "City": TARGET_CITY,
            "Experiment": "ExperimentC2_RuralNative",
            "Phase": "Phase6D_PermutationImportance",
            "Champion_Nodes": CHAMP_NODES,
            "Champion_Radius_mi": CHAMP_RADIUS,
            "Champion_K": CHAMP_K,
            "Feature_Name": feat_name,
            "Baseline_AUC_PR": round(baseline_auc, 4),
            "Permuted_AUC_PR_Mean": round(auc_mean, 4),
            "Permuted_AUC_PR_Std": round(auc_std, 4),
            "Delta_AUC_PR_vs_Baseline": round(delta_auc, 4),
            "Baseline_F1": round(baseline_f1, 4),
            "Permuted_F1_Mean": round(f1_mean, 4),
            "Permuted_F1_Std": round(f1_std, 4),
            "Delta_F1_vs_Baseline": round(delta_f1, 4),
            "N_Permutations": N_PERMUTATIONS
        })

        print(f"    -> [SUMMARY] {feat_name}: "
              f"AUC-PR {auc_mean:.4f} ± {auc_std:.4f} | ΔAUC={delta_auc:.4f}")

        # SCIENTIFIC FIX: Incremental saving to protect against Colab disconnects
        pd.DataFrame(perm_results).to_csv(PERM_RESULTS_PATH, index=False)

    # ------------------------------------------------------------------------------
    # 6. FINAL RESULTS DISPLAY
    # ------------------------------------------------------------------------------
    perm_df = pd.DataFrame(perm_results)
    print(f"\n[SUCCESS] Permutation-importance table saved to: {PERM_RESULTS_PATH}")
    print("\n[*] Permutation Importance Results:")
    print(perm_df.to_string(index=False))

    # ------------------------------------------------------------------------------
    # 7. PLOT ΔAUC-PR vs BASELINE
    # ------------------------------------------------------------------------------
    PERM_PLOT_PATH = f"{BASE_DIR}/{TARGET_CITY}_ExperimentC2_Phase6D_Permutation_Importance.png"

    plot_df = perm_df.copy()
    plot_df = plot_df.sort_values(by="Delta_AUC_PR_vs_Baseline", ascending=True)

    plt.figure(figsize=(10, 6))
    bars = plt.barh(
        plot_df["Feature_Name"],
        plot_df["Delta_AUC_PR_vs_Baseline"],
        color="#1a9641", # Using green to denote rural Iowa scheme
        edgecolor="black"
    )

    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.title(
        f"ST-GNN Permutation Importance (ΔAUC-PR vs Baseline)\n"
        f"Domain: {TARGET_CITY.upper()} | Champion: {CHAMP_NODES} Nodes, {CHAMP_RADIUS}mi, K={CHAMP_K}",
        fontweight="bold"
    )
    plt.xlabel("Delta AUC-PR vs Baseline (More Negative = More Important)")
    plt.ylabel("Feature")

    for bar, delta in zip(bars, plot_df["Delta_AUC_PR_vs_Baseline"]):
        plt.text(
            bar.get_width() + (0.001 if delta >= 0 else -0.001),
            bar.get_y() + bar.get_height() / 2,
            f"{delta:.4f}",
            va="center",
            ha="left" if delta >= 0 else "right",
            fontsize=10,
            fontweight="bold"
        )

    plt.tight_layout()
    plt.savefig(PERM_PLOT_PATH, dpi=300, bbox_inches="tight")
    print(f"[SUCCESS] Permutation-importance plot saved to: {PERM_PLOT_PATH}")
    plt.show()
    plt.close()

    # ------------------------------------------------------------------------------
    # 8. OPTIONAL ONE-LINE MORPHOLOGY-LINKED INTERPRETATION HINT
    # ------------------------------------------------------------------------------
    most_important_row = plot_df.iloc[0]   # most negative delta = biggest performance drop
    most_important_feature = most_important_row["Feature_Name"]
    most_important_delta = most_important_row["Delta_AUC_PR_vs_Baseline"]

    print("\n[*] Morphology-Linked Interpretation Hint:")
    print(
        f"Permuting '{most_important_feature}' produced the largest drop in Test AUC-PR "
        f"({most_important_delta:.4f}), suggesting that the Iowa champion ST-GNN relies most strongly "
        f"on this signal when predicting rare cascade risks across a sparse, macro-regional topology."
    )

finally:
    # ------------------------------------------------------------------------------
    # 9. CLEANUP (Guaranteed Execution)
    # ------------------------------------------------------------------------------
    if perm_model is not None:
        del perm_model
    if early_stop_perm is not None:
        del early_stop_perm
    if 'X_tr' in locals() and X_tr is not None:
        del X_tr, X_val, X_te, y_tr, y_val, y_te, adj

    tf.keras.backend.clear_session()
    gc.collect()
    print("\n[SUCCESS] Phase 6D complete and memory safely cleared.")

## PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
**Methodology:** To visually validate the necessity of the Spatio-Temporal Graph Neural Network (ST-GNN), this phase maps the empirical ground truth of the "Butterfly Effect" within the target urban morphology.
By isolating the highest-density temporal window (the "Worst Day"), we plot the primary physical triggers (Primary Accidents) against the resulting friction cascades (Secondary Accidents). This visual artifact provides spatial evidence of the Champion Topology discovered in Phase 5, displaying the true physical spread of the traffic shockwave across the target urban morphology.

In [ ]:
# ==============================================================================
# PHASE 7: Spatial Morphology Visualization (Empirical Ground Truth)
# ==============================================================================
print(f"\n[*] Initializing Phase 7: Spatial Morphology Visualization for {TARGET_CITY.upper()}...")

df_map = df_target.copy()
df_map['Start_Time'] = pd.to_datetime(df_map['Start_Time'])
daily_secondary_counts = df_map[df_map[f'Is_STPC_{CHAMP_RADIUS}'] == 1].groupby(df_map['Start_Time'].dt.date).size()

if len(daily_secondary_counts) == 0:
    print("[WARNING] No STPCs found to map in Phase 7.")
else:
    worst_day = daily_secondary_counts.idxmax()
    print(f"[*] Isolating Maximum Density Temporal Window: {worst_day}")

    day_data = df_map[df_map['Start_Time'].dt.date == worst_day]
    primary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 0]
    secondary_accidents = day_data[day_data[f'Is_STPC_{CHAMP_RADIUS}'] == 1]

    num_primary = len(primary_accidents); num_secondary = len(secondary_accidents)
    print(f"[*] Plotted Graph Nodes: {num_primary} Primary Triggers | {num_secondary} Secondary Cascades")

    center_lat = day_data['Start_Lat'].mean(); center_lng = day_data['Start_Lng'].mean()
    road_map = folium.Map(location=[center_lat, center_lng], zoom_start=8, control_scale=True)

    for _, row in primary_accidents.iterrows():
        folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                            radius=5, color='#3498db', fill=True, fill_color='#3498db', fill_opacity=0.6,
                            tooltip="Primary Accident (Isolated Trigger)").add_to(road_map)
    for _, row in secondary_accidents.iterrows():
        folium.CircleMarker(location=[row['Start_Lat'], row['Start_Lng']],
                            radius=8, color='black', weight=1.5, fill=True, fill_color='#e74c3c', fill_opacity=0.9,
                            tooltip=f"STPC (Proximate Collision Cascade) (Within {CHAMP_RADIUS}mi)").add_to(road_map)

    legend_html = f'''
    <div style="
        position: fixed;
        bottom: 50px; left: 50px; width: 320px; height: auto;
        background-color: white; border: 2px solid grey; z-index: 9999;
        font-size: 14px; font-family: Arial, sans-serif;
        padding: 15px; box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
        ">
        <h4 style="margin-top: 0; margin-bottom: 5px; font-weight: bold; font-size: 15px;">
            Empirical Spatial Morphology
        </h4>
        <hr style="margin: 5px 0;">
        <b>Domain:</b> {TARGET_CITY.upper()}<br>
        <b>Temporal Window:</b> {worst_day}<br>
        <b>Champion Topology:</b> {CHAMP_RADIUS}mi Radius<br>
        <hr style="margin: 10px 0;">
        <div style="margin-bottom: 8px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: #3498db; border-radius: 50%; margin-right: 8px; opacity: 0.6;"></span>
            <b>Primary Trigger</b> (n={num_primary})
        </div>
        <div>
            <span style="display: inline-block; width: 14px; height: 14px; background-color: #e74c3c; border: 2px solid black; border-radius: 50%; margin-right: 8px;"></span>
            <b>Secondary Cascade</b> (n={num_secondary})
        </div>
    </div>
    '''
    road_map.get_root().html.add_child(Element(legend_html))
    output_html = f'{BASE_DIR}/{TARGET_CITY}_Experiment_C2_Shockwave_Morphology_Map.html'
    road_map.save(output_html)
    print(f"\n==================================================================")
    print(f"[SUCCESS] THESIS ARTIFACT GENERATED: {output_html}")
    print(f"==================================================================")

    from IPython.display import display
    display(road_map)

##Phase 8: Temporal Drift & Calibration

In [ ]:
# ==============================================================================
# PHASE 8: TEMPORAL DRIFT & CALIBRATION (2020 Adversarial External Validation)
# ==============================================================================
print(f"\n[*] Initiating Phase 8: Temporal Drift Protocol for {TARGET_CITY.upper()} (Experiment C2)...")

import statsmodels.api as sm
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def build_st_graph_tensors_temporal_external(
    df_raw, n_nodes, radius_mi, k_neighbors, seq_len=4, cutoff=pd.Timestamp('2020-01-01')
):
    df = df_raw.copy().sort_values('Start_Time').reset_index(drop=True)
    target_col = f'Is_STPC_{radius_mi}'

    transformer, epsg_code = _dynamic_utm_transformer(df)
    coords_utm = _project_to_utm(df, transformer)
    df['_X'], df['_Y'] = coords_utm[:, 0], coords_utm[:, 1]

    train_mask = df['Start_Time'] < cutoff
    # test_mask explicitly removed per critique (unused variable)

    train_points = df.loc[train_mask, ['_X', '_Y']].values
    if train_points.shape[0] < n_nodes:
        raise ValueError(
            f"[ERROR] External-validation pre-cutoff set has only {train_points.shape[0]} points, "
            f"which is fewer than n_clusters={n_nodes}."
        )

    kmeans = KMeans(n_clusters=n_nodes, random_state=42, n_init=10)
    kmeans.fit(train_points)

    df['Region_ID'] = kmeans.predict(df[['_X', '_Y']].values)
    adj = _build_adjacency_from_centers_utm(kmeans.cluster_centers_, k_neighbors)

    city_features, city_targets, full_time = _aggregate_to_hourly(df, target_col)

    X, y, t_end = [], [], []
    for i in range(len(city_features) - seq_len):
        X.append(city_features[i:i+seq_len])
        y.append(city_targets[i+seq_len])
        t_end.append(full_time[i+seq_len])

    X, y, t_end = np.array(X), np.array(y), np.array(t_end)

    dev_idx = np.where(t_end < cutoff)[0]
    test_idx = np.where((t_end >= pd.Timestamp('2020-01-01')) & (t_end < pd.Timestamp('2021-01-01')))[0]

    if len(dev_idx) == 0 or len(test_idx) == 0:
        raise ValueError("[ERROR] Not enough samples in dev or 2020 windows.")

    dev_X, dev_y = X[dev_idx], y[dev_idx]
    train_bound = int(dev_X.shape[0] * 0.82)
    purge = seq_len

    X_tr_raw, y_tr = dev_X[:train_bound], dev_y[:train_bound]

    # SCIENTIFIC FIX: Guard against an empty training split
    if len(X_tr_raw) == 0:
        raise ValueError("[ERROR] External-validation training split is empty after chronological split.")

    X_val_raw, y_val = dev_X[train_bound + purge:], dev_y[train_bound + purge:]
    X_te_raw, y_te = X[test_idx], y[test_idx]

    # Critical guard after purge
    if len(X_val_raw) == 0 or len(X_te_raw) == 0:
        raise ValueError("[ERROR] Temporal purge emptied validation or 2020 test set.")

    num_features = X_tr_raw.shape[-1]
    scaler = MinMaxScaler()
    X_tr = scaler.fit_transform(X_tr_raw.reshape(-1, num_features)).reshape(X_tr_raw.shape)
    X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
    X_te  = scaler.transform(X_te_raw.reshape(-1, num_features)).reshape(X_te_raw.shape)

    df.drop(columns=['_X', '_Y'], inplace=True)
    return X_tr, X_val, X_te, y_tr, y_val, y_te, adj

model_drift = None
try:
    X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext = build_st_graph_tensors_temporal_external(
        df_target, CHAMP_NODES, CHAMP_RADIUS, CHAMP_K, seq_len=SEQ_LENGTH, cutoff=pd.Timestamp('2020-01-01')
    )

    print(f"[*] Validation Era Prevalence (Pre-2020) : {float(y_val_ext.mean()):.4f}")
    print(f"[*] Adversarial Era Prevalence (2020)    : {float(y_te_2020.mean()):.4f}")

    pos_ext = y_tr_ext.sum()
    if pos_ext < 10:
        raise ValueError(
            f"[ERROR] Pre-2020 dev set produced only {int(pos_ext)} positives. "
            "Insufficient for training."
        )

    pos_te_2020 = y_te_2020.sum()
    if pos_te_2020 < 10:
        raise ValueError(
            f"[ERROR] 2020 adversarial test set produced only {int(pos_te_2020)} positives. "
            "Insufficient for calibration assessment."
        )

    y_true_2020 = y_te_2020.flatten()

    # SCIENTIFIC FIX: Guard against single-class test sets
    if len(np.unique(y_true_2020)) < 2:
        raise ValueError("[ERROR] 2020 adversarial test set contains only one class. Calibration assessment is not mathematically possible.")

    pos_weight_ext = min(float((y_tr_ext.size - pos_ext) / max(pos_ext, 1)), 1000.0)

    model_drift = build_st_gnn_model(
        SEQ_LENGTH, CHAMP_NODES, X_tr_ext.shape[-1], adj_ext, pos_weight_ext
    )
    model_drift.fit(
        X_tr_ext, y_tr_ext,
        validation_data=(X_val_ext, y_val_ext),
        epochs=15,
        batch_size=64,
        verbose=0
    )

    y_pred_probs_2020 = model_drift.predict(X_te_2020, verbose=0).flatten()

    # --- RESTORED: Statistical Depth (Bootstrapping & GLM) ---
    def bootstrap_pr_auc(y_true, y_pred, n_bootstraps=500):
        scores = []
        N = len(y_pred)
        for _ in range(n_bootstraps):
            idx = np.random.randint(0, N, N)
            if len(np.unique(y_true[idx])) < 2:
                continue
            scores.append(average_precision_score(y_true[idx], y_pred[idx]))
        if len(scores) == 0:
            return np.nan, np.nan
        return np.percentile(scores, [2.5, 97.5])

    def calibration_slope_intercept(y_true, p_hat):
        eps = 1e-6
        logit_ph = np.log(np.clip(p_hat, eps, 1-eps) / np.clip(1-p_hat, eps, 1-eps))
        X = sm.add_constant(logit_ph)  # intercept + slope
        model = sm.GLM(y_true, X, family=sm.families.Binomial())
        res = model.fit()
        return float(res.params[0]), float(res.params[1])

    drift_auc_pr = average_precision_score(y_true_2020, y_pred_probs_2020)
    boot_ci_lo, boot_ci_hi = bootstrap_pr_auc(y_true_2020, y_pred_probs_2020)
    drift_brier = brier_score_loss(y_true_2020, y_pred_probs_2020)

    try:
        cal_intercept_2020, cal_slope_2020 = calibration_slope_intercept(y_true_2020, y_pred_probs_2020)
    except Exception:
        cal_intercept_2020, cal_slope_2020 = float('nan'), float('nan')

    print("\n========================================================")
    print("[*] TEMPORAL DRIFT & CALIBRATION ASSESSMENT (2020)")
    print("========================================================")
    print(f"Ranking Retention (PR-AUC) : {drift_auc_pr:.4f} (95% CI: [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}])")
    print(f"Calibration Brier Score    : {drift_brier:.4f}")
    print(f"Calibration Intercept      : {cal_intercept_2020:.3f} (Deviation from 0 = Risk Shift)")
    print(f"Calibration Slope          : {cal_slope_2020:.3f} (Deviation from 1 = Over/Underfitting)")

    prob_true, prob_pred = calibration_curve(
        y_true_2020, y_pred_probs_2020, n_bins=10, strategy='quantile'
    )

    # --- RESTORED: Binning Sanity Check ---
    bins = np.percentile(y_pred_probs_2020, np.linspace(0, 100, 11))
    bin_indices = np.digitize(y_pred_probs_2020, bins[1:-1])
    bin_counts = np.bincount(bin_indices, minlength=10)

    print("\n[*] Reliability Curve Binning Sanity Check:")
    for idx, count in enumerate(bin_counts):
        print(f"    -> Bin {idx+1}: n = {int(count)} samples")
    if np.min(bin_counts) < 50:
        print("    [WARNING] Some bins have < 50 samples. Reliability estimates may be unstable.")

    plt.figure(figsize=(8, 6))
    plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='ST-GNN (2020 Adversarial Test)')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.title(f"Reliability Curve: Structural Integrity vs Baseline Shift\nDomain: {TARGET_CITY.upper()} (2020)")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction of Actual STPCs")
    plt.legend()
    plt.grid(True)
    plt.show()

except Exception as e:
    print(f"\n[!] Phase 8 skipped or failed due to sparse 2020 timeline data in {TARGET_CITY}. Reason: {e}")

finally:
    # SCIENTIFIC FIX: Guaranteed memory release regardless of crash status
    if model_drift is not None:
        del model_drift
    if 'X_tr_ext' in locals():
        del X_tr_ext, X_val_ext, X_te_2020, y_tr_ext, y_val_ext, y_te_2020, adj_ext
    tf.keras.backend.clear_session()
    gc.collect()